# LLMFA: annual-report analysis for any company you upload

**What it does**
Reads annual-report PDFs and builds, for each company you pick, one table of financial variables, derived metrics, valuation, scenarios, 22 answered questions, charts and an HTML report. Each variable is filled by three routes combined under a fixed rule (Cell 15):
- **P**: a rule-based PDF parser on the statement pages.
- **L**: a Llama row-picker (`NousResearch/Meta-Llama-3.1-8B-Instruct`, 4-bit NF4; fallbacks are set in Cell 1). Llama copies a row number and a value, and Python validates it and does all arithmetic.
- **O**: a web override from Screener.in and Yahoo Finance (`yfinance`), used only after an identity check confirms the web series describes the same company as the report (Cell 14C). Market data (prices) is dated and never mixed with the report period.

After that: QA and restatement checks, derived metrics, valuation (P/E, P/B, EV/EBITDA, dividend yield, earnings yield), a reverse DCF and a stress test, fact-linked narratives, the 22 questions in `22_questions_LLM.json` answered by Llama from those narratives, 7 comparable charts and share-price charts. A separate research stage (Cells 25-27) answers the questions in `LLM_api_calling_questions.json` for **one** selected company using **Tavily** web search and **Gemini**.

**What you need**
- A **GPU runtime** (Runtime > Change runtime type > T4/L4/A100) for the Llama cells.
- API keys, read from Colab Secrets, then an environment variable, then a hidden prompt: `GEMINI_API_KEY` and `TAVILY_API_KEY` (research stage, Cells 25-27) and `HF_TOKEN` (Hugging Face; optional for the open mirrors used here).
- In `BASE_DIR` (set in the first cell; default `MyDrive/LLMFA/Model_1`): `22_questions_LLM.json`, `LLM_api_calling_questions.json`, and the reports in `FY_Reports/`.
- Reports named **`COMPANY_YEAR.pdf`**, for example `HAL_2025.pdf`, `HAL_2024.pdf`, `TATA_POWER_2025.pdf`. YEAR is the financial-year *end* year (FY 2024-25 is 2025). Any number of companies and years.

**How to use**
1. Run the three setup cells at the top (Drive mount and `BASE_DIR`, secrets, dependencies), then **Cell 1** (settings: report folder, Llama model, valuation and scenario assumptions) and **Cell 2** (scans `FY_Reports/` and shows a company picker).
2. **Pick the companies** in the box (one or several), then run **Cell 2B** and every cell after it up to **Cell 24**. Only the companies you picked are analysed. Cell 24 builds the report (all questions and answers per company, charts, share-price charts) and downloads it if `AUTO_DOWNLOAD_REPORT` is `True`.
3. Research stage: run **Cell 25**, pick ONE company and press Confirm, then run **Cell 26** (Tavily + Gemini answers) and **Cell 27** (single-company report).
4. The **final cell (Cell 28)** saves the day's analysis (22 answers, charts, research answers) as an HTML file in `BASE_DIR/Analysis/`.

**Notes**
- Web data (Screener.in, Yahoo Finance, the NSE company list, Tavily, Gemini) is fetched live, so results can change between runs. Web responses are cached for `WEB_CACHE_DAYS` days.
- The Gemini model names in Cells 25 and 26 (`GEMINI_MODEL`, `MODEL_CHAIN`) are settings; use names that are available on your account.
- To re-run, use *Runtime > Restart session and run all*. The final cell turns `BASE_DIR` into a string, so re-running earlier cells afterwards fails.


In [ ]:
# ===== CONFIG: Drive mount, BASE_DIR, derived directories =====
import importlib.util
from pathlib import Path

# True in Google Colab (mounts Google Drive). Set to False to run locally.
USE_COLAB_DRIVE = importlib.util.find_spec("google.colab") is not None

DRIVE_MOUNT = Path("/content/drive")

if USE_COLAB_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
    BASE_DIR = DRIVE_MOUNT / "MyDrive" / "LLMFA" / "Model_1"   # <-- edit this
else:
    BASE_DIR = Path("LLMFA") / "Model_1"                        # <-- edit this

# derived directories
LOCAL_PDF_DIR = Path("/content/_pdf_local") if USE_COLAB_DRIVE else BASE_DIR / "_pdf_local"


In [ ]:
# ===== SECRETS: Colab Secrets -> environment variable -> hidden prompt =====
import os
from getpass import getpass


def get_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = os.getenv(name)
    if value:
        return value
    return getpass(f"Enter {name} (input hidden, leave blank to skip): ") or None


GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

for _name, _value in (("GEMINI_API_KEY", GEMINI_API_KEY),
                      ("TAVILY_API_KEY", TAVILY_API_KEY),
                      ("HF_TOKEN", HF_TOKEN)):
    if _value:
        os.environ[_name] = _value


In [ ]:
# ===== DEPENDENCIES =====
import subprocess
import sys

# Install the notebook's packages in the active Python environment.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pymupdf",
    "yfinance",
    "requests",
    "beautifulsoup4",
    "lxml",
    "pandas",
    "numpy",
    "openpyxl",
    "plotly==6.3.0",
    "pikepdf",
    "ipywidgets",
    "google-genai",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers",
    "accelerate",
    "bitsandbytes",
])


In [ ]:
# ===== CELL 1 | CONFIG + INSTALL + DRIVE MOUNT =====
# ---------- where things live on Drive ----------
DRIVE_ROOT = str(BASE_DIR)
REPORT_FOLDER = "FY_Reports"
QUESTION_BANK = "22_questions_LLM.json"
BASIS = "consolidated"

SELECT_COMPANIES = None
DEFAULT_SELECT_ALL = False
COMPANY_ALIASES = {}
LEGACY_CACHE_RUN = "Aero_Defence_Sector"

# ---------- company identity ----------
IDENTITY_OVERRIDE = {}
ALLOW_KEY_AS_SYMBOL = True
ID_SCAN_PAGES = 60
MANUAL_MARKET = {}
COMPANY_NOTES = {}

# ---------- LLM ----------
LLM_MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"
LLM_FALLBACKS = [
    "NousResearch/Hermes-2-Pro-Llama-3-8B",
    "NousResearch/Llama-2-7b-chat-hf",
]

# ---------- valuation / scenario assumptions ----------
COST_OF_EQUITY = 0.12
TERMINAL_G = 0.05
DCF_YEARS = 10
STRESS_REV_DROP = 0.10
STRESS_RATE_UP = 0.02
INCLUDE_LEASES_IN_DEBT = False
FORCE_REFRESH = False
WEB_CACHE_DAYS = 7

# ---------- final report ----------
REPORT_TITLE = "LLMFA Company Analysis Report"
AUTO_DOWNLOAD_REPORT = True

print("Config loaded. Report folder on Drive:", f"{DRIVE_ROOT}/{REPORT_FOLDER}")

In [ ]:
# ===== CELL 2 | SCAN REPORT FOLDER AND PICK COMPANIES =====
import re
import json
import math
import time
import gc
import os
import hashlib
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

REPORT_ROOT = Path(DRIVE_ROOT) / REPORT_FOLDER

if not REPORT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Folder not found: {REPORT_ROOT}\n"
        "Create it in Drive and upload reports named like HAL_2025.pdf."
    )

_SKIP = {"FY", "AR", "ANNUAL", "REPORT", "INTEGRATED", "THE"}
_RANGE = r"(?<!\d)(20\d{2})\s*[-_/]\s*(\d{2})(?!\d)"


def parse_report_name(path):
    """Return (company_key, FY_end_year) from a report filename."""
    stem = path.stem
    years = []

    for start_year, end_year in re.findall(_RANGE, stem):
        years.append(int(start_year[:2] + end_year))

    rest = re.sub(_RANGE, " ", stem)

    years += [
        int(year)
        for year in re.findall(r"(?<!\d)(20\d{2})(?!\d)", rest)
    ]

    years += [
        2000 + int(year)
        for year in re.findall(r"(?i)FY\s*[-_]?(\d{2})(?!\d)", rest)
    ]

    years = [year for year in years if 2010 <= year <= 2035]
    if not years:
        return None

    fy_end = max(years)

    company_text = re.sub(_RANGE, " ", stem)
    company_text = re.sub(r"(?i)FY\s*[-_]?\d{2}(?!\d)", " ", company_text)
    company_text = re.sub(r"(?<!\d)20\d{2}(?!\d)", " ", company_text)

    tokens = [
        token.upper()
        for token in re.split(r"[^A-Za-z0-9&]+", company_text)
        if token
        and re.search(r"[A-Za-z]", token)
        and token.upper() not in _SKIP
    ]

    if not tokens:
        return None

    return "_".join(tokens), fy_end


pdfs = sorted(
    path for path in REPORT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

ALL_REPORTS = {}
SCAN_PROBLEMS = []

for path in pdfs:
    parsed = parse_report_name(path)

    if parsed is None:
        SCAN_PROBLEMS.append(
            f"Cannot read company/year from: {path.name} "
            "(use a name like HAL_2025.pdf)"
        )
        continue

    company, fy = parsed
    report_id = f"{company}_{fy}"

    if report_id in ALL_REPORTS:
        SCAN_PROBLEMS.append(
            f"Duplicate report for {report_id}: {path.name}; "
            f"keeping {ALL_REPORTS[report_id]['path'].name}"
        )
        continue

    ALL_REPORTS[report_id] = {
        "company": company,
        "fy": fy,
        "path": path,
    }

ALL_COMPANIES = sorted({
    report["company"] for report in ALL_REPORTS.values()
})

if not ALL_REPORTS:
    raise RuntimeError(
        f"No readable company/year PDF filenames found in {REPORT_ROOT}."
    )

print(f"Found {len(ALL_REPORTS)} reports for {len(ALL_COMPANIES)} companies.")

if SCAN_PROBLEMS:
    print("Filename scan notes:")
    for problem in SCAN_PROBLEMS:
        print(" -", problem)

# Start with nothing selected. Confirm is required before Cell 2B can run.
CONFIRMED_COMPANIES = None

try:
    import ipywidgets as widgets
    from IPython.display import display

    company_selector = {
        company: widgets.Checkbox(
            value=False,
            description=company,
            indent=False,
            layout=widgets.Layout(width="100%"),
        )
        for company in ALL_COMPANIES
    }

    selection_status = widgets.HTML(
        value="<span style='color:#9a6700'>Select companies, then confirm.</span>"
    )

    def _select_all_companies(_):
        for checkbox in company_selector.values():
            checkbox.value = True

    def _clear_company_selection(_):
        for checkbox in company_selector.values():
            checkbox.value = False

    def _selection_changed(_):
        global CONFIRMED_COMPANIES
        CONFIRMED_COMPANIES = None
        selection_status.value = (
            "<span style='color:#9a6700'>Selection changed. "
            "Click Confirm selection.</span>"
        )

    def _confirm_company_selection(_):
        global CONFIRMED_COMPANIES
        selected = [
            company
            for company, checkbox in company_selector.items()
            if checkbox.value
        ]

        if not selected:
            CONFIRMED_COMPANIES = None
            selection_status.value = (
                "<span style='color:#b00020'>Select at least one company, "
                "then click Confirm selection.</span>"
            )
            return

        CONFIRMED_COMPANIES = selected
        selection_status.value = (
            "<span style='color:#18723b'><b>Confirmed:</b> "
            + ", ".join(selected)
            + "</span>"
        )

    select_all_button = widgets.Button(
        description="Select all",
        button_style="primary",
        layout=widgets.Layout(width="110px"),
    )
    clear_selection_button = widgets.Button(
        description="Clear selection",
        layout=widgets.Layout(width="130px"),
    )
    confirm_selection_button = widgets.Button(
        description="Confirm selection",
        button_style="success",
        icon="check",
        layout=widgets.Layout(width="180px"),
    )

    select_all_button.on_click(_select_all_companies)
    clear_selection_button.on_click(_clear_company_selection)
    confirm_selection_button.on_click(_confirm_company_selection)

    for checkbox in company_selector.values():
        checkbox.observe(_selection_changed, names="value")

    company_choices = widgets.VBox(
        list(company_selector.values()),
        layout=widgets.Layout(max_height="320px", overflow_y="auto"),
    )

    display(widgets.VBox([
        widgets.HTML("<b>Select the companies to analyse:</b>"),
        widgets.HBox([
            select_all_button,
            clear_selection_button,
            confirm_selection_button,
        ]),
        company_choices,
        selection_status,
    ]))

except Exception as error:
    company_selector = None
    print("Company checkboxes could not be displayed:", error)
    print("Set SELECT_COMPANIES in Cell 1 to a list of company names.")

**Pick the companies in the box above, then run this cell and everything below.**

In [ ]:
# ===== CELL 2B | LOCK THE CONFIRMED COMPANY SELECTION =====

if SELECT_COMPANIES is not None:
    _picked = list(SELECT_COMPANIES)
else:
    _confirmed = globals().get("CONFIRMED_COMPANIES")
    if not _confirmed:
        raise RuntimeError(
            "No confirmed companies found. Run Cell 2, select at least one company, "
            "click Confirm selection, then run Cell 2B."
        )
    _picked = list(_confirmed)

SELECTED = [
    COMPANY_ALIASES.get(str(company).upper(), str(company).upper())
    for company in _picked
]

_unknown = [company for company in SELECTED if company not in ALL_COMPANIES]
if _unknown:
    raise ValueError(
        f"Company not found in {REPORT_FOLDER}: {_unknown}. "
        f"Available companies: {ALL_COMPANIES}"
    )

if not SELECTED:
    raise RuntimeError("Select at least one company before running Cell 2B.")

REPORTS = {
    report_id: report
    for report_id, report in ALL_REPORTS.items()
    if report["company"] in SELECTED
}


def rebuild_indexes():
    global COMPANIES, FYS
    COMPANIES = sorted({
        report["company"] for report in REPORTS.values()
    })
    FYS = {
        company: sorted(
            report["fy"]
            for report in REPORTS.values()
            if report["company"] == company
        )
        for company in COMPANIES
    }


rebuild_indexes()

# One output folder per company selection.
_company_run_name = "_".join(COMPANIES)
RUN_NAME = (
    _company_run_name
    if len(_company_run_name) <= 60
    else (
        "_".join(COMPANIES[:3])
        + f"_plus{len(COMPANIES) - 3}_"
        + hashlib.md5("|".join(COMPANIES).encode()).hexdigest()[:6]
    )
)

FINAL_ROOT = Path(DRIVE_ROOT) / "Final_Output"
CACHE = FINAL_ROOT / "_cache"
OUT = FINAL_ROOT / RUN_NAME

DIRS = {
    "text": CACHE / "text",
    "llm": CACHE / "llm",
    "web": CACHE / "web",
    "answers": OUT / "answers",
    "charts": OUT / "charts",
    "tables": OUT / "tables",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

TODAY = dt.date.today().isoformat()

# Reuse prior LLM/web caches when configured.
if LEGACY_CACHE_RUN and (FINAL_ROOT / LEGACY_CACHE_RUN).is_dir():
    import shutil

    copied_count = 0
    for subfolder in ("llm", "web"):
        source_folder = FINAL_ROOT / LEGACY_CACHE_RUN / subfolder
        if source_folder.is_dir():
            for source_file in source_folder.glob("*"):
                destination = DIRS[subfolder] / source_file.name
                if source_file.is_file() and not destination.exists():
                    try:
                        shutil.copy2(source_file, destination)
                        copied_count += 1
                    except Exception:
                        pass

    if copied_count:
        print(
            f"Reused {copied_count} cached files from "
            f"'{LEGACY_CACHE_RUN}'."
        )

problems = []
for company in COMPANIES:
    years = FYS[company]
    gaps = [
        year for year in range(years[0], years[-1] + 1)
        if year not in years
    ]

    if len(years) < 2:
        problems.append(
            f"{company}: only {len(years)} report(s); "
            "growth, averages and CAGR need multiple years."
        )

    if gaps:
        problems.append(
            f"{company}: missing FY{gaps}; metrics needing a prior year "
            "may use the next report's prior-year column where available."
        )

print(
    f"Confirmed companies: {COMPANIES}\n"
    f"Reports to analyse: {len(REPORTS)}\n"
    f"Output folder: {OUT}"
)

display(pd.DataFrame([
    {
        "company": company,
        "reports": len(FYS[company]),
        "first_FY": FYS[company][0],
        "last_FY": FYS[company][-1],
        "files": ", ".join(
            REPORTS[f"{company}_{year}"]["path"].name
            for year in FYS[company]
        ),
    }
    for company in COMPANIES
]))

if problems:
    print("NOTES:")
    for problem in problems:
        print(" -", problem)

In [ ]:
# ===== CELL 2C | PDF HEALTH CHECK: every selected PDF must open (repairs a damaged one; never overwrites the original) =====
import shutil, fitz
LOCAL_DIR, REPAIR_DIR = LOCAL_PDF_DIR, CACHE / "repaired"
LOCAL_DIR.mkdir(parents=True, exist_ok=True); REPAIR_DIR.mkdir(parents=True, exist_ok=True)

def try_open(p):
    try:
        d = fitz.open(p); n = len(d); d.close(); return n, None
    except Exception as e:
        return None, str(e)[:90]

HEALTH = []
for rid in sorted(REPORTS):
    p = REPORTS[rid]["path"]; n, err = try_open(p); status = "ok"
    if n is None:
        L = LOCAL_DIR / p.name
        try:
            shutil.copy2(p, L); n, err2 = try_open(L)
        except Exception as e:
            n, err2 = None, str(e)[:90]
        if n: REPORTS[rid]["path"] = L; status = "opened from a local copy (Drive read glitch)"
        else:
            try:
                import pikepdf
                R = REPAIR_DIR / p.name
                with pikepdf.open(L if L.exists() else p, attempt_recovery=True) as pdf: pdf.save(R)
                n, err3 = try_open(R)
                if n: REPORTS[rid]["path"] = R; status = "repaired copy used (original untouched)"
                else: status = f"UNREADABLE, dropped ({err3})"
            except Exception as e:
                status = f"UNREADABLE, dropped ({str(e)[:70]})"
    if status.startswith("UNREADABLE"): REPORTS.pop(rid)
    HEALTH.append({"report": rid, "file": p.name, "pages": n, "status": status})
rebuild_indexes()
display(pd.DataFrame(HEALTH))
if not REPORTS: raise RuntimeError("None of the selected PDFs could be opened. Re-upload them and rerun from Cell 2.")
lost = [c for c in SELECTED if c not in COMPANIES]
if lost: print("Dropped (no readable report left):", lost)

In [ ]:
# ===== CELL 2D | COMPANY IDENTITY: score every NSE company on name/symbol match + report evidence =====
import difflib
import re
import time
import unicodedata
from collections import Counter

import pandas as pd
import requests
import fitz

UA = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}
NSE_URLS = [
    "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv",
    "https://archives.nseindia.com/content/equities/EQUITY_L.csv",
]
MASTER_FILE = CACHE / "EQUITY_L.csv"

LEGAL_WORDS = {"limited", "ltd", "private", "pvt", "public", "the"}


def norm_name(value):
    """Normalize names for comparison, removing punctuation and legal suffixes."""
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode().lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(w for w in value.split() if w not in LEGAL_WORDS)


def alnum(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def compact(value):
    """Normalized name with all spaces removed (so 'Micro Systems' == 'Microsystems')."""
    return re.sub(r"[^a-z0-9]", "", norm_name(value))


def compact_text(value):
    """Lower-case letters/digits only, '&' read as 'and' (same convention as compact())."""
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", value.lower().replace("&", "and"))


def clean_key(value):
    """Company key or file stem -> compact name without years / 'annual report' words."""
    tokens = [
        t for t in norm_name(str(value).replace("_", " ")).split()
        if not re.fullmatch(r"(?:fy)?\d+", t) and t not in {"annual", "report", "ar", "fy"}
    ]
    return "".join(tokens)


def load_master():
    """Load the NSE equity master, refreshing the local cache when stale."""
    stale = (
        FORCE_REFRESH
        or not MASTER_FILE.exists()
        or time.time() - MASTER_FILE.stat().st_mtime > 30 * 86400
    )
    if stale:
        for url in NSE_URLS:
            try:
                response = requests.get(url, headers=UA, timeout=30)
                if response.status_code == 200 and "SYMBOL" in response.text[:300].upper():
                    MASTER_FILE.write_text(response.text, encoding="utf-8")
                    break
            except Exception:
                continue

    if not MASTER_FILE.exists():
        return None, (
            "NSE master list unavailable. Save EQUITY_L.csv from nseindia.com at "
            f"{MASTER_FILE}"
        )
    try:
        frame = pd.read_csv(MASTER_FILE)
        frame.columns = [c.strip().upper() for c in frame.columns]
        frame = frame.rename(columns={"NAME OF COMPANY": "NAME", "ISIN NUMBER": "ISIN"})
        missing = {"SYMBOL", "NAME", "ISIN", "FACE VALUE"} - set(frame.columns)
        if missing:
            return None, f"NSE master is missing columns: {sorted(missing)}"

        frame = frame[["SYMBOL", "NAME", "ISIN", "FACE VALUE"]].copy()
        frame = frame.dropna(subset=["SYMBOL", "NAME"]).drop_duplicates("SYMBOL")
        frame["SYMBOL"] = frame["SYMBOL"].astype(str).str.strip().str.upper()
        frame["NAME"] = frame["NAME"].astype(str).str.strip()
        frame["ISIN"] = frame["ISIN"].fillna("").astype(str).str.strip().str.upper()
        frame["FACE_VALUE"] = frame["FACE VALUE"]          # itertuples-safe copy
        frame["NORM"] = frame["NAME"].map(norm_name)
        frame["COMPACT"] = frame["NAME"].map(compact)
        frame["ALN"] = frame["SYMBOL"].map(alnum)
        return frame.reset_index(drop=True), ("downloaded/refreshed" if stale else "cached copy")
    except Exception as error:
        return None, f"NSE master unreadable: {type(error).__name__}: {error}"


MASTER, MASTER_NOTE = load_master()
if MASTER is None:
    print("NSE master list:", MASTER_NOTE)
else:
    print(f"NSE master list: {len(MASTER)} listed companies ({MASTER_NOTE})")


def pdf_head_text(path, pages=None):
    """Read the opening pages that commonly contain corporate identity details."""
    page_limit = max(int(ID_SCAN_PAGES), 8) if pages is None else pages
    with fitz.open(path) as document:
        return "\n".join(
            document[i].get_text() for i in range(min(page_limit, len(document)))
        )


def report_evidence(company_key):
    """Collect identity evidence (text, ISIN counts, filenames) from all selected reports."""
    texts = []
    for fy in FYS[company_key]:
        report = REPORTS.get(f"{company_key}_{fy}")
        if not report:
            continue
        try:
            texts.append(pdf_head_text(report["path"]))
        except Exception as error:
            print(f"Could not read identity text for {company_key} FY{fy}: {error}")

    raw_text = "\n".join(texts)
    isin_counts = Counter(re.findall(r"\bIN[A-Z0-9]{10}\b", raw_text.upper()))
    filenames = [
        REPORTS[f"{company_key}_{fy}"]["path"].stem
        for fy in FYS[company_key]
        if f"{company_key}_{fy}" in REPORTS
    ]
    return raw_text, isin_counts, filenames, len(texts)


def face_value_or_none(value):
    try:
        return None if pd.isna(value) else float(value)
    except (TypeError, ValueError):
        return None


def key_similarity(keys, name_compact, symbol_aln):
    """0..1 similarity between the file key(s) and an NSE company (name or symbol)."""
    best = 0.0
    for k in keys:
        if not k:
            continue
        if k.upper() == symbol_aln:
            return 1.0
        if name_compact:
            best = max(best, difflib.SequenceMatcher(None, k, name_compact).ratio())
            if len(k) >= 5 and len(name_compact) >= 5 and (
                name_compact.startswith(k) or k.startswith(name_compact)
            ):
                best = max(best, 0.9)
    return best


def score_candidates(company_key, raw_text, isin_counts, filenames):
    """Rank NSE companies: key similarity + how strongly the report itself points at them."""
    keys = [k for k in dict.fromkeys([clean_key(company_key)] + [clean_key(n) for n in filenames]) if k]
    text_c = compact_text(raw_text)
    rows = []
    for row in MASTER.itertuples():
        ks = key_similarity(keys, row.COMPACT, row.ALN)
        name_hits = text_c.count(row.COMPACT) if len(row.COMPACT) >= 7 else 0
        isin_hits = isin_counts.get(row.ISIN, 0) if row.ISIN else 0
        ev = min(name_hits, 50) + 5 * min(isin_hits, 10)
        if ev > 0 or ks >= 0.6:
            rows.append({
                "symbol": row.SYMBOL, "name": row.NAME, "isin": row.ISIN,
                "face_value": face_value_or_none(row.FACE_VALUE),
                "ks": ks, "ev": ev, "name_hits": name_hits, "isin_hits": isin_hits,
            })
    max_ev = max((r["ev"] for r in rows), default=0)
    for r in rows:
        r["score"] = r["ks"] + (r["ev"] / max_ev if max_ev else 0.0)
    rows.sort(key=lambda r: r["score"], reverse=True)
    return rows


def yahoo_lookup(company_key):
    """Fallback: Yahoo Finance search, NSE equities only. Returns (symbol, name, similarity) or None."""
    key_c = clean_key(company_key)
    query = " ".join(norm_name(company_key.replace("_", " ")).split())
    if not key_c or not query:
        return None

    data = None
    for host in ("query2", "query1"):
        try:
            r = requests.get(
                f"https://{host}.finance.yahoo.com/v1/finance/search",
                params={"q": query, "quotesCount": 8, "newsCount": 0},
                headers=UA, timeout=20,
            )
            if r.status_code == 200:
                data = r.json()
                break
        except Exception:
            continue
    if not data:
        return None

    found = []
    for item in data.get("quotes", []):
        sym = str(item.get("symbol", ""))
        if not sym.endswith(".NS") or item.get("quoteType") not in (None, "EQUITY"):
            continue
        nm = item.get("longname") or item.get("shortname") or ""
        sim = key_similarity([key_c], compact(nm), alnum(sym[:-3]))
        found.append((sim, sym[:-3].upper(), nm))
    found.sort(reverse=True)
    if not found:
        return None
    second = found[1][0] if len(found) > 1 else 0.0
    if found[0][0] >= 0.75 and found[0][0] - second >= 0.08:
        return found[0][1], found[0][2], found[0][0]
    return None


def resolve_company(company_key):
    # 1. Explicit user override always wins.
    override = (globals().get("IDENTITY_OVERRIDE", {}) or {}).get(company_key)
    if override:
        symbol = str(override.get("symbol", "")).strip().upper()
        return {
            "status": "override",
            "symbol": symbol or None,
            "official_name": override.get("name") or company_key,
            "isin": override.get("isin"),
            "face_value": override.get("face_value"),
            "evidence": ["IDENTITY_OVERRIDE in Cell 1"],
            "alternatives": [],
            "pdf_names": [],
        }

    raw_text, isin_counts, filenames, report_count = report_evidence(company_key)
    base = {
        "status": "unresolved", "symbol": None, "official_name": company_key,
        "isin": None, "face_value": None, "evidence": [], "alternatives": [],
        "pdf_names": [], "isins_in_report": sorted(isin_counts),
    }

    # 2. Score every NSE company; accept the leader only if it is clearly ahead.
    ranked = []
    if MASTER is None:
        base["evidence"] = [MASTER_NOTE]
    else:
        ranked = score_candidates(company_key, raw_text, isin_counts, filenames)
        base["alternatives"] = [
            f"{r['symbol']} ({r['name']}) key-sim {r['ks']:.2f}, report hits {r['ev']}"
            for r in ranked[:5]
        ]
        base["pdf_names"] = [
            f"{r['name']} (NSE: {r['symbol']})" for r in ranked[:5] if r["ev"] > 0
        ]
        if ranked:
            best = ranked[0]
            second_score = ranked[1]["score"] if len(ranked) > 1 else 0.0
            max_ev = max(r["ev"] for r in ranked)
            report_backed = best["ev"] >= 6 and best["ev"] == max_ev
            if best["score"] - second_score >= 0.12 and (best["ks"] >= 0.55 or report_backed):
                return {
                    **base,
                    "status": "resolved",
                    "symbol": best["symbol"],
                    "official_name": best["name"],
                    "isin": best["isin"] or None,
                    "face_value": best["face_value"],
                    "evidence": [
                        f"Name/symbol similarity {best['ks']:.2f}; official name found "
                        f"{best['name_hits']}x and ISIN {best['isin_hits']}x in the opening "
                        f"pages of {report_count} report(s); clear lead over the next candidate"
                    ],
                }

    # 3. Fallback: Yahoo Finance search (the numeric gate in Cell 14C still verifies it).
    hit = yahoo_lookup(company_key)
    if hit:
        sym, yname, sim = hit
        official, isin, fv = yname or company_key, None, None
        if MASTER is not None:
            match = MASTER[MASTER["SYMBOL"] == sym]
            if len(match):
                official = match.iloc[0]["NAME"]
                isin = match.iloc[0]["ISIN"] or None
                fv = face_value_or_none(match.iloc[0]["FACE_VALUE"])
        return {
            **base,
            "status": "resolved",
            "symbol": sym,
            "official_name": official,
            "isin": isin,
            "face_value": fv,
            "evidence": [
                f"Yahoo Finance search matched '{yname}' (similarity {sim:.2f}); "
                "to be verified by the numeric gate in Cell 14C"
            ],
        }

    if ranked and ranked[0]["ks"] >= 0.7:
        base["status"] = "ambiguous"
        base["evidence"] = ["Several close candidates and no clear leader; see 'other candidates'"]
    else:
        base["evidence"] = base["evidence"] or ["No NSE company matched the key or the report evidence"]
    return base


IDENT = {company: resolve_company(company) for company in COMPANIES}

# If multiple file keys map to the same symbol, flag all as ambiguous.
resolved_symbols = [
    i["symbol"] for i in IDENT.values()
    if i["status"] in ("resolved", "override") and i["symbol"]
]
symbol_counts = Counter(resolved_symbols)
for company, identity in IDENT.items():
    if identity["status"] == "resolved" and identity["symbol"] and symbol_counts[identity["symbol"]] > 1:
        identity["status"] = "ambiguous"
        identity["evidence"].append(
            f"NSE symbol {identity['symbol']} matched more than one company key"
        )
        identity["symbol"] = None

META = {}
for company, identity in IDENT.items():
    usable = identity["status"] in ("resolved", "override")
    symbol = identity["symbol"] if usable else None
    META[company] = {
        "name": identity["official_name"] or company,
        "symbol": symbol,
        "isin": identity.get("isin"),
        "face_value": identity.get("face_value"),
        "status": identity["status"],
        "yf": [f"{symbol}.NS"] if symbol else [],
        "screener": [symbol] if symbol else [],
    }

display(pd.DataFrame([
    {
        "company": company,
        "status": identity["status"],
        "NSE symbol": identity["symbol"],
        "official name": identity["official_name"],
        "ISIN": identity.get("isin"),
        "face value": identity.get("face_value"),
        "evidence": " | ".join(identity["evidence"])[:260],
        "other candidates": " ; ".join(identity["alternatives"])[:260],
    }
    for company, identity in IDENT.items()
]))

unresolved = [c for c in IDENT if not META[c]["symbol"]]
if unresolved:
    print(
        "IDENTITY NOT RESOLVED:", unresolved,
        "- report-only analysis will continue; web market data will be skipped for these. "
        "Fix by adding IDENTITY_OVERRIDE = {'KEY': {'symbol': 'XYZ'}} in Cell 1."
    )

In [ ]:
# ===== CELL 3 | HELPERS: number tokens, row splitting, label normalising, unit detection =====
DASHES = "-\u2212\u2013\u2014"
def is_num_tok(t):
    return bool(re.fullmatch(r"\(?[-\u2212\u2013]?\(?\d[\d,]*(?:\.\d+)?\)?", t)) or bool(re.fullmatch(rf"[{DASHES}]+", t))

def parse_num(t):
    """'(1,250)' -> -1250.0 | '-' -> 'DASH' | '1,234.5' -> 1234.5"""
    if re.fullmatch(rf"[{DASHES}]+", t): return "DASH"
    neg = "(" in t or t[:1] in DASHES
    s = re.sub(r"[^\d.]", "", t)
    if not s or s == ".": return None
    v = float(s); return -v if neg else v

def split_row(line):
    """label, numeric tokens (FIRST numeric group only), merged-line flag (numbers, words, numbers on one line)."""
    toks = line.split(); first = None
    for i, t in enumerate(toks):
        if i > 0 and is_num_tok(t): first = i; break
    if first is None: return line.strip(), [], False
    j = first
    while j < len(toks) and is_num_tok(toks[j]): j += 1
    tail = toks[j:]
    merged = any(re.search(r"[A-Za-z]", x) for x in tail) and any(is_num_tok(x) for x in tail)
    return " ".join(toks[:first]), toks[first:j], merged

def norm_label(s):
    s = s.lower().replace("&", " and ").replace("\u20b9", " ")
    s = re.sub(r"\(\s*(?:refer\s+)?notes?[^)]*\)", " ", s)
    s = re.sub(r"\(\s*(?:[ivx]+\s*[-+\u2013]\s*)*[ivx]+\s*\)", " ", s)
    s = re.sub(r"\(\s*[a-z]\s*\)\s*$", " ", s)                                       # trailing "(A)"
    for _ in range(2):
        s = re.sub(r"^\s*(\(?[a-z]{1,4}\)|\(?[ivx]{1,5}\)|\(?\d{1,2}\)|[a-z]\.|[ivx]+\.|\d+\.|[ivx]{1,5}\s|\d{1,2}\s(?=[a-z]))\s*", "", s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def join_wrapped(pairs):
    """pairs = [(page, line)]. A line that starts with a number directly after a label-only line is the wrapped value part of that label."""
    out = []
    for pg, ln in pairs:
        t = ln.split()
        if out and t and is_num_tok(t[0]) and out[-1][0] == pg:
            prev = out[-1][1]; pl, pn, _ = split_row(prev)
            if not pn and re.search(r"[A-Za-z]", prev) and len(prev) < 140:
                out[-1] = (pg, prev + " " + ln); continue
        out.append((pg, ln))
    return out

_UNIT_RE = re.compile(r"(?:\bin\b|amount in|figures in|\u20b9|`|\brs\.?|\binr\b)\s*(?:rs\.?|\u20b9|inr|rupees)?\s*(crores?|lakhs?|lacs?|millions?|billions?|thousands?)\b", re.I)
_UNIT_F = {"crore": 1.0, "lakh": 0.01, "lac": 0.01, "million": 0.1, "billion": 100.0, "thousand": 1e-4}
def detect_unit(lines):
    """factor to INR crore from the statement header; (None, '') if not printed."""
    for ln in lines:
        m = _UNIT_RE.search(ln)
        if m:
            w = m.group(1).lower().rstrip("s")
            return _UNIT_F.get(w), m.group(0)
    return None, ""

def modal_K(rows):
    c = Counter(len(r["nums"]) for r in rows if r["norm"].startswith("total") and 2 <= len(r["nums"]) <= 4 and not r["merged"])
    return c.most_common(1)[0][0] if c else 2

def cur_prior(nums, K):
    """current year = FIRST of the LAST K numeric tokens; prior = the next one. DASH -> None."""
    if len(nums) < K: return None, None
    v = [parse_num(x) for x in nums]
    cu = v[-K]; pr = v[-K + 1] if K >= 2 else None
    f = lambda x: None if (x == "DASH" or x is None) else x
    return f(cu), f(pr)

# self-test of the row logic
_t = split_row("Finance costs 29 1,150 1,010 Basic (in Rs) 32 - 6.62")
assert _t[1][-1] == "1,010" and _t[2] is True and cur_prior(_t[1], 2)[0] == 1150.0
assert cur_prior(split_row("Other equity 16 241 (1,250) (900)")[1], 2) == (-1250.0, -900.0)
assert norm_label("(a) Profit/(loss) before tax") == "profit loss before tax"
print("Helpers OK.")

In [ ]:
# ===== CELL 4 | PDF -> PAGE LINES (cached on Drive, no LLM) =====
import fitz
def file_sig(p):
    p = Path(p); sz = p.stat().st_size
    with open(p, "rb") as fh:
        a = fh.read(65536); fh.seek(max(0, sz - 65536)); b = fh.read(65536)
    return f"{sz}-{hashlib.sha1(a + b).hexdigest()[:16]}"

def page_lines(page, ytol=3.0):
    words = page.get_text("words")
    if not words: return []
    words.sort(key=lambda w: ((w[1] + w[3]) / 2, w[0]))
    groups, cur, cy = [], [], None
    for w in words:
        yc = (w[1] + w[3]) / 2
        if cy is None or abs(yc - cy) <= ytol:
            cur.append(w); cy = yc if cy is None else (cy * (len(cur) - 1) + yc) / len(cur)
        else:
            groups.append(cur); cur = [w]; cy = yc
    groups.append(cur)
    out = [" ".join(w[4] for w in sorted(g, key=lambda w: w[0])).strip() for g in groups]
    return [l for l in out if l]

TEXT, TXT_INFO = {}, []
t0 = time.time()
for k, rid in enumerate(sorted(REPORTS), 1):
    f = DIRS["text"] / f"{rid}.json"; sig = file_sig(REPORTS[rid]["path"]); hit = None     # cache is valid only for the same PDF (size + content signature)
    if f.exists() and not FORCE_REFRESH:
        try:
            ck = json.loads(f.read_text())
            if isinstance(ck, dict) and ck.get("sig") == sig: hit = ck["pages"]
        except Exception: hit = None
    if hit is not None: TEXT[rid] = hit; src = "cache"
    else:
        doc = fitz.open(REPORTS[rid]["path"]); TEXT[rid] = [page_lines(p) for p in doc]; doc.close()
        f.write_text(json.dumps({"sig": sig, "pages": TEXT[rid]})); src = "pdf"
    n = len(TEXT[rid]); empty = sum(1 for p in TEXT[rid] if len(p) < 3)
    TXT_INFO.append({"report": rid, "pages": n, "empty_pages": empty, "source": src})
    print(f"[{k}/{len(REPORTS)}] {rid}: {n} pages, {empty} empty ({src}) | {time.time()-t0:.0f}s", flush=True)
ti = pd.DataFrame(TXT_INFO); display(ti)
bad = ti[ti.empty_pages > 0.3 * ti.pages]
if len(bad): print("WARNING: mostly empty pages (scanned PDF? OCR is not supported):", bad.report.tolist())

In [ ]:
# ===== CELL 5 | PAGE FILTER: score every page for each statement / narrative group =====
TITLE_LINES = 18
KIND_RULES = {
 "bs": {"title": r"balance\s*sheet", "must": [r"total assets", r"total equity and liabilities", r"total liabilities and equity"],
        "bonus": [r"total current liabilities", r"total current assets", r"inventories", r"trade receivables", r"borrowings", r"cash and cash equivalents"],
        "neg_title": r"cash\s*flow|changes in equity|notes? (forming|to)|statement of profit",
        "anchor": r"total equity and liabilities|total liabilities and equity"},
 "pl": {"title": r"profit\s*(and|&)\s*loss|income statement|statement of profit", "must": [r"revenue from operations", r"total income", r"profit.{0,12}before tax"],
        "bonus": [r"total expenses", r"finance costs?", r"earnings per", r"other income", r"tax expense", r"depreciation"],
        "neg_title": r"balance\s*sheet|cash\s*flow|changes in equity|notes? (forming|to)",
        "anchor": r"earnings per|total comprehensive income|profit.{0,20}for the (year|period)"},
 "cf": {"title": r"cash\s*flows?", "must": [r"operating activities"],
        "bonus": [r"investing activities", r"financing activities", r"cash and cash equivalents at the (end|close)", r"purchase of property", r"dividend"],
        "neg_title": r"balance\s*sheet|profit and loss|changes in equity|notes? (forming|to)",
        "anchor": r"financing activities"},
}
NARR_RULES = {
 "segment":       {"pos": [r"segment information", r"segment reporting", r"reportable segments?", r"segment revenue", r"segment results?", r"geographical (information|segments?)", r"major customers?", r"export"], "top": 3},
 "related_party": {"pos": [r"related part(y|ies)", r"key management personnel", r"transactions with related", r"holding company", r"subsidiar"], "top": 3},
 "contingent":    {"pos": [r"contingent liabilit", r"claims against the company", r"not acknowledged as debts", r"commitments"], "top": 3},
 "shareholding":  {"pos": [r"shareholding pattern", r"promoter", r"pledged", r"encumbered", r"president of india", r"public shareholding"], "top": 3},
 "auditor":       {"pos": [r"independent auditor", r"emphasis of matter", r"basis for (qualified )?opinion", r"key audit matters?", r"qualified opinion", r"material uncertainty", r"going concern"], "top": 3},
 "kmp_pay":       {"pos": [r"remuneration", r"key managerial personnel", r"managing director", r"sitting fees", r"chairman and managing"], "top": 3},
 "dividend":      {"pos": [r"dividend per share", r"interim dividend", r"final dividend", r"dividend payout", r"dividend distribution"], "top": 3},
}

def page_feat(lines):
    low = [l.lower() for l in lines]; title = " ".join(low[:TITLE_LINES]); body = "\n".join(low)
    basis = "consolidated" if "consolidated" in title else ("standalone" if re.search(r"standalone|separate", title) else "unknown")
    ndig = sum(1 for l in lines for t in l.split() if is_num_tok(t))
    return {"title": title, "body": body, "basis": basis, "ndig": ndig}

def score_kind(kind, f):
    R = KIND_RULES[kind]
    if not re.search(R["title"], f["title"]) or not any(re.search(m, f["body"]) for m in R["must"]): return 0.0
    s = 65.0 + min(40, 8 * sum(1 for b in R["bonus"] if re.search(b, f["body"])))
    if re.search(R["neg_title"], f["title"]): s -= 40
    if "independent auditor" in f["title"]: s -= 40
    if f["ndig"] < 20: s -= 30
    s += 15 if f["basis"] == BASIS else (-10 if f["basis"] not in ("unknown", BASIS) else 0)
    return s

def pick_statement(kind, feats):
    sc = [(score_kind(kind, f), i) for i, f in enumerate(feats)]
    best = max(sc, key=lambda x: (x[0], -x[1]))
    if best[0] <= 0: return None, 0.0
    i = best[1]; pages = [i]
    if not re.search(KIND_RULES[kind]["anchor"], feats[i]["body"]) and i + 1 < len(feats): pages.append(i + 1)   # statement continues on the next page
    return pages, best[0]

def narrative_pages(feats):
    out = {}
    for g, R in NARR_RULES.items():
        sc = []
        for i, f in enumerate(feats):
            s = sum(min(len(re.findall(p, f["body"])), 3) * 4 for p in R["pos"]) + sum(8 for p in R["pos"] if re.search(p, f["title"]))
            if f["ndig"] < 8: s -= 6
            if s > 12: sc.append((s, i))
        out[g] = [i for s, i in sorted(sc, key=lambda x: (-x[0], x[1]))[:R["top"]]]
    return out
print("Page scorer ready.")

In [ ]:
# ===== CELL 6 | RUN PAGE FILTER ON ALL REPORTS =====
SEL, rows_out = {}, []
for rid in sorted(REPORTS):
    feats = [page_feat(p) for p in TEXT[rid]]
    s = {"flags": [], "basis": {}, "score": {}}
    for kind in ("bs", "pl", "cf"):
        pages, sc = pick_statement(kind, feats)
        s[kind] = pages or []; s["score"][kind] = round(sc)
        s["basis"][kind] = feats[pages[0]]["basis"] if pages else None
        if not pages: s["flags"].append(f"{kind}: not found")
        elif s["basis"][kind] != BASIS: s["flags"].append(f"{kind}: basis={s['basis'][kind]} (wanted {BASIS})")
    s["narr"] = narrative_pages(feats)
    SEL[rid] = s
    rows_out.append({"report": rid, **{f"{k}_page": ([p + 1 for p in s[k]] if s[k] else None) for k in ("bs", "pl", "cf")},
                     **{f"{k}_basis": s["basis"][k] for k in ("bs", "pl", "cf")}, **{f"{k}_score": s["score"][k] for k in ("bs", "pl", "cf")},
                     "flags": "; ".join(s["flags"])})
SEL_DF = pd.DataFrame(rows_out); display(SEL_DF)
# the three statements of one report should share a basis; a mix means a wrong page was chosen
for rid, s in SEL.items():
    b = {x for x in s["basis"].values() if x}
    if len(b) > 1: print("CHECK basis mix in", rid, s["basis"])
print("Narrative pages (1-based) for the first report:", {k: [p + 1 for p in v] for k, v in SEL[sorted(REPORTS)[0]]["narr"].items()})
print("Missing statements:", SEL_DF[SEL_DF["flags"].str.contains("not found")]["report"].tolist() or "none")

In [ ]:
# ===== CELL 6B | FALLBACK: find a missing statement by CONTENT (no title needed), then show the evidence =====
FALLBACK_BODY = {
 "bs": (r"total equity and liabilities|total liabilities and equity", r"total assets|total current liabilities|total current assets"),
 "pl": (r"revenue from operations|total income", r"profit.{0,12}before tax"),
 "cf": (r"operating activities", r"financing activities|investing activities"),
}
def content_fallback(rid, kind):
    feats = [page_feat(p) for p in TEXT[rid]]; a, b = FALLBACK_BODY[kind]; cands = []
    ref = (SEL[rid]["pl"][0] if SEL[rid].get("pl") else None)           # a balance sheet almost always sits within 3 pages BEFORE the P&L
    for i, f in enumerate(feats):
        if f["ndig"] < 20 or "independent auditor" in f["title"]: continue
        if re.search(a, f["body"]) and re.search(b, f["body"]):
            s = f["ndig"] + (30 if f["basis"] == BASIS else 0) + (25 if re.search(r"balance\s*sheet|assets and liabilities", f["body"][:1500]) else 0)
            if kind == "bs" and ref is not None and 0 < ref - i <= 3: s += 60
            if re.search(r"notes? (forming|to)", f["title"]): s -= 60
            cands.append((s, i))
    if not cands: return None
    i = max(cands, key=lambda x: (x[0], -x[1]))[1]
    pages = [i]
    if not re.search(KIND_RULES[kind]["anchor"], feats[i]["body"]) and i + 1 < len(feats): pages.append(i + 1)
    return pages

for rid in sorted(REPORTS):
    for kind in ("bs", "pl", "cf"):
        if not SEL[rid][kind]:
            pg = content_fallback(rid, kind)
            if pg:
                SEL[rid][kind] = pg; SEL[rid]["basis"][kind] = page_feat(TEXT[rid][pg[0]])["basis"]
                SEL[rid]["flags"].append(f"{kind}: found by content fallback (check the page)")
                print(f"{rid} {kind}: fallback chose page(s) {[p + 1 for p in pg]}")
                print("   first lines:", " | ".join(TEXT[rid][pg[0]][:6])[:230])
            else:
                print(f"{rid} {kind}: STILL NOT FOUND")
# basis report: what each statement says about itself
for rid in sorted(REPORTS):
    if any(b != BASIS for b in SEL[rid]["basis"].values() if b):
        print(f"BASIS NOTE {rid}: {SEL[rid]['basis']}  -> open the PDF and confirm whether statements are consolidated or standalone-only")
SEL_DF = pd.DataFrame([{"report": r, **{f"{k}_page": ([p + 1 for p in s[k]] if s[k] else None) for k in ("bs", "pl", "cf")},
                        **{f"{k}_basis": s["basis"][k] for k in ("bs", "pl", "cf")}, "flags": "; ".join(s["flags"])} for r, s in SEL.items()])
display(SEL_DF[SEL_DF["flags"] != ""] if (SEL_DF["flags"] != "").any() else "no flags: every statement found by its title")

In [ ]:
# ===== CELL 6C (v2) | TWO-PANEL PAGES: split at the second 'Particulars' header; gutter detection as fallback =====
def lines_from_words(words, ytol=3.0):
    if not words: return []
    words = sorted(words, key=lambda w: ((w[1] + w[3]) / 2, w[0])); groups, cur, cy = [], [], None
    for w in words:
        yc = (w[1] + w[3]) / 2
        if cy is None or abs(yc - cy) <= ytol:
            cur.append(w); cy = yc if cy is None else (cy * (len(cur) - 1) + yc) / len(cur)
        else: groups.append(cur); cur = [w]; cy = yc
    groups.append(cur)
    return [l for l in (" ".join(w[4] for w in sorted(g, key=lambda w: w[0])).strip() for g in groups) if l]

def split_by_particulars(page):
    words = page.get_text("words"); W = page.rect.width
    xs = sorted(w[0] for w in words if w[4].strip().lower() == "particulars")
    if len(xs) < 2 or xs[-1] - xs[0] < 0.3 * W: return None
    gx = xs[-1] - 4.0
    left = [w for w in words if (w[0] + w[2]) / 2 < gx]; right = [w for w in words if (w[0] + w[2]) / 2 >= gx]
    return lines_from_words(left) + lines_from_words(right)

def split_by_gutter(page):
    words = page.get_text("words"); n = len(words)
    if n < 60: return None
    W = page.rect.width; best = None
    for gx in np.arange(0.35 * W, 0.65 * W, 2.0):
        cross = sum(1 for w in words if w[0] < gx < w[2])
        if best is None or cross < best[0]: best = (cross, gx)
    cross, gx = best
    if cross > max(3, 0.02 * n): return None
    band = [w for w in words if gx <= w[0] < gx + 0.10 * W]; alpha = [w for w in band if re.search(r"[A-Za-z]", w[4])]
    left = [w for w in words if (w[0] + w[2]) / 2 < gx]; right = [w for w in words if (w[0] + w[2]) / 2 >= gx]
    if len(alpha) < 12 or len(alpha) < 0.4 * len(band) or len(left) < 0.25 * n or len(right) < 0.25 * n: return None
    return lines_from_words(left) + lines_from_words(right)

merged_cnt = lambda ls: sum(1 for l in ls if split_row(l)[2])
SPLIT_LOG = []
for rid in sorted(REPORTS):
    doc = fitz.open(REPORTS[rid]["path"])
    for kind in ("bs", "pl", "cf"):
        for pi in SEL[rid][kind]:
            pg = doc[pi]; before = merged_cnt(TEXT[rid][pi])
            new, how = split_by_particulars(pg), "particulars"
            if new is None:
                new, how = split_by_gutter(pg), "gutter"
                if new is not None and not (before >= 3 and merged_cnt(new) <= 0.5 * before): new = None
            if new is not None:
                TEXT[rid][pi] = new
                SPLIT_LOG.append({"report": rid, "stmt": kind, "page": pi + 1, "method": how, "merged_rows_before": before, "merged_rows_after": merged_cnt(new)})
    doc.close()
print("Two-panel pages split (in memory; rerun this cell after a restart):")
display(pd.DataFrame(SPLIT_LOG) if SPLIT_LOG else "none")

In [ ]:
# ===== CELL 6D | A MISSING BALANCE SHEET sits directly before the P&L page and says 'Balance Sheet' near the top =====
for rid in sorted(REPORTS):
    if SEL[rid]["bs"] or not SEL[rid]["pl"]: continue
    p = SEL[rid]["pl"][0]
    for q in (p - 1, p - 2, p - 3):
        if q < 0: continue
        head = " ".join(TEXT[rid][q][:8]).lower(); body = "\n".join(TEXT[rid][q]).lower()
        if "balance sheet" in head and "current assets" in body and "equity" in body:
            SEL[rid]["bs"] = [q]; SEL[rid]["basis"]["bs"] = page_feat(TEXT[rid][q])["basis"]
            SEL[rid]["flags"].append("bs: found by position before the P&L (Cell 6D)")
            print(f"{rid}: balance sheet = PDF page {q + 1} | {' | '.join(TEXT[rid][q][:3])[:140]}"); break
    else:
        print(f"{rid}: STILL no balance sheet found")

In [ ]:
# ===== CELL 7 (COMPLETE) | STATEMENT ROWS: numbered rows, column count K, unit, sections, unlabeled sub-totals =====
def split_row(line):
    """label, numeric tokens (FIRST numeric group only), merged-line flag. A numbers-only line returns an empty label (unlabeled sub-total)."""
    toks = line.split()
    if len(toks) >= 2 and all(is_num_tok(t) for t in toks): return "", toks, False
    first = None
    for i, t in enumerate(toks):
        if i > 0 and is_num_tok(t): first = i; break
    if first is None: return line.strip(), [], False
    j = first
    while j < len(toks) and is_num_tok(toks[j]): j += 1
    tail = toks[j:]
    merged = any(re.search(r"[A-Za-z]", x) for x in tail) and any(is_num_tok(x) for x in tail)
    return " ".join(toks[:first]), toks[first:j], merged

def join_wrapped(pairs):
    """a line starting with a number right after a label-only line (that starts like a real label) is the wrapped value part of that label"""
    out = []
    for pg, ln in pairs:
        t = ln.split()
        if out and t and is_num_tok(t[0]) and out[-1][0] == pg:
            prev = out[-1][1]; _, pn, _ = split_row(prev)
            if not pn and re.search(r"[A-Za-z]", prev) and len(prev) < 140 and (prev[0].isupper() or prev[0] in "(0123456789"):
                out[-1] = (pg, prev + " " + ln); continue
        out.append((pg, ln))
    return out

def detect_K(rows, head):
    """value columns = number of distinct years in the statement header (normally 2); modal count of 'total' rows otherwise"""
    yrs = set()
    for ln in head:
        l = ln.lower()
        if re.search(r"march|as at|as on|year ended|period ended", l): yrs |= set(re.findall(r"(?<!\d)(20\d{2})(?!\d)", l))
    return 2 if len(yrs) == 2 else modal_K(rows)

BS_SEC = [(r"^non current assets$", "NCA"), (r"^current assets$", "CA"), (r"^non current liabilities$", "NCL"), (r"^current liabilities$", "CL"), (r"^equity$", "EQ")]
CF_SEC = [(r"operating activities", "OP"), (r"investing activities", "INV"), (r"financing activities", "FIN")]
SUBTOTAL_NAME = {"NCA": "total non current assets", "CA": "total current assets", "EQ": "total equity",
                 "NCL": "total non current liabilities", "CL": "total current liabilities"}

def assign_sections(rows):
    """balance-sheet sections from the TOTAL rows (robust when headings are numbered or missing); keeps heading tags otherwise"""
    idx = lambda *names: next((i for i, r in enumerate(rows) if r["norm"] in names), None)
    tnca, tca = idx("total non current assets"), idx("total current assets")
    teq, tncl, tcl = idx("total equity"), idx("total non current liabilities"), idx("total current liabilities")
    for i, r in enumerate(rows):
        if tnca is not None and tca is not None and tnca < tca:
            if i <= tnca: r["section"] = "NCA"
            elif i <= tca: r["section"] = "CA"
            elif teq is not None and i <= teq: r["section"] = "EQ"
        if teq is not None and tncl is not None and tcl is not None and teq < tncl < tcl:
            if teq < i <= tncl: r["section"] = "NCL"
            elif tncl < i <= tcl: r["section"] = "CL"
    return rows

def build_stmt(rid, kind):
    pages = SEL[rid][kind]
    if not pages: return None
    pairs = join_wrapped([(p + 1, ln) for p in pages for ln in TEXT[rid][p]])
    rows, sec, n_total = [], None, 0
    for n, (pg, ln) in enumerate(pairs, 1):
        label, nums, merged = split_row(ln); nl = norm_label(label)
        if not nums:
            if kind == "bs":
                for pat, tag in BS_SEC:
                    if re.fullmatch(pat, nl): sec = tag
            elif kind == "cf" and not nl.startswith("net cash"):
                for pat, tag in CF_SEC:
                    if re.search(pat, nl): sec = tag
        elif kind == "bs":
            if not label.strip():                                   # bare numbers = the sub-total of the section above
                if sec in SUBTOTAL_NAME: nl = SUBTOTAL_NAME[sec]
            elif nl == "total":                                     # 'TOTAL' printed twice: assets side, then equity and liabilities
                n_total += 1; nl = "total assets" if n_total == 1 else "total equity and liabilities"
        rows.append({"n": n, "page": pg, "raw": ln, "label": label if label.strip() else "(unlabeled sub-total)",
                     "norm": nl, "nums": nums, "merged": merged, "section": sec})
    head = [ln for p in pages for ln in TEXT[rid][p][:40]]
    K = detect_K(rows, head)
    if kind == "bs": assign_sections(rows)
    unit, unit_txt = detect_unit(head)
    return {"rows": rows, "K": K, "unit": unit, "unit_txt": unit_txt, "pages": [p + 1 for p in pages]}

STMT, STMT_ERR = {}, []
for rid in sorted(REPORTS):
    for kind in ("bs", "pl", "cf"):
        try:
            st = build_stmt(rid, kind)
            if st: STMT[(rid, kind)] = st
        except Exception as e:
            STMT_ERR.append((rid, kind, f"{type(e).__name__}: {e}"))
print("Statements built:", len(STMT), "| errors:", len(STMT_ERR))
for x in STMT_ERR[:5]: print("  ERROR", *x)
if not STMT: raise RuntimeError("No statements built. Check Cell 6 (SEL) and the first error printed above.")
tab = [{"report": rid, "stmt": k, "pages": st["pages"], "rows": len(st["rows"]), "K": st["K"], "unit": st["unit_txt"] or "NOT PRINTED", "factor": st["unit"]}
       for (rid, k), st in sorted(STMT.items())]
display(pd.DataFrame(tab))
print("Units not printed (taken from the company's other reports, else inferred from web data):", sum(1 for t in tab if t["factor"] is None), "of", len(tab))
print("Reports with fewer than 3 statements:", [r for r in sorted(REPORTS) if sum((r, k) in STMT for k in ("bs", "pl", "cf")) < 3])

In [ ]:
# ===== CELL 7B (v2) | balance-sheet repair: wrapped numbers, section re-tag, Apollo "TOTAL LIABILITIES" = current total =====
import re, pandas as pd

def _rows_of(obj): return obj["rows"] if isinstance(obj, dict) else obj
def _K_of(obj):    return (obj.get("K") or 2) if isinstance(obj, dict) else 2
def _txt(r):       return str(r.get("label") or r.get("raw") or "")
def _clean(s):
    s = str(s or "").lower()
    s = re.sub(r"^[\W\d_]+", "", s)                    # leading numbers / bullets / page numbers
    s = re.sub(r"^(annual|report)\s+", "", s)          # sidebar words glued into the row
    s = re.sub(r"^[\W\d_]+", "", s)
    s = re.sub(r"^liabilities\b[\W\d_]*", "", s)       # "LIABILITIES 2 Non-current liabilities"
    return s.strip()
def _has_digit_tok(r): return any(re.search(r"\d", str(t)) for t in (r.get("nums") or []))
def _numeric_only(r, K):
    raw = str(r.get("raw") or r.get("label") or "").strip()
    return len(r.get("nums") or []) >= K and bool(re.fullmatch(r"[\d,\.\-\(\)\s]+", raw))

WRAP_PAT = re.compile(r"creditors other than micro|dues of micro|outstanding dues")
TE_PAT   = r"^total equity(?!\s+(and|attributable))"
REPAIR_LOG = []

for rid in sorted({k[0] for k in STMT if k[1] == "bs"}):
    obj = STMT[(rid, "bs")]; rows = _rows_of(obj); K = _K_of(obj)

    # (1) wrapped label (no digits) + numbers-only next row -> move the numbers up to the label row
    for i in range(len(rows) - 1):
        r, nx = rows[i], rows[i + 1]
        if (not _has_digit_tok(r)) and WRAP_PAT.search(_txt(r).lower()) and _numeric_only(nx, K):
            r["nums"] = list(nx["nums"])
            nx["nums"] = []; nx["label"] = "wrapped numbers moved"; nx["norm"] = "wrapped numbers moved"
            if "raw" in nx: nx["raw"] = "wrapped numbers moved"
            REPAIR_LOG.append((rid, "wrap_moved", _txt(r)[:70]))

    ta = next((i for i, r in enumerate(rows) if str(r.get("norm") or "") == "total assets"), None)
    if ta is None: continue
    post = rows[ta + 1:]
    if any(r.get("section") == "NCL" for r in post):      # tagging already fine -> leave the report alone
        continue

    # (2) re-tag sections after TOTAL ASSETS: EQ -> NCL (right after Total equity or NCL heading) -> CL -> END
    state = "EQ"; rows[ta]["section"] = "EQ"
    for r in post:
        c = _clean(_txt(r))
        if state == "END": r["section"] = "END"; continue
        if re.match(r"total equity and liabilities", c): r["section"] = "CL"; state = "END"; continue
        if re.search(r"(?<!other )(?<!total )non[- ]?current liabilities", c) and len(r.get("nums") or []) < K: state = "NCL"
        elif re.match(r"current liabilities", c): state = "CL"
        r["section"] = state
        if state == "EQ" and re.match(TE_PAT, c) and len(r.get("nums") or []) >= K: state = "NCL"
    REPAIR_LOG.append((rid, "retagged", f"{len(post)} rows"))

    # (3) name numbers-only subtotals by section (only when no labelled total of that kind exists)
    labelled = {str(r.get("norm") or "") for r in rows if not _numeric_only(r, K)}
    done = set()
    for r in post:
        if _numeric_only(r, K):
            if r["section"] == "NCL" and "total non current liabilities" not in labelled and "ncl" not in done:
                r["norm"] = "total non current liabilities"; r["label"] = "Total non-current liabilities (unlabeled)"; done.add("ncl")
                REPAIR_LOG.append((rid, "named_NCL_total", ""))
            elif r["section"] == "CL" and "total current liabilities" not in labelled and "cl" not in done:
                r["norm"] = "total current liabilities"; r["label"] = "Total current liabilities (unlabeled)"; done.add("cl")
                REPAIR_LOG.append((rid, "named_CL_total", ""))

    # (4) Apollo pattern: no "Total current liabilities"; "TOTAL LIABILITIES" in CL is the current total (identity-checked)
    if not any(str(r.get("norm") or "") == "total current liabilities" for r in rows):
        def find(pat, sec=None):
            return next((r for r in rows if (sec is None or r.get("section") == sec) and re.search(pat, _clean(_txt(r)))), None)
        tl, tel, te = find(r"^total liabilities$", "CL"), find(r"^total equity and liabilities"), find(TE_PAT)
        ncl = next((r for r in rows if str(r.get("norm") or "") == "total non current liabilities"), None)
        if all(x is not None for x in (tl, tel, te, ncl)):
            v = [parse_num(x["nums"][-K]) for x in (tl, tel, te, ncl)]       # tl, tel, te, ncl
            if None not in v and abs((v[2] + v[3] + v[0]) - v[1]) <= 0.005 * abs(v[1]):
                tl["norm"] = "total current liabilities"
                REPAIR_LOG.append((rid, "total_liabilities_is_current_total", f"{v[0]:,.2f} (identity ok)"))
            else:
                REPAIR_LOG.append((rid, "IDENTITY_FAILED", f"tl={v[0]} tel={v[1]} te={v[2]} ncl={v[3]}"))

lg = pd.DataFrame(REPAIR_LOG, columns=["report", "repair", "detail"])
print("Repairs applied:", len(lg)); display(lg.groupby("repair").size().to_frame("n"))
print(lg[lg.repair != "wrap_moved"].to_string())
for rid in [x for x in sorted(REPORTS) if (x, "bs") in STMT][:2]:
    for r in _rows_of(STMT[(rid, "bs")]):
        if re.search(r"equity share capital|borrowings|total liabilities", _txt(r).lower()):
            print(rid, r.get("section"), "|", _txt(r)[:50], r.get("nums"), "| norm:", r.get("norm"))

In [ ]:
# ===== CELL 8 | P ROUTE (fixed PDF parser): field spec + extraction on the filtered statement rows =====
# kind, label patterns on the NORMALISED label (first match order = priority), optional section filter, mode, absent_nil, scale
F = lambda kind, pats, **kw: {"kind": kind, "pats": pats, "sec": kw.get("sec"), "mode": kw.get("mode", "first"), "nil": kw.get("nil", False),
                              "excl": kw.get("excl"), "scale": kw.get("scale", "crore"), "abs": kw.get("abs", False)}
FIELD_SPEC = {
 # balance sheet
 "total_assets":        F("bs", [r"^total assets$"]),
 "total_equity":        F("bs", [r"^total equity$", r"^total shareholders funds?$", r"^total equity attributable to equity holders$"]),
 "equity_share_capital":F("bs", [r"^equity share capital$", r"^share capital$"], sec=["EQ", None]),
 "current_assets":      F("bs", [r"^total current assets$"]),
 "current_liabilities": F("bs", [r"^total current liabilities$"]),
 "trade_receivables":   F("bs", [r"^trade receivables?$"], sec=["CA"]),
 "inventories":         F("bs", [r"^inventories$"], sec=["CA"]),
 "cash_and_cash_equivalents": F("bs", [r"^cash and cash equivalents$"], sec=["CA"]),
 "borrowings_noncurrent": F("bs", [r"^(long term )?borrowings$"], sec=["NCL"], nil=True),
 "borrowings_current":    F("bs", [r"^(short term )?borrowings$"], sec=["CL"], nil=True),
 "borrowings_current_maturities": F("bs", [r"^current maturities of long term (debt|borrowings)"], sec=["CL"], nil=True),
 "lease_liabilities":   F("bs", [r"^lease liabilit(y|ies)$"], mode="sum", nil=True),
 # profit and loss
 "revenue_from_operations": F("pl", [r"^revenue from operations$", r"^revenue from operations net$", r"^total revenue from operations$"]),
 "other_income":        F("pl", [r"^other income$", r"^other income net$"]),
 "finance_costs":       F("pl", [r"^finance costs?$", r"^interest and finance charges$"]),
 "depreciation_and_amortisation": F("pl", [r"^depreciation and amorti[sz]ation( expenses?| and impairment( expenses?)?)?$", r"^depreciation amorti[sz]ation and impairment( expenses?)?$", r"^depreciation and amorti[sz]ation"]),
 "profit_before_tax":   F("pl", [r"^profit (loss )?before tax$", r"^profit (loss )?before tax"], excl=[r"exceptional", r"share of"], mode="last"),
 "exceptional_items":   F("pl", [r"^exceptional items?( net)?$"], nil=True),
 "tax_expense":         F("pl", [r"^total tax expenses?$", r"^tax expenses?$", r"^total tax$"]),
 "pat":                 F("pl", [r"^profit (loss )?for the (year|period)$", r"^net profit (loss )?for the (year|period)$", r"^profit (loss )?after tax$"]),
 "eps_basic":           F("pl", [r"^basic\b"], scale="rupee"),
 "cost_of_materials":   F("pl", [r"^cost of .{0,25}materials.{0,30}consumed"]),
 "purchases_stock":     F("pl", [r"^purchases? of (stock in trade|traded goods)"]),
 "change_inventories":  F("pl", [r"^(increase decrease|decrease increase|changes?) in inventories"]),
 "employee_benefit_expenses": F("pl", [r"^employee benefits?( expenses?)?$"]),
 "total_expenses":      F("pl", [r"^total expenses$"]),
 # cash flow
 "operating_cash_flow": F("cf", [r"^net cash.*operating activit"], excl=[r"discontinued"]),
 "investing_cash_flow": F("cf", [r"^net cash.*investing activit"], excl=[r"discontinued"]),
 "financing_cash_flow": F("cf", [r"^net cash.*financing activit"], excl=[r"discontinued"]),
 "capex":               F("cf", [r"^(purchase|acquisition|addition)s? (of|to) (property plant|fixed assets|tangible|intangible|investment property|capital work|ppe|plant)", r"^payments? (for|towards) (property plant|purchase of property|acquisition of property|capital expenditure)", r"^capital expenditure"], sec=["INV"], mode="sum", abs=True),
 "dividends_paid":      F("cf", [r"^dividends? paid", r"^payment of (final |interim )?dividend"], sec=["FIN"], mode="sum", abs=True, nil=True),
 "buybacks":            F("cf", [r"buy ?back"], sec=["FIN"], mode="sum", abs=True, nil=True),
 "repayment_borrowings":F("cf", [r"^repayment of (long term |short term |non current )?(borrowings|loans)"], sec=["FIN"], mode="sum", abs=True, nil=True),
 "proceeds_borrowings": F("cf", [r"^proceeds from (long term |short term |non current )?(borrowings|loans)"], sec=["FIN"], mode="sum", abs=True, nil=True),
 "acquisitions":        F("cf", [r"^(acquisition|investment)s? (of|in) (subsidiar|business|joint venture|associate)"], sec=["INV"], mode="sum", abs=True, nil=True),
}
VAR_BY_KIND = {k: [v for v, s in FIELD_SPEC.items() if s["kind"] == k] for k in ("bs", "pl", "cf")}
MULTI_VARS = {v for v, s in FIELD_SPEC.items() if s["mode"] == "sum"}

def p_extract(rid, var):
    sp = FIELD_SPEC[var]; st = STMT.get((rid, sp["kind"]))
    if st is None: return {"status": "no_statement"}
    K = st["K"]; cands = []
    for pat in sp["pats"]:
        for r in st["rows"]:
            if not r["nums"] or len(r["nums"]) < K: continue
            if sp["sec"] and r["section"] not in sp["sec"]: continue
            if re.search(pat, r["norm"]) and not (sp["excl"] and any(re.search(e, r["norm"]) for e in sp["excl"])): cands.append(r)
        if cands and sp["mode"] != "sum": break
    if var == "trade_payables" and not cands:        # header prints no numbers: add the MSME and other-creditor sub-rows
        cands = [r for r in st["rows"] if len(r["nums"]) >= K and r["section"] in ("CL", None)
                 and re.search(r"\bdues of (micro|creditors|other)", r["norm"])]
        sp = {**sp, "mode": "sum"}
    seen, sig, uniq = set(), set(), []
    for r in cands:                                  # an identical line (same label, same numbers) is a duplicate, not a second item
        k = (r["norm"], tuple(r["nums"]))
        if r["n"] in seen or k in sig: continue
        seen.add(r["n"]); sig.add(k); uniq.append(r)
    cands = uniq
    if not cands:
        if sp["nil"] and len(st["rows"]) > 15: return {"status": "nil_row_absent", "cur": 0.0, "prior": 0.0, "rows": [], "flag": False}
        return {"status": "missing"}
    if sp["mode"] == "sum":
        cps = [cur_prior(r["nums"], K) for r in cands]
        cur = sum(abs(c[0]) if sp["abs"] else c[0] for c in cps if c[0] is not None)
        pri = sum(abs(c[1]) if sp["abs"] else c[1] for c in cps if c[1] is not None)
        return {"status": "found", "cur": cur, "prior": pri, "rows": [r["n"] for r in cands], "flag": any(r["merged"] for r in cands),
                "label": " + ".join(r["label"] for r in cands)[:120], "page": cands[0]["page"]}
    r = cands[-1] if sp["mode"] == "last" else cands[0]
    cu, pr = cur_prior(r["nums"], K)
    flag = r["merged"] or len({cur_prior(c["nums"], K)[0] for c in cands}) > 1 or (len(r["nums"]) - K) > 2
    if cu is None: return {"status": "dash", "cur": None, "prior": pr, "rows": [r["n"]], "flag": flag, "label": r["label"], "page": r["page"]}
    return {"status": "found", "cur": cu, "prior": pr, "rows": [r["n"]], "flag": flag, "label": r["label"], "page": r["page"]}

def p_total_equity_fallback(rid, res):
    """no 'Total equity' row: owners' equity + non-controlling interest, flagged (v4 rule 4, Python only)."""
    st = STMT.get((rid, "bs"))
    if st is None or res.get("status") == "found": return res
    K = st["K"]; own = nci = None
    for r in st["rows"]:
        if len(r["nums"]) < K: continue
        if own is None and re.search(r"^equity attributable to (the )?(owners|equity holders|shareholders)", r["norm"]): own = r
        if nci is None and re.search(r"^(non controlling|minority) interests?$", r["norm"]): nci = r
    if own is None: return res
    a = cur_prior(own["nums"], K); b = cur_prior(nci["nums"], K) if nci else (0.0, 0.0)
    if a[0] is None: return res
    return {"status": "found", "cur": a[0] + (b[0] or 0), "prior": (a[1] or 0) + (b[1] or 0), "rows": [own["n"]] + ([nci["n"]] if nci else []),
            "flag": True, "label": "owners equity + NCI (derived)", "page": own["page"]}
print("P route spec:", len(FIELD_SPEC), "variables |", {k: len(v) for k, v in VAR_BY_KIND.items()})

In [ ]:
# ===== CELL 8B (v2) | PATTERN FIXES from the real labels in your reports =====
FIELD_SPEC["revenue_from_operations"]["pats"] = [r"^revenue from operations?$", r"^revenue from operations? net$", r"^total revenue from operations?$",
                                                 r"^revenue from contract with customers$", r"^revenue from operations? gross$"]
FIELD_SPEC["pat"]["pats"] = [r"^profit (loss )?for the (year|period)$", r"^net profit (loss )?for the (year|period)$", r"^profit (loss )?after tax$",
                             r"^profit (loss )?for the (year|period) from continuing operations$"]
FIELD_SPEC["profit_before_tax"].update({"pats": [r"^profit (loss )?before tax$", r"^profit (loss )?before tax",
                                                r"^profit before share of net profit of associates? accounted"],      # BEL: pre-tax result before the associate's share
                                        "excl": [r"exceptional"], "mode": "first"})
FIELD_SPEC["tax_expense"]["pats"] = [r"^total tax expenses?$", r"^tax expenses?$", r"^total tax$", r"^total provision for taxation$", r"^provision for taxation$"]
FIELD_SPEC["dividends_paid"]["pats"] = [r"dividend"]
FIELD_SPEC["dividends_paid"]["excl"] = [r"received|income|unpaid|unclaimed|proposed|declared"]
print("Pattern fixes applied.")

In [ ]:
# ===== CELL 9 | RUN P ROUTE: every variable, every report; coverage table =====
PR = {}   # (rid, var) -> result in printed units
for rid in sorted(REPORTS):
    for var in FIELD_SPEC:
        res = p_extract(rid, var)
        if var == "total_equity": res = p_total_equity_fallback(rid, res)
        PR[(rid, var)] = res
cov = []
for var in FIELD_SPEC:
    st_ = Counter(PR[(rid, var)]["status"] for rid in REPORTS)
    cov.append({"variable": var, "found": st_["found"], "nil_row_absent": st_["nil_row_absent"], "missing": st_["missing"] + st_["no_statement"] + st_["dash"],
                "flagged": sum(1 for rid in REPORTS if PR[(rid, var)].get("flag"))})
display(pd.DataFrame(cov))
need = ["total_assets", "total_equity", "current_liabilities", "revenue_from_operations", "profit_before_tax", "pat", "operating_cash_flow"]
print("Core variables still missing after P route:")
for var in need:
    miss = [rid for rid in sorted(REPORTS) if PR[(rid, var)]["status"] not in ("found", "nil_row_absent")]
    print(f"  {var:28s} -> {miss or 'none'}")

In [ ]:
# ===== CELL 9B (corrected) | trade_payables: header row if it has numbers, otherwise the sum of the MSME + other-creditor sub-rows =====
FIELD_SPEC["trade_payables"] = F("bs", [r"^trade payables?$"], sec=["CL"])
VAR_BY_KIND = {k: [v for v, s in FIELD_SPEC.items() if s["kind"] == k] for k in ("bs", "pl", "cf")}
MULTI_VARS = {v for v, s in FIELD_SPEC.items() if s["mode"] == "sum"}
for rid in sorted(REPORTS):
    PR[(rid, "trade_payables")] = p_extract(rid, "trade_payables")
print("trade_payables found in", sum(PR[(r, "trade_payables")]["status"] == "found" for r in REPORTS), "of", len(REPORTS), "reports")

In [ ]:
# ===== CELL 9D | DERIVED FIELDS where the report prints no line (identity-checked, flagged "derived") =====
def _sec_sum(st, tag):
    K = st["K"]; cs = ps = 0.0; ids = []
    for r in st["rows"]:
        if r["section"] == tag and len(r["nums"]) >= K and not r["norm"].startswith("total"):
            cu, pr = cur_prior(r["nums"], K)
            if cu is not None: cs += cu; ps += (pr or 0.0); ids.append(r["n"])
    return cs, ps, ids

DERIVED_LOG = []
for rid in sorted(REPORTS):
    st = STMT.get((rid, "bs")); ta, te = PR[(rid, "total_assets")], PR[(rid, "total_equity")]
    if st is not None and ta.get("status") == "found":
        nca, ca, ncl, cl = (_sec_sum(st, t) for t in ("NCA", "CA", "NCL", "CL"))
        pg = lambda ids: st["rows"][ids[0] - 1]["page"]
        if PR[(rid, "current_assets")].get("status") != "found" and ca[2] and nca[2] and abs(nca[0] + ca[0] - ta["cur"]) <= 0.005 * abs(ta["cur"]):
            PR[(rid, "current_assets")] = {"status": "found", "cur": ca[0], "prior": ca[1], "rows": ca[2], "flag": True, "label": "sum of current-asset rows (derived)", "page": pg(ca[2])}
            DERIVED_LOG.append((rid, "current_assets"))
        if PR[(rid, "current_liabilities")].get("status") != "found" and cl[2] and te.get("status") == "found" and abs((ta["cur"] - te["cur"] - ncl[0]) - cl[0]) <= 0.005 * abs(ta["cur"]):
            PR[(rid, "current_liabilities")] = {"status": "found", "cur": cl[0], "prior": cl[1], "rows": cl[2], "flag": True, "label": "sum of current-liability rows (derived)", "page": pg(cl[2])}
            DERIVED_LOG.append((rid, "current_liabilities"))
    t, pbt, pat = PR[(rid, "tax_expense")], PR[(rid, "profit_before_tax")], PR[(rid, "pat")]
    if t.get("status") != "found" and pbt.get("status") == "found" and pat.get("status") == "found":
        pr_ = (pbt["prior"] - pat["prior"]) if (pbt.get("prior") is not None and pat.get("prior") is not None) else None
        PR[(rid, "tax_expense")] = {"status": "found", "cur": pbt["cur"] - pat["cur"], "prior": pr_, "rows": [], "flag": True, "label": "PBT - PAT (derived)", "page": pbt.get("page")}
        DERIVED_LOG.append((rid, "tax_expense"))
print("Derived cells:", len(DERIVED_LOG)); print(Counter(v for _, v in DERIVED_LOG))
for var in ("current_assets", "current_liabilities", "profit_before_tax", "tax_expense", "pat", "revenue_from_operations"):
    print(f"  still missing {var:26s}:", [r for r in sorted(REPORTS) if PR[(r, var)].get("status") not in ("found", "nil_row_absent")] or "none")

In [ ]:
# ===== CELL 9I | (A) dash = nil borrowings  (B) trailing note-token window fix =====
import re
def _rows(rid, kind):
    obj = STMT.get((rid, kind)); return [] if obj is None else (obj["rows"] if isinstance(obj, dict) else obj)
def _Kk(rid, kind):
    obj = STMT[(rid, kind)]; return (obj.get("K") or 2) if isinstance(obj, dict) else 2
_DASH = {"-", "–", "—"}
def _isdash(t): return str(t).strip() in _DASH

# ---- A: borrowings printed as dashes (BEL, DATA) ----
BORR = {"borrowings_noncurrent": 0, "borrowings_current": 1}
fixedA, stillA = [], []
for rid in sorted(REPORTS):
    if (rid, "bs") not in STMT: continue
    need = [v for v in BORR if PR[(rid, v)].get("status") == "missing"]
    if not need: continue
    rows, K = _rows(rid, "bs"), _Kk(rid, "bs")
    start = next((i for i, r in enumerate(rows) if str(r.get("norm") or "") == "total assets"), 0)
    cand = [r for r in rows[start + 1:]
            if re.search(r"\bborrowings?\b", str(r.get("label") or "").lower())
            and not re.search(r"lease|maturit|repay|proceeds|accrued", str(r.get("label") or "").lower())
            and len(r.get("nums") or []) >= K]
    for v in need:
        r = None
        if len(cand) == 2: r = cand[BORR[v]]
        elif len(cand) == 1: r = cand[0] if (cand[0].get("section") == ("NCL" if v == "borrowings_noncurrent" else "CL")) else None
        if r is None: stillA.append((rid, v, f"{len(cand)} borrowing rows")); continue
        w = r["nums"][-K:]
        cur = 0.0 if _isdash(w[0]) else parse_num(w[0])
        pri = None if K < 2 else (0.0 if _isdash(w[1]) else parse_num(w[1]))
        if cur is None: stillA.append((rid, v, f"unparsed {w}")); continue
        PR[(rid, v)] = {"status": "found", "cur": cur, "prior": pri, "flag": False, "page": r.get("page"),
                        "label": str(r.get("label")) + (" (dash = nil)" if cur == 0.0 else "")}
        fixedA.append((rid, v, w))
print("A) borrowings filled:", len(fixedA), "| still missing:", len(stillA))
for x in fixedA: print("  fixed", *x)
for x in stillA: print("  STILL", *x)

# ---- B: trailing note token after the value columns (e.g. BEL_2026 tax ['2,02,949','1,81,185','19']) ----
def _big(t): return bool(re.search(r"[,\.]", str(t))) or len(re.sub(r"\D", "", str(t))) >= 3
anom, fixedB = [], []
for (rid, kind) in sorted(STMT):
    rows, K = _rows(rid, kind), _Kk(rid, kind)
    for r in rows:
        n = r.get("nums") or []
        if len(n) >= K + 1 and re.fullmatch(r"\d{1,2}", str(n[-1]).strip()) and _big(n[-K - 1]) and not _isdash(n[-K - 1]):
            anom.append((rid, kind, r.get("page"), str(r.get("label"))[:55], n))
            win = n[-K - 1:-1]
            newc = parse_num(win[0]); newp = parse_num(win[1]) if K >= 2 else None
            mis = parse_num(n[-K])
            for var, sp in FIELD_SPEC.items():
                if sp["kind"] != kind: continue
                p = PR[(rid, var)]
                if (p.get("status") == "found" and str(p.get("label")) == str(r.get("label")) and p.get("page") == r.get("page")
                        and p.get("cur") is not None and mis is not None and abs(p["cur"] - mis) < 1e-9 and newc is not None):
                    old = p["cur"]; p["cur"], p["prior"] = newc, newp
                    fixedB.append((rid, var, old, newc))
print("\nB) rows with a trailing note token:", len(anom))
for a in anom: print("  ", *a)
print("B) PR entries corrected:", len(fixedB))
for x in fixedB: print("  fixed", *x)

In [ ]:
# ===== CELL 9K | borrowings printed as "-" : fill every cell that is not a usable number =====
import re
def _rows(rid, kind):
    obj = STMT.get((rid, kind)); return [] if obj is None else (obj["rows"] if isinstance(obj, dict) else obj)
def _Kk(rid, kind):
    obj = STMT[(rid, kind)]; return (obj.get("K") or 2) if isinstance(obj, dict) else 2
_isdash = lambda t: str(t).strip() in {"-", "–", "—"}

BORR = {"borrowings_noncurrent": "NCL", "borrowings_current": "CL"}
filled, left = [], []
for rid in sorted(REPORTS):
    if (rid, "bs") not in STMT: continue
    rows, K = _rows(rid, "bs"), _Kk(rid, "bs")
    start = next((i for i, r in enumerate(rows) if str(r.get("norm") or "") == "total assets"), 0)
    for var, sec in BORR.items():
        p = PR[(rid, var)]
        usable = (p.get("status") == "nil_row_absent") or (p.get("status") == "found" and p.get("cur") is not None)
        if usable: continue
        cand = [r for r in rows[start + 1:] if r.get("section") == sec
                and re.search(r"\bborrowings?\b", str(r.get("label") or "").lower())
                and not re.search(r"lease|maturit|repay|proceeds|accrued|interest", str(r.get("label") or "").lower())
                and len(r.get("nums") or []) >= K]
        if len(cand) != 1: left.append((rid, var, f"{len(cand)} candidate rows")); continue
        r = cand[0]; w = r["nums"][-K:]
        cur = 0.0 if _isdash(w[0]) else parse_num(w[0])
        pri = None if K < 2 else (0.0 if _isdash(w[1]) else parse_num(w[1]))
        if cur is None: left.append((rid, var, f"unparsed {w}")); continue
        PR[(rid, var)] = {"status": "found", "cur": cur, "prior": pri, "flag": False, "page": r.get("page"),
                          "label": str(r.get("label")) + (" (dash = nil)" if cur == 0.0 else "")}
        filled.append((rid, var, w))
print("borrowings filled:", len(filled), "| left:", len(left))
for x in filled: print("  fixed", *x)
for x in left:   print("  LEFT ", *x)

In [ ]:
# ===== CELL 10 | LOAD LLM (4-bit NF4, needs a GPU runtime; Runtime > Change runtime type > T4/L4/A100) =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > GPU, then run again from Cell 1.")

def _hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t: return t
    except Exception: pass
    return os.environ.get("HF_TOKEN")
HF_TOKEN = _hf_token()
print("HF_TOKEN:", "found" if HF_TOKEN else "not set (open mirrors do not need it)")

tok = llm = LLM_LOADED = None
for _name in [LLM_MODEL] + [m for m in LLM_FALLBACKS if m != LLM_MODEL]:
    _t0 = time.time()
    try:
        tok = AutoTokenizer.from_pretrained(_name, token=HF_TOKEN)
        llm = AutoModelForCausalLM.from_pretrained(_name, device_map="auto", torch_dtype=torch.float16, token=HF_TOKEN,
              quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                     bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
        llm.eval(); LLM_LOADED = _name
        print(f"Loaded {_name} (4-bit NF4) on {torch.cuda.get_device_name(0)} in {time.time()-_t0:.0f}s | GPU {torch.cuda.memory_allocated()/1e9:.1f} GB")
        break
    except Exception as e:
        print(f"LOAD FAILED {_name}: {type(e).__name__}: {str(e)[:160]} -> next model")
        tok = llm = None; gc.collect(); torch.cuda.empty_cache()
if llm is None: raise RuntimeError("No model loaded. Set HF_TOKEN in Colab Secrets or change LLM_MODEL in Cell 1.")
if tok.pad_token is None: tok.pad_token = tok.eos_token
LLM_TAG = re.sub(r"[^A-Za-z0-9]+", "_", LLM_LOADED)
LLM_CTX = 4096 if "llama-2" in LLM_LOADED.lower() else 8192
_vocab = tok.get_vocab()
_EOS = [i for i in {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>") if "<|eot_id|>" in _vocab else None,
                    tok.convert_tokens_to_ids("<|im_end|>") if "<|im_end|>" in _vocab else None} if i is not None]

def n_tok(s): return len(tok(s, add_special_tokens=False)["input_ids"])

def llm_chat(system, user, prefill="", max_new_tokens=400):
    """greedy decoding; returns only the generated continuation (caller re-adds the prefill)."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           eos_token_id=_EOS or None, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
print("LLM ready. Context budget:", LLM_CTX, "tokens")

In [ ]:
# ===== CELL 11 | L ROUTE (Llama row-picker): prompts + validation. Llama copies a row number and the current-year number; Python validates and does all arithmetic =====
DEFS = {
 "total_assets": "the grand total of ALL assets: row 'Total assets' (a bank layout prints only 'TOTAL' once per side: use the assets side). Never a sub-total.",
 "total_equity": "'Total equity' (owners plus non-controlling interest). If no such row exists, list the owners' equity row(s) and the non-controlling interest row.",
 "equity_share_capital": "the paid-up 'Equity share capital' row only (not other equity).",
 "current_assets": "the sub-total 'Total current assets'.",
 "current_liabilities": "the sub-total 'Total current liabilities'.",
 "trade_receivables": "'Trade receivables' under CURRENT assets (not non-current, not contract assets, not other financial assets).",
 "inventories": "'Inventories' under current assets.",
 "cash_and_cash_equivalents": "'Cash and cash equivalents' under current assets (NOT 'bank balances other than cash and cash equivalents').",
 "borrowings_noncurrent": "'Borrowings' under NON-CURRENT liabilities. [] if none.",
 "borrowings_current": "'Borrowings' under CURRENT liabilities. [] if none.",
 "borrowings_current_maturities": "'Current maturities of long-term borrowings/debt' under current liabilities. [] if none.",
 "lease_liabilities": "every 'Lease liabilities' row (non-current and current), one entry each. [] if none.",
 "trade_payables": "'Trade payables' under current liabilities (if only sub-rows print numbers use the 'creditors other than micro and small enterprises' row).",
 "revenue_from_operations": "'Revenue from operations' (not total income).",
 "other_income": "'Other income' row.",
 "finance_costs": "the single TOTAL 'Finance costs' row (not a sub-item, not finance income).",
 "depreciation_and_amortisation": "'Depreciation and amortisation expense' row.",
 "profit_before_tax": "'Profit/(loss) before tax': the final pre-tax result AFTER exceptional items. NOT 'before exceptional items'.",
 "exceptional_items": "'Exceptional items' row. [] if none.",
 "tax_expense": "'Total tax expense' row.",
 "pat": "'Profit/(loss) for the year/period' after tax, total including non-controlling share, BEFORE other comprehensive income.",
 "eps_basic": "'Basic' earnings per share row (rupees per share).",
 "cost_of_materials": "'Cost of materials consumed' row.",
 "purchases_stock": "'Purchases of stock-in-trade' row. [] if none.",
 "change_inventories": "'Changes in inventories of finished goods, work-in-progress and stock-in-trade' row (brackets mean negative). [] if none.",
 "employee_benefit_expenses": "'Employee benefits expense' row.",
 "total_expenses": "'Total expenses' row.",
 "operating_cash_flow": "'Net cash from/(used in) operating activities'.",
 "investing_cash_flow": "'Net cash from/(used in) investing activities'.",
 "financing_cash_flow": "'Net cash from/(used in) financing activities'.",
 "capex": "EVERY row in the investing section that buys property, plant and equipment, intangible assets, capital work-in-progress or investment property (one entry each). NOT purchase of investments.",
 "dividends_paid": "every 'Dividend paid' row in the financing section. [] if none.",
 "buybacks": "every share buy-back row in the financing section. [] if none.",
 "repayment_borrowings": "every 'Repayment of borrowings/loans' row in the financing section. [] if none.",
 "proceeds_borrowings": "every 'Proceeds from borrowings/loans' row in the financing section. [] if none.",
 "acquisitions": "rows for acquisition of / investment in subsidiaries, businesses, joint ventures or associates in the investing section. [] if none.",
}
RULES = """You read the raw text rows of ONE consolidated financial statement. Each row is numbered like "12| label note 1,234 1,100".
Rows are copied from a PDF, so columns can be mixed up. Rules for each number you report:
- "value" is the CURRENT-YEAR number copied EXACTLY, character for character, from the cited row (keep commas, decimals and brackets).
- A row normally ends with the value columns, current year FIRST, then the previous year.
- IGNORE a note number or page reference that comes BEFORE the value columns (e.g. "Inventories 10 406 53,181 54,435": 10 is the note, 406 the page, the current year is 53,181).
- Brackets mean negative: copy them, e.g. "(1,250)".
- If a row holds two statements merged on one line, use ONLY the FIRST group of numbers.
- Use the row number from the left of the row. If no row fits, use []. Never invent a number. Never add or subtract numbers yourself.
- Reply with ONE JSON object and nothing else."""

EXAMPLES = {
 "bs": ("""--- statement rows ---
1| Particulars Note As at March 31, 2025 As at March 31, 2024
2| ASSETS
3| Current assets
4| Inventories 9 3,250 2,980
5| Trade receivables 10 1,800 1,500
6| Cash and cash equivalents 11 600 450
7| Total current assets 8,400 7,700
8| Total assets 20,000 18,500
9| EQUITY AND LIABILITIES
10| Equity share capital 15 100 100
11| Other equity 16 (1,250) (900)
12| Non-controlling interests 17 300 250
13| Total equity 9,200 8,350
14| Non-current liabilities
15| Borrowings 18 700 900
16| Current liabilities
17| Borrowings 19 400 300
18| Trade payables 20 1,100 950
19| Total current liabilities 5,100 4,800
20| Total equity and liabilities 20,000 18,500""",
  {"total_assets": [(8, "20,000")], "total_equity": [(13, "9,200")], "equity_share_capital": [(10, "100")], "current_assets": [(7, "8,400")],
   "current_liabilities": [(19, "5,100")], "trade_receivables": [(5, "1,800")], "inventories": [(4, "3,250")], "cash_and_cash_equivalents": [(6, "600")],
   "borrowings_noncurrent": [(15, "700")], "borrowings_current": [(17, "400")], "trade_payables": [(18, "1,100")]}),
 "pl": ("""--- statement rows ---
1| Revenue from operations 22 40,000 35,000
2| Other income 23 900 800
3| Cost of materials consumed 24 15,000 13,000
4| Purchases of stock-in-trade 400 350
5| Changes in inventories of finished goods (300) 200
6| Employee benefits expense 25 5,000 4,500
7| Finance costs 26 1,150 1,010 Basic (in Rs) 32 - 6.62
8| Depreciation and amortisation expense 27 1,300 1,200
9| Total expenses 36,000 31,500
10| Profit before exceptional items and tax 4,100 3,600
11| Exceptional items (net) 28 (250) -
12| Profit/(loss) before tax 3,850 3,600
13| Total tax expense 29 950 900
14| Profit/(loss) for the year 2,900 2,700
15| Basic (in Rs) 30 14.50 13.50""",
  {"revenue_from_operations": [(1, "40,000")], "other_income": [(2, "900")], "finance_costs": [(7, "1,150")], "depreciation_and_amortisation": [(8, "1,300")],
   "profit_before_tax": [(12, "3,850")], "exceptional_items": [(11, "(250)")], "tax_expense": [(13, "950")], "pat": [(14, "2,900")], "eps_basic": [(15, "14.50")],
   "cost_of_materials": [(3, "15,000")], "purchases_stock": [(4, "400")], "change_inventories": [(5, "(300)")], "employee_benefit_expenses": [(6, "5,000")],
   "total_expenses": [(9, "36,000")]}),
 "cf": ("""--- statement rows ---
1| A. Cash flow from operating activities
2| Profit before tax 3,850 3,600
3| Net cash from operating activities (A) 3,200 2,900
4| B. Cash flow from investing activities
5| Purchase of property, plant and equipment (1,400) (1,100)
6| Purchase of intangible assets (100) (80)
7| Purchase of investments (500) (300)
8| Net cash used in investing activities (B) (2,000) (1,500)
9| C. Cash flow from financing activities
10| Repayment of borrowings (300) (200)
11| Dividends paid (800) (700)
12| Net cash used in financing activities (C) (1,100) (900)""",
  {"operating_cash_flow": [(3, "3,200")], "investing_cash_flow": [(8, "(2,000)")], "financing_cash_flow": [(12, "(1,100)")],
   "capex": [(5, "(1,400)"), (6, "(100)")], "dividends_paid": [(11, "(800)")], "repayment_borrowings": [(10, "(300)")]}),
}

def build_system(kind):
    vs = VAR_BY_KIND[kind]
    shape = "{" + ", ".join('"%s": [{"row": <number>, "value": "<text>"}]' % v for v in vs) + "}"
    rows_text, ans = EXAMPLES[kind]
    full = {v: [{"row": r, "value": t} for r, t in ans.get(v, [])] for v in vs}
    return (RULES + "\n\nReply with exactly this JSON shape (use [] when no row fits):\n" + shape + "\n\nDefinitions:\n" +
            "\n".join("- %s: %s" % (v, DEFS[v]) for v in vs) + "\n\nWORKED EXAMPLE (numbers belong to the example only). Value columns per row: 2.\n" +
            rows_text + "\nAnswer: " + json.dumps(full))
SYS = {k: build_system(k) for k in ("bs", "pl", "cf")}
MAX_NEW, MAX_LINE = 750, 220

def build_user(rid, kind, st, budget):
    head = ("Statement: consolidated %s of report %s.\nValue columns per row: %d (current year first).\nRows:\n" %
            ({"bs": "balance sheet", "pl": "profit and loss statement", "cf": "cash flow statement"}[kind], rid, st["K"]))
    tail = "\nReturn the JSON object now."
    items = [(r, "%d| %s" % (r["n"], r["raw"][:MAX_LINE])) for r in st["rows"]]
    cost = lambda s: n_tok(s) + 1
    def total(its): return n_tok(head) + n_tok(tail) + sum(cost(s) for _, s in its)
    info = {"truncated": False}
    if total(items) > budget:                      # tier 1: drop number-less rows that are not section headings
        items = [(r, s) for r, s in items if r["nums"] or r["section"] is not None and len(r["raw"]) < 40]; info["truncated"] = True
    while items and total(items) > budget:         # tier 2: cut from the end
        items.pop()
    return head + "\n".join(s for _, s in items) + tail, info

def parse_json_obj(txt):
    s = "{" + txt; depth, end = 0, None
    for i, ch in enumerate(s):
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: end = i; break
    body = s[:end + 1] if end is not None else s
    for cand in (body, re.sub(r",\s*([}\]])", r"\1", body)):
        try: return json.loads(cand)
        except Exception: pass
    return None

LOOSE = {"total_assets": r"asset", "total_equity": r"equity|shareholders|capital|reserves|net worth|minority|non controlling", "equity_share_capital": r"capital",
 "current_assets": r"current assets", "current_liabilities": r"current liabilit", "trade_receivables": r"receivable|debtors", "inventories": r"inventor|stock",
 "cash_and_cash_equivalents": r"cash", "borrowings_noncurrent": r"borrowing|debt|loan|debenture", "borrowings_current": r"borrowing|debt|loan|debenture",
 "borrowings_current_maturities": r"borrowing|debt|loan|debenture|maturit", "lease_liabilities": r"lease", "trade_payables": r"payable|creditor",
 "revenue_from_operations": r"revenue|sales|turnover", "other_income": r"other income", "finance_costs": r"finance|interest", "depreciation_and_amortisation": r"deprec|amorti",
 "profit_before_tax": r"before tax", "exceptional_items": r"exceptional", "tax_expense": r"tax", "pat": r"profit|loss|income|surplus", "eps_basic": r"basic|earnings|eps",
 "cost_of_materials": r"material|consumed|cost", "purchases_stock": r"purchase|traded", "change_inventories": r"inventor|stock", "employee_benefit_expenses": r"employee|personnel|salar|wages",
 "total_expenses": r"expenses|expenditure", "operating_cash_flow": r"operating", "investing_cash_flow": r"investing", "financing_cash_flow": r"financing",
 "capex": r"purchase|acquisition|addition|payment|capital|property|fixed|intangible|plant", "dividends_paid": r"dividend", "buybacks": r"buy",
 "repayment_borrowings": r"repay", "proceeds_borrowings": r"proceeds|borrowing|loan", "acquisitions": r"acqui|invest"}
LOOSE["acquisitions"] = r"acquisition of (subsidiar|business|joint|associate)|investment in (subsidiar|joint|associate)"
LOOSE["proceeds_borrowings"] = r"proceeds from"
LOOSE["tax_expense"] = r"^(total )?(tax expenses?|provision for taxation|total provision for taxation|income tax expenses?)$"

def l_validate(rid, kind, obj):
    st = STMT[(rid, kind)]; K = st["K"]; byn = {r["n"]: r for r in st["rows"]}; out = {}
    for var in VAR_BY_KIND[kind]:
        sp = FIELD_SPEC[var]; ents = (obj or {}).get(var) if isinstance(obj, dict) else None
        if not isinstance(ents, list) or not ents: out[var] = {"status": "null"}; continue
        good, seen = [], set()
        for e in ents:
            try: n = int(e.get("row")); txt = str(e.get("value")).strip()
            except Exception: continue
            r = byn.get(n)
            if r is None or n in seen or len(r["nums"]) < K or txt in ("", "-", "null", "None"): continue
            if sp["sec"] and r["section"] not in sp["sec"]: continue
            if not re.search(LOOSE.get(var, "."), r["norm"]): continue
            tv = parse_num(txt.replace(" ", "")); cu, pr = cur_prior(r["nums"], K)
            if tv in (None, "DASH") or cu is None or abs(tv - cu) > 1e-6 * max(1.0, abs(cu)): continue
            good.append((r, cu, pr)); seen.add(n)
        if not good: out[var] = {"status": "invalid"}; continue
        if var == "total_equity":
            tot = [g for g in good if g[0]["norm"] == "total equity"]
            if tot: good = tot[:1]
        elif var not in MULTI_VARS: good = good[:1]
        f = (lambda x: abs(x) if sp["abs"] else x)
        out[var] = {"status": "found", "cur": sum(f(g[1]) for g in good), "prior": sum(f(g[2] or 0) for g in good),
                    "rows": [g[0]["n"] for g in good], "flag": any(g[0]["merged"] for g in good), "page": good[0][0]["page"]}
    return out
print("L route ready. System prompt tokens:", {k: n_tok(v) for k, v in SYS.items()})

In [ ]:
# ===== CELL 12 | RUN L ROUTE: one call per statement per report (resumes if interrupted) =====
LR, LLM_INFO, ncalls, t0 = {}, [], 0, time.time()
for rid in sorted(REPORTS):
    for kind in ("bs", "pl", "cf"):
        st = STMT.get((rid, kind))
        if st is None: continue
        sysm = SYS[kind]; budget = LLM_CTX - n_tok(sysm) - MAX_NEW - 250
        user, info = build_user(rid, kind, st, budget)
        h = hashlib.sha256((sysm + user).encode()).hexdigest()[:16]
        f = DIRS["llm"] / f"{LLM_TAG}_{rid}_{kind}.json"; raw = None
        if f.exists() and not FORCE_REFRESH:
            ck = json.loads(f.read_text())
            if ck.get("hash") == h: raw = ck["raw"]
        if raw is None:
            raw = llm_chat(sysm, user, prefill="{", max_new_tokens=MAX_NEW); ncalls += 1
            f.write_text(json.dumps({"hash": h, "raw": raw, "info": info}))
        obj = parse_json_obj(raw); res = l_validate(rid, kind, obj)
        LR.update({(rid, v): r for v, r in res.items()})
        c = Counter(r["status"] for r in res.values())
        LLM_INFO.append({"report": rid, "stmt": kind, "json_ok": obj is not None, "truncated": info["truncated"], **dict(c)})
        print(f"{rid} {kind}: json={'ok' if obj else 'FAIL'} {dict(c)} | calls {ncalls} | {time.time()-t0:.0f}s", flush=True)
display(pd.DataFrame(LLM_INFO).fillna(0))

In [ ]:
# ===== CELL 13 | O ROUTE DEFINITIONS (override sources). Values normalised to INR crore; EPS stays in rupees =====
import requests, yfinance as yf
from bs4 import BeautifulSoup
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36", "Accept-Language": "en-US,en;q=0.9"}
CR = 1e7
def fy_of(d):
    d = pd.Timestamp(d); return d.year if d.month <= 6 else d.year + 1

YF_MAP = {
 "balance_sheet": {"total_assets": ["Total Assets"], "total_equity": ["Total Equity Gross Minority Interest", "Stockholders Equity"], "current_assets": ["Current Assets"],
                   "current_liabilities": ["Current Liabilities"], "inventories": ["Inventory"], "trade_receivables": ["Accounts Receivable", "Receivables"],
                   "cash_and_cash_equivalents": ["Cash And Cash Equivalents"], "trade_payables": ["Accounts Payable", "Payables"]},
 "income_stmt": {"revenue_from_operations": ["Operating Revenue", "Total Revenue"], "finance_costs": ["Interest Expense"], "depreciation_and_amortisation": ["Reconciled Depreciation"],
                 "profit_before_tax": ["Pretax Income"], "tax_expense": ["Tax Provision"], "pat": ["Net Income Including Noncontrolling Interests", "Net Income"],
                 "pat_owners": ["Net Income Common Stockholders", "Net Income"], "eps_basic": ["Basic EPS"]},
 "cashflow": {"operating_cash_flow": ["Operating Cash Flow"], "investing_cash_flow": ["Investing Cash Flow"], "financing_cash_flow": ["Financing Cash Flow"],
              "capex": ["Capital Expenditure"], "dividends_paid": ["Cash Dividends Paid", "Common Stock Dividend Paid"], "buybacks": ["Repurchase Of Capital Stock"]}}
YF_ABS = {"capex", "dividends_paid", "buybacks"}

def name_sim(a, b):
    A, B = norm_name(a).split(), norm_name(b).split()
    if not A or not B: return 0.0
    i = len(set(A) & set(B)); f1 = 2 * i / (len(A) + len(B))
    return max(f1, difflib.SequenceMatcher(None, " ".join(A), " ".join(B)).ratio())

def yf_name_ok(t, c):
    """Yahoo's own name for the symbol must resemble the official name from the report (soft check; Cell 14C is the hard check)."""
    off = META[c].get("name")
    if not off or off == c: return True, ""
    try:
        info = t.info or {}; nm = info.get("longName") or info.get("shortName")
    except Exception: return True, ""
    if not nm: return True, ""
    s = name_sim(nm, off)
    return (s >= 0.5), ("" if s >= 0.5 else f"name mismatch: Yahoo says '{nm}', the report's company is '{off}' (similarity {s:.2f})")

def fetch_yf(c):
    err = []
    if not META[c]["yf"]: return {}, None, "no verified NSE symbol for this company"
    for sym in META[c]["yf"]:
        out = {}
        try:
            t = yf.Ticker(sym)
            for attr, mp in YF_MAP.items():
                df = getattr(t, attr)
                if df is None or df.empty: continue
                for col in df.columns:
                    fy = fy_of(col)
                    for var, labs in mp.items():
                        for lab in labs:
                            if lab in df.index and pd.notna(df.loc[lab, col]):
                                v = float(df.loc[lab, col]); v = v if var == "eps_basic" else v / CR
                                out[(fy, var)] = abs(v) if var in YF_ABS else v; break
            if out:
                ok, why = yf_name_ok(t, c)
                if ok: return out, sym, ""
                err.append(f"{sym}: {why}")
        except Exception as e: err.append(f"{sym}: {type(e).__name__} {str(e)[:80]}")
    return {}, None, "; ".join(err)

def http_get(url, params=None, headers=None, kind="text"):
    err = ""
    for k in range(3):
        try:
            r = requests.get(url, params=params, headers={**UA, **(headers or {})}, timeout=30)
            if r.status_code == 200: return (r.json() if kind == "json" else r.text), ""
            err = f"HTTP {r.status_code}"
        except Exception as e: err = type(e).__name__
        time.sleep(2 * (k + 1))
    return None, err

def cached(path, fn, kind="text"):
    """a file you save by hand at this path is used as-is (open the page in a browser > Save as HTML)."""
    if path.exists() and not FORCE_REFRESH and (time.time() - path.stat().st_mtime) < WEB_CACHE_DAYS * 86400:
        txt = path.read_text(encoding="utf-8"); return (json.loads(txt) if kind == "json" else txt), "cache"
    val, err = fn()
    if val is not None: path.write_text(json.dumps(val) if kind == "json" else val, encoding="utf-8"); time.sleep(2)
    return val, err

def scr_num(s):
    s = s.replace(",", "").replace("\xa0", " ").strip()
    return float(s) if re.fullmatch(r"-?\d+(\.\d+)?", s) else None
def scr_table(soup, sid):
    sec = soup.find("section", id=sid)
    return None if sec is None else (sec.find("table", class_="data-table") or sec.find("table"))
def scr_rows(tbl):
    if tbl is None or tbl.find("thead") is None or tbl.find("tbody") is None: return {}
    heads = [th.get_text(" ", strip=True) for th in tbl.find("thead").find_all("th")]; data = {}
    for tr in tbl.find("tbody").find_all("tr"):
        tds = tr.find_all("td")
        if not tds: continue
        lab = re.sub(r"[\s\xa0]*\+?$", "", tds[0].get_text(" ", strip=True)).lower().strip()
        data[lab] = {heads[i + 1]: scr_num(td.get_text(" ", strip=True)) for i, td in enumerate(tds[1:]) if i + 1 < len(heads)}
    return data
def per_fy(h):
    m = re.match(r"([A-Za-z]{3})\s+(\d{4})", h)
    if not m: return None
    y = int(m.group(2)); return y if m.group(1).lower() in ("jan", "feb", "mar", "apr", "may", "jun") else y + 1

SCR_PL = {"revenue_from_operations": ["sales", "revenue"], "other_income": ["other income"], "finance_costs": ["interest"], "depreciation_and_amortisation": ["depreciation"],
          "profit_before_tax": ["profit before tax"], "pat_owners": ["net profit"], "eps_basic": ["eps in rs"]}
SCR_BS = {"total_assets": ["total assets"], "equity_share_capital": ["equity capital"], "inventories": ["inventories"], "trade_receivables": ["trade receivables"],
          "cash_and_cash_equivalents": ["cash equivalents"], "trade_payables": ["trade payables"]}
SCR_CF = {"operating_cash_flow": ["cash from operating activity"], "investing_cash_flow": ["cash from investing activity"], "financing_cash_flow": ["cash from financing activity"]}

def screener_page_ok(soup, c):
    """the Screener page must be for THIS company: its NSE symbol (or, failing that, its title) must match the one resolved from the report."""
    sym, off = META[c].get("symbol"), META[c].get("name")
    txt = soup.get_text(" ", strip=True)
    found = {alnum(x) for x in re.findall(r"NSE\s*:\s*([A-Za-z0-9&\-]+)", txt)}
    for a in soup.find_all("a", href=True):
        h = re.search(r"nseindia\.com[^\"']*symbol=([A-Za-z0-9&%\-]+)", a["href"])
        if h: found.add(alnum(requests.utils.unquote(h.group(1))))
    if sym and found:
        return (alnum(sym) in found), ("" if alnum(sym) in found else f"page is for NSE:{sorted(found)}, expected {sym}")
    h1 = soup.find("h1"); nm = h1.get_text(" ", strip=True) if h1 else ""
    if nm and off and off != c:
        s = name_sim(nm, off); return (s >= 0.6), ("" if s >= 0.6 else f"page title '{nm}' does not match '{off}' ({s:.2f})")
    return False, "page carries no NSE symbol or readable title to verify against"

def screener_search(c):
    q = META[c].get("symbol") or META[c].get("name")
    js, err = http_get("https://www.screener.in/api/company/search/", params={"q": q, "v": 3, "fts": 1}, kind="json")
    slugs = []
    for it in (js if isinstance(js, list) else []):
        m = re.match(r"/company/([^/]+)/", str((it or {}).get("url", "")))
        if m and m.group(1) not in slugs: slugs.append(m.group(1))
    return slugs[:3]

def _screener_try(c, slug):
    url = f"https://www.screener.in/company/{slug}/consolidated/"
    html, err = cached(DIRS["web"] / f"screener_{slug}_consolidated.html", lambda u=url: http_get(u))
    if html is None: return None, f"{slug}: {err}"
    soup = BeautifulSoup(html, "lxml")
    ok, why = screener_page_ok(soup, c)
    if not ok: return None, f"{slug}: rejected, {why}"
    errs, out, ratios = [], {}, {}
    if "Consolidated Figures" not in soup.get_text(" ", strip=True): errs.append(f"{slug}: page is not the consolidated view")
    tabs = {"pl": scr_rows(scr_table(soup, "profit-loss")), "bs": scr_rows(scr_table(soup, "balance-sheet")), "cf": scr_rows(scr_table(soup, "cash-flow"))}
    info = soup.find(id="company-info"); cid = info.get("data-company-id") if info is not None else None
    if cid:
        for parent in ("Other Assets", "Other Liabilities"):
            sch, e2 = cached(DIRS["web"] / f"screener_{slug}_{norm_label(parent).replace(' ', '_')}.json",
                             lambda p=parent, i=cid, u=url: http_get(f"https://www.screener.in/api/company/{i}/schedules/", params={"parent": p, "section": "balance-sheet", "consolidated": ""},
                                                                     headers={"Referer": u, "X-Requested-With": "XMLHttpRequest"}, kind="json"), kind="json")
            if isinstance(sch, dict):
                for lab, vals in sch.items():
                    if isinstance(vals, dict): tabs["bs"].setdefault(lab.lower().strip(), {k: (scr_num(str(v)) if not isinstance(v, (int, float)) else float(v)) for k, v in vals.items()})
    for tk, mp in (("pl", SCR_PL), ("bs", SCR_BS), ("cf", SCR_CF)):
        for var, labs in mp.items():
            for lab in labs:
                if lab in tabs[tk]:
                    for h, v in tabs[tk][lab].items():
                        fy = per_fy(h)
                        if fy and v is not None: out[(fy, var)] = v
                    break
    if "equity capital" in tabs["bs"] and "reserves" in tabs["bs"]:       # v4 convention: equity = capital + reserves
        for h, v in tabs["bs"]["equity capital"].items():
            r = tabs["bs"]["reserves"].get(h); fy = per_fy(h)
            if fy and v is not None and r is not None: out[(fy, "total_equity")] = v + r
    for li in soup.select("#top-ratios li"):
        n = li.select_one(".name"); v = li.select_one(".value")
        if n and v:
            m = re.search(r"-?[\d,]+(?:\.\d+)?", v.get_text(" ", strip=True))
            if m: ratios[n.get_text(strip=True)] = float(m.group(0).replace(",", ""))
    return (out, ratios), "; ".join(errs)

def fetch_screener(c):
    errs, last = [], ({}, {})
    slugs = list(META[c]["screener"])
    if not slugs: return {}, {}, "no verified NSE symbol for this company"
    for round_ in (0, 1):
        for slug in slugs:
            res, e = _screener_try(c, slug)
            if e: errs.append(e)
            if res is not None:
                if res[0]: return res[0], res[1], "; ".join(errs)
                last = res
        if round_ == 0:
            slugs = [s for s in screener_search(c) if s not in META[c]["screener"]]
            if not slugs: break
    return last[0], last[1], "; ".join(errs)

print("O route fetchers defined.")

In [ ]:
# ===== CELL 14 | RUN O ROUTE + MARKET DATA (market data is dated; never mixed with the report period) =====
OWEB, MARKET, PRICE_FY, WEB_LOG = defaultdict(dict), {}, {}, []
RATIOS, YF_SYM = {}, {}
TODAY = dt.date.today().isoformat()
for c in COMPANIES:
    yo, ysym, yerr = fetch_yf(c)
    for (fy, var), v in yo.items(): OWEB[(c, fy, var)]["yfinance"] = v
    so, ratios, serr = fetch_screener(c)
    RATIOS[c] = ratios; YF_SYM[c] = ysym
    for (fy, var), v in so.items(): OWEB[(c, fy, var)]["screener"] = v
    price, psrc, shares_yf = None, None, None
    try:
        if not ysym: raise ValueError("no verified Yahoo series")
        t = yf.Ticker(ysym)
        hist = t.history(start=f"{min(FYS[c]) - 1}-03-01", auto_adjust=False)
        if len(hist):
            price, psrc = float(hist["Close"].iloc[-1]), "yfinance"
            hist.index = pd.to_datetime(hist.index).tz_localize(None)
            PRICE_FY[c] = {fy: float(hist.loc[:f"{fy}-03-31", "Close"].iloc[-1]) for fy in FYS[c] if len(hist.loc[:f"{fy}-03-31"])}
        try: shares_yf = float(t.info.get("sharesOutstanding") or 0) / CR or None
        except Exception: pass
    except Exception as e: WEB_LOG.append({"company": c, "note": f"price: {type(e).__name__}"})
    if price is None and ratios.get("Current Price"): price, psrc = ratios["Current Price"], "screener"
    mcap = ratios.get("Market Cap")
    MARKET[c] = {"price": price, "price_source": psrc, "mcap_cr": mcap, "book_value": ratios.get("Book Value"), "face_value": ratios.get("Face Value") or META[c].get("face_value"),
                 "pe_screener": ratios.get("Stock P/E"), "shares_cr": (mcap / price) if (mcap and price) else shares_yf, "as_of": TODAY}
    WEB_LOG.append({"company": c, "yf_cells": len(yo), "screener_cells": len(so), "yf_err": yerr, "screener_err": serr, "price": price})
display(pd.DataFrame(WEB_LOG)); display(pd.DataFrame(MARKET).T)
print("If Screener shows HTTP 403/empty: open https://www.screener.in/company/<SLUG>/consolidated/ in a browser, Save as HTML into", DIRS["web"],
      "as screener_<SLUG>_consolidated.html and rerun this cell. The notebook then still works with yfinance + the reports.")

In [ ]:
# ===== CELL 14B | MANUAL MARKET FILL (only where the web returned nothing) =====
for c, d in MANUAL_MARKET.items():
    for k, v in d.items():
        if c in MARKET and MARKET[c].get(k) is None: MARKET[c][k] = v
for c, m in MARKET.items():
    if m.get("mcap_cr") is None and m.get("price") and m.get("shares_cr"):
        m["mcap_cr"] = m["price"] * m["shares_cr"]; m["mcap_note"] = "price x yfinance shares outstanding"
display(pd.DataFrame(MARKET).T)

In [ ]:
# ===== CELL 14C | WEB-DATA IDENTITY GATE: a web series is used only if it describes the SAME company as the report =====
# Names can be wrong; numbers cannot. Unit-free fingerprint: total assets / revenue per FY, web vs the report (same company -> same ratio).
GATE_TOL, GATE_MIN_SHARE = 0.30, 0.6
GATE = []
def _pdf_cur(rid, var):
    p = PR.get((rid, var)) or {}
    return p.get("cur") if p.get("status") == "found" else None

def drop_source(c, src):
    for k in [k for k in OWEB if k[0] == c]:
        OWEB[k].pop(src, None)
        if not OWEB[k]: del OWEB[k]

for c in COMPANIES:
    verdict = {}
    for src in ("yfinance", "screener"):
        rel = []
        for fy in FYS[c]:
            rid = f"{c}_{fy}"
            ta_p, rv_p = _pdf_cur(rid, "total_assets"), _pdf_cur(rid, "revenue_from_operations")
            ta_w = (OWEB.get((c, fy, "total_assets")) or {}).get(src); rv_w = (OWEB.get((c, fy, "revenue_from_operations")) or {}).get(src)
            if None in (ta_p, rv_p, ta_w, rv_w) or min(ta_p, rv_p, ta_w, rv_w) <= 0: continue
            rel.append(abs((ta_w / rv_w) / (ta_p / rv_p) - 1))
        has_data = any(k[0] == c and src in v for k, v in OWEB.items())
        if not has_data and not (src == "yfinance" and (MARKET.get(c) or {}).get("price_source") == "yfinance"):
            v_ = "no data"
        elif not rel: v_ = "untestable"
        else:
            share = sum(1 for x in rel if x <= GATE_TOL) / len(rel)
            v_ = "pass" if (share >= GATE_MIN_SHARE and float(np.median(rel)) <= GATE_TOL) else "FAIL"
        verdict[src] = v_
        GATE.append({"company": c, "symbol": META[c].get("symbol"), "source": src, "comparable_FYs": len(rel),
                     "median_ratio_deviation": (round(float(np.median(rel)), 3) if rel else None), "verdict": v_})
    mk = MARKET.setdefault(c, {})
    if verdict.get("screener") == "FAIL":
        drop_source(c, "screener")
        for k in ("mcap_cr", "book_value", "pe_screener"): mk[k] = None
        mk["face_value"] = META[c].get("face_value")
        RATIOS[c] = {}
    if verdict.get("yfinance") == "FAIL":
        drop_source(c, "yfinance"); PRICE_FY.pop(c, None)
        mk.update(price=None, price_source=None, shares_cr=None)
        if verdict.get("screener") != "FAIL" and (RATIOS.get(c) or {}).get("Current Price"):
            mk["price"], mk["price_source"] = RATIOS[c]["Current Price"], "screener"
            if mk.get("mcap_cr"): mk["shares_cr"] = mk["mcap_cr"] / mk["price"]
    mk["web_check"] = ("FAILED: web data discarded" if set(verdict.values()) == {"FAIL"} else
                       ("partly discarded: " + ", ".join(s for s, v in verdict.items() if v == "FAIL") if "FAIL" in verdict.values() else
                        ("passed" if "pass" in verdict.values() else "not testable / no web data")))
GATE_DF = pd.DataFrame(GATE); display(GATE_DF)
GATE_DF.to_csv(DIRS["tables"] / "web_identity_gate.csv", index=False)
failed = GATE_DF[GATE_DF.verdict == "FAIL"]
if len(failed): print("DISCARDED (web series does not match the report, wrong company or wrong page):", failed[["company", "source"]].values.tolist())
else: print("No web series was discarded.")

In [ ]:
# ===== CELL 15 | FROZEN v4 INTEGRATION RULE (P = PDF parser, L = Llama, O = web override) -> VARIABLE TABLE =====
V4_RULE_TEXT = [
 "P = fixed PDF route (status found and a value exists). L = validated Llama extraction (number grounded in the cited raw row, current-year column). O = override.",
 "1. P and L both exist and agree within 0.1% -> use P.",
 "2. Both exist and disagree: if O supports exactly one of them within 1% -> use that one. Otherwise: use L when P came from a flagged row, else keep P and add the cell to REVIEW.",
 "3. Only one of P, L exists: accept it if O is missing or agrees within 1%. If O differs 1% to 10% take O. If O differs by MORE than 10% keep the document value and add to REVIEW.",
 "4. Only O exists -> O. None -> missing."]
RULE_SHA = hashlib.sha256(json.dumps(V4_RULE_TEXT).encode()).hexdigest()
(OUT / "frozen_v4_rule.json").write_text(json.dumps({"sha256": RULE_SHA, "rule": V4_RULE_TEXT, "written": TODAY}, indent=2))

def close(a, b, rel, floor=0.05):
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), floor)

def integrate(p, l, o, pflag):
    if p is not None and l is not None:
        if close(p, l, 0.001): return p, "R1:P=L", "P", "validated", False
        sp = o is not None and close(p, o, 0.01); sl = o is not None and close(l, o, 0.01)
        if sp and not sl: return p, "R2:O supports P", "P", "validated", False
        if sl and not sp: return l, "R2:O supports L", "L", "validated", False
        if pflag: return l, "R2:P flagged -> L", "L", "warning", False
        return p, "R2:keep P (REVIEW)", "P", "warning", True
    d = p if p is not None else l; src = "P" if p is not None else "L"
    if d is not None:
        if o is None: return d, "R3:document only", src, "unverified", False
        if close(d, o, 0.01): return d, "R3:O agrees", src, "validated", False
        if abs(d - o) / max(abs(d), abs(o), 1e-9) <= 0.10: return o, "R3:O differs 1-10% -> O", "O", "warning", False
        return d, "R3:O implausible >10% keep document (REVIEW)", src, "warning", True
    if o is not None: return o, "R4:O only", "O", "unverified", False
    return None, "none", "", "missing", False

O_VOTE = {"total_assets", "total_equity", "current_assets", "current_liabilities", "inventories", "trade_receivables", "trade_payables", "cash_and_cash_equivalents",
          "revenue_from_operations", "finance_costs", "depreciation_and_amortisation", "profit_before_tax", "pat", "operating_cash_flow", "investing_cash_flow",
          "financing_cash_flow", "capex", "dividends_paid", "eps_basic"}
O_SCR_FIRST = {"finance_costs", "depreciation_and_amortisation", "profit_before_tax", "revenue_from_operations"}
# web definition differs from the document line (Screener "Interest" != Finance costs; web EPS is split-adjusted):
# keep the web value for display, but never let it override or flag a document value
DEF_DIFF = {"finance_costs", "eps_basic"}
# lines a company simply does not print when the amount is nil: no source -> 0 (not a REVIEW item)
OPT_NIL = {"purchases_stock", "exceptional_items"}

def o_value(c, fy, var):
    d = OWEB.get((c, fy, var))
    if not d: return None, None, False
    vals = [(s, d[s]) for s in (["screener", "yfinance"] if var in O_SCR_FIRST else ["yfinance", "screener"]) if d.get(s) is not None]
    if not vals: return None, None, False
    return vals[0][1], vals[0][0], (len(vals) == 2 and not close(vals[0][1], vals[1][1], 0.01))

def unit_factor(rid, kind):
    st = STMT.get((rid, kind)); c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]
    if st is None: return None, "none"
    if st["unit"]: return st["unit"], "detected"
    sib = {STMT[(r2, kind)]["unit"] for r2 in REPORTS if REPORTS[r2]["company"] == c and (r2, kind) in STMT and STMT[(r2, kind)]["unit"]}
    if len(sib) == 1: return sib.pop(), "same unit as the company's other reports"
    rs = []
    for var in VAR_BY_KIND[kind]:
        if FIELD_SPEC[var]["scale"] != "crore" or var not in O_VOTE: continue
        p = PR[(rid, var)]; o = o_value(c, fy, var)[0]
        if p.get("status") == "found" and p.get("cur") and o and p["cur"] * o > 0: rs.append(o / p["cur"])
    if not rs: return None, "unknown"
    r = float(np.median(rs)); best = min([1, 0.01, 0.1, 1e-7, 1e-4, 100], key=lambda f: abs(math.log(r / f)))
    return (best, "inferred from web data") if abs(r / best - 1) < 0.05 else (None, "unknown")

VT_ROWS, PRIORCOL = [], {}
first_rid = {c: min((r for r in REPORTS if REPORTS[r]["company"] == c), key=lambda r: REPORTS[r]["fy"]) for c in COMPANIES}
for rid in sorted(REPORTS):
    c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]; ufs = {k: unit_factor(rid, k) for k in ("bs", "pl", "cf")}
    for var, sp in FIELD_SPEC.items():
        fac, fsrc = ufs[sp["kind"]]; p = PR[(rid, var)]; l = LR.get((rid, var), {"status": "na"})
        sc = (lambda x: None if x is None else (x if sp["scale"] == "rupee" else (x * fac if fac is not None else None)))
        lv = sc(l.get("cur")) if l.get("status") == "found" else None
        pv = ((None if lv is not None else 0.0) if p.get("status") == "nil_row_absent" else (sc(p.get("cur")) if p.get("status") == "found" else None))
        ov, osrc, odis = o_value(c, fy, var) if var in O_VOTE else (None, None, False)
        ov_use = None if (var in DEF_DIFF and (pv is not None or lv is not None)) else ov

        val, branch, src, vs, rev = integrate(pv, lv, ov_use, p.get("flag", False))
        # P sums several rows by design (payables sub-rows, NCL+current leases, A + B cash-flow lines); L reads a single row
        if pv is not None and lv is not None and lv > 0 and not close(pv, lv, 0.001):
            if var in {"capex", "dividends_paid", "acquisitions", "repayment_borrowings", "proceeds_borrowings"} and close(pv, 2 * lv, 0.001):
                val, branch, src, vs, rev = lv, "R2:P = 2 x L (double count) -> L", "L", "warning", False
            elif (var in {"trade_payables", "lease_liabilities"} or " + " in str(p.get("label") or "")) and lv < pv:
                val, branch, src, vs, rev = pv, "R2:P sums several rows, L single row -> P", "P", "warning", False

        if var in DEF_DIFF and ov is not None and ov_use is None: branch += " | web definition differs (web value not used)"
        if val is None and var in OPT_NIL:
            val, branch, src, vs, rev = 0.0, "nil: line not printed", "nil", "unverified", False
        elif p.get("status") == "nil_row_absent" and lv is None:
            vs, branch = "warning", branch + " | no line printed: treated as nil"
        if odis and vs == "validated": vs = "warning"; branch += " | web sources disagree"
        st = STMT.get((rid, sp["kind"])); basis = SEL[rid]["basis"].get(sp["kind"])
        vstat = ("nil" if branch.startswith("nil:") else
                 "derived" if str(p.get("label", "")).endswith("(derived)") else
                 ("estimated" if src == "O" and val is not None else "reported"))
        VT_ROWS.append({"company": c, "fy": fy, "variable": var, "value": val, "unit": "INR" if sp["scale"] == "rupee" else "INR crore", "method": src, "rule_branch": branch,
                        "validation_status": vs, "value_status": vstat,
                        "actual_or_forecast": "actual", "reporting_basis": basis, "original_unit": (st or {}).get("unit_txt") or fsrc, "unit_source": fsrc,
                        "p_value": pv, "l_value": lv, "o_value": ov, "p_status": p.get("status"),
                        "source_document": REPORTS[rid]["path"].name, "source_page": p.get("page") or l.get("page"),
                        "source_label": p.get("label"), "review": rev})
        if rid == first_rid[c] and p.get("status") in ("found", "nil_row_absent") and p.get("prior") is not None:
            pp = 0.0 if p.get("status") == "nil_row_absent" else sc(p["prior"])
            if pp is not None: PRIORCOL[(c, fy - 1, var)] = pp
for c in COMPANIES:                                      # O-only variable used for P/E on owners' profit
    for fy in FYS[c]:
        ov, osrc, _ = o_value(c, fy, "pat_owners")
        if ov is not None:
            VT_ROWS.append({"company": c, "fy": fy, "variable": "pat_owners", "value": ov, "unit": "INR crore", "method": "O", "rule_branch": "O-only variable",
                            "validation_status": "unverified", "value_status": "estimated", "actual_or_forecast": "actual", "reporting_basis": BASIS, "original_unit": "web", "unit_source": osrc,
                            "p_value": None, "l_value": None, "o_value": ov, "p_status": None, "source_document": osrc, "source_page": None,
                            "source_label": "net profit attributable to owners", "review": False})
VT = pd.DataFrame(VT_ROWS)
VT["restatement_status"] = "not_checked"
print("Variable table:", VT.shape, "| rule SHA", RULE_SHA[:16])
display(VT.groupby(["validation_status"]).size().to_frame("cells").T)
display(VT.groupby("method").size().to_frame("cells").T)
print("REVIEW cells:", int(VT.review.sum()), "| missing cells:", int((VT.validation_status == "missing").sum()),
      "| nil_not_printed cells:", int((VT.rule_branch == "nil: line not printed").sum()))

In [ ]:
# ===== CELL 15A | HELPERS: vt_get + ONE tolerance rule (QA cells only) =====
# Accept a difference between a and b using the PROTOTYPE tolerance rule, unchanged:
#   INR crore variables : exact -> "exact"; gap <= 0.1% of the larger magnitude OR gap <= rounding floor -> "within_tolerance";
#                         gap <= 1% -> "near"; otherwise "mismatch".  Rounding floor = 0.5 x precision(A) + 0.5 x precision(B), in crore.
#   other variables (EPS, per-share, ratios): same classes with 0.1% / 1%, no rounding floor.
# Precision is MEASURED from the data exactly as in the prototype: the smallest of 1, 0.1, 0.01, 0.001 that >= 95% of a
# source's values are a multiple of (fewer than 5 values -> default). Two pools here: "doc" (PDF / Llama values) and "web" (override values).
# Accepted pair (exact or within_tolerance) = the PDF (document) value stays. On any real mismatch the PDF value is used; the web value is informational only.
# The frozen source rule in Cell 15 decides what goes into VT; this tolerance only flags.
import pandas as pd
from decimal import Decimal

TOL_PCT, NEAR_PCT = Decimal("0.1"), Decimal("1.0")
PREC_T = {"doc": Decimal("0.01"), "web": Decimal("1")}      # defaults (prototype DEFAULT_PREC style); replaced by measured values below
_D = lambda x: Decimal(repr(float(x)))

def tol_class(a, b, unit="INR crore", sa="doc", sb="doc"):
    a, b = _D(a), _D(b); gap = abs(a - b)
    floor = Decimal("0.5") * PREC_T[sa] + Decimal("0.5") * PREC_T[sb] if unit == "INR crore" else Decimal(0)
    denom = max(abs(a), abs(b)); rel = None if denom == 0 else gap / denom * 100
    if gap == 0: return "exact"
    if gap <= floor or (rel is not None and rel <= TOL_PCT): return "within_tolerance"
    if rel is not None and rel <= NEAR_PCT: return "near"
    return "mismatch"

def within_tol(a, b, unit="INR crore", sa="doc", sb="doc"):
    if a is None or b is None or pd.isna(a) or pd.isna(b): return False
    return tol_class(a, b, unit, sa, sb) in ("exact", "within_tolerance")

def observe_precision(vals):
    vals = [_D(v) for v in vals if v is not None and not pd.isna(v)]
    if len(vals) < 5: return None
    for p in (Decimal(1), Decimal("0.1"), Decimal("0.01"), Decimal("0.001")):
        if sum(1 for v in vals if v % p == 0) / len(vals) >= 0.95: return p
    return Decimal(0)

def vt_get(c, fy, var):
    s = VT[(VT.company == c) & (VT.fy == fy) & (VT.variable == var)]
    return None if s.empty or pd.isna(s.value.iloc[0]) else float(s.value.iloc[0])

# self-test (no data needed; uses the default precisions)
assert tol_class(92100.58, 92100.58) == "exact" and tol_class(50000, 50030) == "within_tolerance"      # exact; 0.06% <= 0.1%
assert within_tol(40.6, 41, "INR crore", "doc", "web") and not within_tol(40.6, 42, "INR crore", "doc", "web")   # rounding floor
assert tol_class(100.9, 100.0) == "near" and tol_class(100.0, 120.0) == "mismatch"
assert within_tol(19.48, 19.49, "other") and not within_tol(7.04, 7.05, "other")   # per-share: 0.1%

# measure precision from the data (as the prototype does), if the variable table exists
if "VT" in globals():
    _cr = VT[VT.unit == "INR crore"]
    _doc = list(_cr.p_value.dropna()) + list(_cr.l_value.dropna()); _web = list(_cr.o_value.dropna())
    for _k, _v in (("doc", observe_precision(_doc)), ("web", observe_precision(_web))):
        if _v is not None: PREC_T[_k] = _v
    print("measured precision (crore):", {k: str(v) for k, v in PREC_T.items()})
print("helpers ready | INR crore = 0.1% or rounding floor, near <= 1% (prototype rule) | other variables = 0.1% | PDF value wins on mismatch")


In [ ]:
# ===== CELL 15B (v4, generic) | next report's prior-year column wins for trade payables when it equals the web value and the first report does not =====
# (replaces the two hand-audited overrides: the same evidence rule is now applied to every company)
OVR_LOG = []
for c in COMPANIES:
    for fy in FYS[c][:-1]:
        rid, nrid, var = f"{c}_{fy}", f"{c}_{fy + 1}", "trade_payables"
        if nrid not in REPORTS or rid not in REPORTS: continue
        pr = PR.get((nrid, var), {}); sp = FIELD_SPEC[var]
        if pr.get("status") != "found" or pr.get("prior") is None: continue
        fac = unit_factor(nrid, sp["kind"])[0]
        if not fac: continue
        prior_v, first, web = pr["prior"] * fac, vt_get(c, fy, var), o_value(c, fy, var)[0]
        if first is None or web is None: continue
        if (not within_tol(first, prior_v)) and close(prior_v, web, 0.01) and not close(first, web, 0.01):
            m = (VT.company == c) & (VT.fy == fy) & (VT.variable == var)
            VT.loc[m, "value"] = prior_v; VT.loc[m, "method"] = "P_next_prior"
            VT.loc[m, "rule_branch"] = f"override: FY{fy + 1} report restated the prior year to {prior_v:,.2f} (= web value {web:,.2f}; first report {first:,.2f})"
            VT.loc[m, "validation_status"] = "validated"; VT.loc[m, "review"] = False; VT.loc[m, "value_status"] = "reported_restated"
            VT.loc[m, "source_document"] = REPORTS[nrid]["path"].name; VT.loc[m, "source_page"] = pr.get("page")
            OVR_LOG.append((c, fy, var, first, prior_v))
print("trade_payables overrides from the next report's prior column:", len(OVR_LOG))
if OVR_LOG: display(pd.DataFrame(OVR_LOG, columns=["company", "fy", "variable", "was", "now"]))

In [ ]:
# ===== CELL 15C | net_borrowings = sum of all financing-section borrowing lines (gross/net presentation independent) =====
EXCL = r"interest|finance charge|lease|cost|issue|share|dividend"
nb_rows, NB = [], {}
for rid in sorted(REPORTS):
    if (rid, "cf") not in STMT: continue
    rows, K = _rows(rid, "cf"), _Kk(rid, "cf"); fac = unit_factor(rid, "cf")[0]
    cur_sum = pri_sum = 0.0; used = []
    for r in rows:
        lab = str(r.get("label") or "").lower()
        if r.get("section") != "FIN" or "borrowing" not in lab or re.search(EXCL, lab): continue
        w = (r.get("nums") or [])[-K:]
        if len(w) < K: continue
        c0 = 0.0 if _isdash(w[0]) else parse_num(w[0])
        p0 = 0.0 if (K < 2 or _isdash(w[1])) else parse_num(w[1])
        if c0 is None: continue
        cur_sum += c0; pri_sum += (p0 or 0.0); used.append(str(r.get("label"))[:45])
    if fac is None: continue
    NB[rid] = (cur_sum * fac, pri_sum * fac, used)
    c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]
    nb_rows.append({"company": c, "fy": fy, "variable": "net_borrowings", "value": cur_sum * fac, "unit": "INR crore", "method": "P_sum",
                    "rule_branch": "sum of financing-section borrowing lines (net of repayments)", "validation_status": "unverified",
                    "value_status": "derived", "actual_or_forecast": "actual", "source_document": REPORTS[rid]["path"].name,
                    "source_label": " + ".join(used) if used else "no borrowing line printed", "review": False})
VT = pd.concat([VT[VT.variable != "net_borrowings"], pd.DataFrame(nb_rows)], ignore_index=True)

# consistency: FY value vs the next report's prior column
bad = []
for rid in NB:
    c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]; nrid = f"{c}_{fy + 1}"
    if nrid in NB and not close(NB[rid][0], NB[nrid][1], 0.02, floor=0.2): bad.append((c, fy, round(NB[rid][0], 2), round(NB[nrid][1], 2)))
print("net_borrowings rows:", len(nb_rows), "| inconsistent with next report's prior column:", len(bad))
for b in bad: print("  CHECK", *b)
display(VT[VT.variable == "net_borrowings"].pivot(index="company", columns="fy", values="value").round(2))

In [ ]:
# ===== CELL 15C-fix (v3, generic) | net_borrowings: use the NEXT report's prior column where the gap is outside tolerance AND a lease-proceeds line explains it =====
def fin_rows(rid, idx):
    K, fac = _Kk(rid, "cf"), unit_factor(rid, "cf")[0]; out = []
    for r in _rows(rid, "cf"):
        lab = str(r.get("label") or ""); w = (r.get("nums") or [])[-K:]
        if r.get("section") == "FIN" and re.search(r"borrow|lease", lab, re.I) and len(w) >= K and fac:
            v = parse_num(w[idx]); out.append((lab, None if v == "DASH" or v is None else round(v * fac, 2)))
    return out

NB_FIX, NB_OPEN = [], []
for c in COMPANIES:
    for fy in FYS[c][:-1]:
        rid0, rid1 = f"{c}_{fy}", f"{c}_{fy + 1}"
        if rid0 not in NB or rid1 not in NB: continue
        first, restated = NB[rid0][0], NB[rid1][1]
        if within_tol(first, restated): continue
        lease = [v for lab, v in fin_rows(rid1, 1) if re.search(r"lease", lab, re.I) and not re.search(r"interest", lab, re.I) and re.search(r"proceeds|repayment", lab, re.I) and v is not None]
        lease_prior = sum(lease) if lease else None
        explained = lease_prior is not None and abs((first - restated) - lease_prior) <= max(0.02, 0.1 * abs(first - restated))
        if explained:
            m = (VT.company == c) & (VT.fy == fy) & (VT.variable == "net_borrowings")
            VT.loc[m, "value"] = restated; VT.loc[m, "method"] = "P_next_prior"; VT.loc[m, "validation_status"] = "validated"
            VT.loc[m, "value_status"] = "reported_restated"; VT.loc[m, "source_document"] = REPORTS[rid1]["path"].name
            VT.loc[m, "rule_branch"] = "next report prior column: lease proceeds shown separately, so net_borrowings excludes leases"
            NB_FIX.append((c, fy, round(first, 2), round(restated, 2)))
        else:
            NB_OPEN.append((c, fy, round(first, 2), round(restated, 2), lease_prior))
print(f"net_borrowings: {len(NB_FIX)} year(s) taken from the next report's prior column (lease line explains the gap) | {len(NB_OPEN)} gap(s) outside tolerance and NOT explained (first-reported value kept)")
if NB_FIX: display(pd.DataFrame(NB_FIX, columns=["company", "fy", "first reported", "next report prior"]))
if NB_OPEN: display(pd.DataFrame(NB_OPEN, columns=["company", "fy", "first reported", "next report prior", "lease line in next report"]))
display(VT[VT.variable == "net_borrowings"].pivot(index="company", columns="fy", values="value").round(2))

In [ ]:
# ===== CELL 15D | (1) PBT: pick the "profit before tax" row that satisfies PBT - tax = PAT  (2) DATA FY22 wrapped change_inventories =====
import re
def _rows(rid, kind):
    obj = STMT.get((rid, kind)); return [] if obj is None else (obj["rows"] if isinstance(obj, dict) else obj)
def _Kk(rid, kind):
    obj = STMT[(rid, kind)]; return (obj.get("K") or 2) if isinstance(obj, dict) else 2
def _vt(c, fy, var):
    s = VT[(VT.company == c) & (VT.fy == fy) & (VT.variable == var)]
    return None if s.empty or pd.isna(s.value.iloc[0]) else float(s.value.iloc[0])

LOG15D = []
# ---- (1) some P&Ls print two "Profit before tax" lines (before / after exceptional items). The right one satisfies PBT - tax = PAT.
for c in COMPANIES:
    for fy in FYS[c]:
        rid = f"{c}_{fy}"
        if (rid, "pl") not in STMT: continue
        pbt, tax, pat = _vt(c, fy, "profit_before_tax"), _vt(c, fy, "tax_expense"), _vt(c, fy, "pat")
        if None in (pbt, tax, pat): continue
        m = (VT.company == c) & (VT.fy == fy) & (VT.variable == "profit_before_tax")
        tol = 0.01 * max(abs(pat), 1)
        bad_identity = abs((pbt - tax) - pat) > tol
        from_web = (VT.loc[m, "method"].iloc[0] == "O")
        if not (bad_identity or from_web): continue
        rows, K = _rows(rid, "pl"), _Kk(rid, "pl"); fac = unit_factor(rid, "pl")[0]
        if not fac: continue
        cands = []
        for r in rows:
            lab = str(r.get("label") or "").lower()
            if re.search(r"profit.*before.*tax", lab) and not re.search(r"discontinu|share of|associate", lab):
                n = r.get("nums") or []
                v = parse_num(n[-K]) if len(n) >= K else None
                if v is not None: cands.append((v * fac, r.get("page"), str(r.get("label"))))
        ok = [x for x in cands if abs((x[0] - tax) - pat) <= tol and abs(x[0] - pbt) > 1e-6]
        if ok:
            v, pg, lab = ok[-1]
            VT.loc[m, "value"] = v; VT.loc[m, "method"] = "P_alt_row"; VT.loc[m, "source_page"] = pg
            VT.loc[m, "rule_branch"] = f"identity: row '{lab[:40]}' satisfies PBT - tax = PAT"
            VT.loc[m, "validation_status"] = "validated"; VT.loc[m, "review"] = False; VT.loc[m, "value_status"] = "reported"
            LOG15D.append((c, fy, "profit_before_tax", pbt, v, lab[:45]))
        elif bad_identity:
            LOG15D.append((c, fy, "profit_before_tax", pbt, None, "identity fails, no alternative row (associates/NCI?)"))

# ---- (2) wrapped label: "(b) Changes in inventories" prints no numbers, they sit on the next "(unlabeled sub-total)" row
for rid in sorted(REPORTS):
    if (rid, "pl") not in STMT: continue
    c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]
    if _vt(c, fy, "change_inventories") is not None: continue
    rows, K = _rows(rid, "pl"), _Kk(rid, "pl"); fac = unit_factor(rid, "pl")[0]
    if not fac: continue
    i = next((j for j, r in enumerate(rows) if re.search(r"(changes?|increase|decrease).{0,25}in inventories", str(r.get("label") or "").lower()) and not r.get("nums")), None)
    if i is not None and i + 1 < len(rows) and "unlabeled" in str(rows[i + 1].get("label") or "").lower() and len(rows[i + 1].get("nums") or []) >= K:
        nx = rows[i + 1]; w = nx["nums"][-K:]; v = parse_num(w[0])
        if v in (None, "DASH"): continue
        v = v * fac; m = (VT.company == c) & (VT.fy == fy) & (VT.variable == "change_inventories")
        VT.loc[m, "value"] = v; VT.loc[m, "method"] = "P_wrapped"; VT.loc[m, "source_page"] = nx.get("page")
        VT.loc[m, "rule_branch"] = "wrapped label: numbers on the next unlabeled row"
        VT.loc[m, "validation_status"] = "unverified"; VT.loc[m, "review"] = False; VT.loc[m, "value_status"] = "reported"
        LOG15D.append((c, fy, "change_inventories", None, v, str(rows[i].get("label"))[:45]))

display(pd.DataFrame(LOG15D, columns=["company", "fy", "variable", "was", "now", "row"]))
print("REVIEW:", int(VT.review.sum()), "| missing:", int((VT.validation_status == "missing").sum()))

In [ ]:
# ===== CELL 15E | (1) cash-flow lines from ONE document version  (2) audit of every cell taken from the web =====
CF4 = ["operating_cash_flow", "investing_cash_flow", "financing_cash_flow", "capex"]
m = (VT.method == "O") & VT.variable.isin(CF4) & (VT.p_value.notna() | VT.l_value.notna())
fix = []
for i in VT[m].index:
    r = VT.loc[i]; newv = r.p_value if pd.notna(r.p_value) else r.l_value
    fix.append((r.company, r.fy, r.variable, r.value, newv))
    VT.loc[i, "value"] = newv; VT.loc[i, "method"] = "P" if pd.notna(r.p_value) else "L"
    VT.loc[i, "rule_branch"] = "restored: document value (web figure is the restated version; cash-flow lines must come from one report)"
    VT.loc[i, "validation_status"] = "warning"; VT.loc[i, "value_status"] = "reported"; VT.loc[i, "review"] = False
print("cash-flow cells restored to the document value:", len(fix))
display(pd.DataFrame(fix, columns=["company", "fy", "variable", "was (web)", "now (document)"]))

# audit: every other cell where the web value replaced a document value (frozen rule R3: 1-10%)
aud = VT[(VT.method == "O") & (VT.variable != "pat_owners")][["company", "fy", "variable", "value", "p_value", "l_value", "o_value", "rule_branch"]]
print("\nOther cells taken from the web (check each: document value vs web value):", len(aud)); display(aud)

# independent test of the cash-flow triple: CFO + CFI + CFF = change in cash and cash equivalents
rows_ = []
for c in COMPANIES:
    for fy in FYS[c][1:]:
        t = [vt_get(c, fy, v) for v in ("operating_cash_flow", "investing_cash_flow", "financing_cash_flow")]
        c1, c0 = vt_get(c, fy, "cash_and_cash_equivalents"), vt_get(c, fy - 1, "cash_and_cash_equivalents")
        if None in t or c1 is None or c0 is None: continue
        d_cf, d_bs = sum(t), c1 - c0
        rows_.append({"company": c, "fy": fy, "sum_CF": round(d_cf, 1), "change_in_cash_BS": round(d_bs, 1), "gap": round(d_cf - d_bs, 1),
                      "gap_ok": abs(d_cf - d_bs) <= max(0.01 * max(abs(d_cf), abs(d_bs)), 1.0)})
cr = pd.DataFrame(rows_, columns=["company", "fy", "sum_CF", "change_in_cash_BS", "gap", "gap_ok"]); cr["gap_ok"] = cr["gap_ok"].astype(bool)
print("\nCash roll-forward (informational: bank overdrafts or a different cash definition can create gaps):")
display(cr[~cr.gap_ok] if (~cr.gap_ok).any() else cr.head(0))
print("company-years checked:", len(cr), "| ok:", int(cr.gap_ok.sum()))

In [ ]:
# ===== CELL 16 | QA: restatement check (found rows only, cash-neutral aware), sanity checks, REVIEW list, hand-check sample =====

CF_VARS = {"operating_cash_flow", "investing_cash_flow", "financing_cash_flow", "capex"}
GROSS_NET = {"repayment_borrowings", "proceeds_borrowings"}      # gross vs net presentation differs: use net_borrowings instead

def cf_neutral(c, fy):
    nrid = f"{c}_{fy + 1}"; vs = ("operating_cash_flow", "investing_cash_flow", "financing_cash_flow"); fac = unit_factor(nrid, "cf")[0]
    a = [vt_get(c, fy, v) for v in vs]
    b = [(PR[(nrid, v)]["prior"] * fac if PR[(nrid, v)].get("status") == "found" and PR[(nrid, v)].get("prior") is not None and fac else None) for v in vs]
    return None not in a and None not in b and close(sum(a), sum(b), 0.01, floor=1.0)

flag_rows, info_rows, n_skipped = [], [], 0
for c in COMPANIES:
    for fy in FYS[c][:-1]:
        nrid = f"{c}_{fy + 1}"
        if nrid not in REPORTS: continue
        for var, sp in FIELD_SPEC.items():
            if var in GROSS_NET: continue
            pr = PR[(nrid, var)]
            if pr.get("status") != "found" or pr.get("prior") is None:
                n_skipped += 1; continue
            fac = unit_factor(nrid, sp["kind"])[0]
            pv = pr["prior"] * (1 if sp["scale"] == "rupee" else (fac if fac is not None else float("nan")))
            cv = vt_get(c, fy, var)
            if cv is None or pd.isna(pv): continue
            m = (VT.company == c) & (VT.fy == fy) & (VT.variable == var)
            if close(cv, pv, 0.01):
                VT.loc[m, "restatement_status"] = "consistent"; continue
            rel = abs(cv - pv) / max(abs(cv), abs(pv), 1e-9)
            rec = {"company": c, "fy": fy, "variable": var, "as_first_reported": cv, "in_next_report": pv, "rel_diff_%": round(100 * rel, 2)}
            if (var in CF_VARS and (rel < 0.05 or cf_neutral(c, fy))) or rel < 0.03:
                rec["why"] = "cash-neutral reclass" if var in CF_VARS else "small restatement (<3%)"
                VT.loc[m, "restatement_status"] = "restated_info"; info_rows.append(rec)
            else:
                VT.loc[m, "restatement_status"] = "restated_flag"; flag_rows.append(rec)
print(f"Restatement check: flags {len(flag_rows)} | informational {len(info_rows)} | not comparable (no row in next report) {n_skipped}")
print("\nFLAGS (real differences: check page of the first report and the next report's prior column):")
display(pd.DataFrame(flag_rows))
print("INFORMATIONAL:"); display(pd.DataFrame(info_rows))

checks = []
for c in COMPANIES:
    for fy in FYS[c]:
        g = lambda v: vt_get(c, fy, v)
        ta, ca, cl, rev, pbt, tax, pat = g("total_assets"), g("current_assets"), g("current_liabilities"), g("revenue_from_operations"), g("profit_before_tax"), g("tax_expense"), g("pat")
        if ta and ca and ca > ta * 1.001: checks.append((c, fy, "current assets > total assets"))
        if ta and cl and cl > ta: checks.append((c, fy, "current liabilities > total assets"))
        if rev is not None and rev <= 0: checks.append((c, fy, "revenue <= 0"))
        if None not in (pbt, tax, pat) and abs((pbt - tax) - pat) > 0.03 * max(abs(pat), 1): checks.append((c, fy, f"PBT - tax ({pbt - tax:,.0f}) != PAT ({pat:,.0f}) (associates / NCI / exceptional / wrong tax row?)"))
        if g("total_equity") is not None and g("total_equity") <= 0: checks.append((c, fy, "equity <= 0"))
        if ta and g("total_equity") and g("total_equity") > ta: checks.append((c, fy, "equity > total assets"))
        for v in ("total_assets", "revenue_from_operations", "pat", "operating_cash_flow", "total_equity", "current_liabilities", "equity_share_capital"):
            if g(v) is None: checks.append((c, fy, f"MISSING {v}"))
print("\nSanity-check findings:", len(checks))
for x in checks: print(" -", *x)

REVIEW = VT[VT.review | (VT.validation_status == "missing")][["company", "fy", "variable", "value", "p_value", "l_value", "o_value", "rule_branch", "source_page"]]
print("\nREVIEW LIST:", len(REVIEW)); display(REVIEW)

pool = VT[VT.validation_status.isin(["unverified", "warning"]) & VT.value.notna() & VT.source_page.notna() & (VT.value_status != "nil")]
unl = pool[pool.source_label.isna()]
nu = min(8, len(unl)); rest = pool.drop(unl.index)
smp = pd.concat([unl.sample(nu, random_state=7), rest.sample(min(15 - nu, len(rest)), random_state=7)])
smp = smp[["company", "fy", "variable", "value", "unit", "validation_status", "source_document", "source_page", "source_label"]]
print("\nHAND-CHECK SAMPLE (15 cells; unlabeled rows first):"); display(smp)

VT.to_csv(DIRS["tables"] / "variable_table_raw.csv", index=False)
pd.DataFrame(flag_rows + info_rows).to_csv(DIRS["tables"] / "restatement_check.csv", index=False)

In [ ]:
# ===== CELL 16B | P&L identity: revenue + other income - total expenses - exceptional ~ PBT ; components <= total expenses =====
rows_ = []
for c in COMPANIES:
    for fy in FYS[c]:
        g = lambda v: vt_get(c, fy, v)
        rev, oi, te, pbt = g("revenue_from_operations"), g("other_income"), g("total_expenses"), g("profit_before_tax")
        if None in (rev, oi, te, pbt): continue
        exc = g("exceptional_items") or 0.0
        inc = rev + oi; gap = inc - te - exc - pbt
        comp = sum((g(v) or 0.0) for v in ("cost_of_materials", "purchases_stock", "change_inventories", "employee_benefit_expenses", "finance_costs", "depreciation_and_amortisation"))
        rows_.append({"company": c, "fy": fy, "income": round(inc, 1), "total_exp": round(te, 1), "pbt": round(pbt, 1), "gap": round(gap, 1),
                      "gap_%income": round(100 * gap / inc, 2), "components": round(comp, 1), "components_le_total": comp <= te * 1.001})
idc = pd.DataFrame(rows_)
print("P&L identity check on", len(idc), "company-years; failures (|gap| > 0.5% of income or components > total):")
display(idc[(idc["gap_%income"].abs() > 0.5) | (~idc.components_le_total)])

In [ ]:
# ===== CELL 16D (generic) | PBT flagged as restated: does the next report's prior column carry a "profit before tax" row equal to ours? (before vs after exceptional items) =====
RES16D = []
for i, r in VT[(VT.variable == "profit_before_tax") & (VT.restatement_status == "restated_flag")].iterrows():
    c, fy = r.company, int(r.fy); nrid = f"{c}_{fy + 1}"
    if (nrid, "pl") not in STMT: continue
    K, fac = _Kk(nrid, "pl"), unit_factor(nrid, "pl")[0]
    if not fac: continue
    ours = vt_get(c, fy, "profit_before_tax"); hits = []
    for row in _rows(nrid, "pl"):
        lab = str(row.get("label") or ""); w = (row.get("nums") or [])[-K:]
        if len(w) < K or K < 2 or not re.search(r"profit.*before.*tax", lab, re.I) or re.search(r"discontinu|share of|associate", lab, re.I): continue
        v = parse_num(w[1])
        if v not in (None, "DASH"): hits.append(v * fac)
    if ours is not None and any(close(ours, h, 0.005) for h in hits):
        base = str(VT.at[i, "rule_branch"]).split(" | presentation:")[0]
        VT.at[i, "rule_branch"] = base + " | next report prior column has a profit-before-tax row equal to ours (the flag compared a different PBT line)"
        VT.at[i, "restatement_status"] = "consistent"; RES16D.append((c, fy, round(ours, 2), "resolved"))
    else: RES16D.append((c, fy, None if ours is None else round(ours, 2), "still flagged"))
print("PBT restatement flags re-checked:", len(RES16D));
if RES16D: display(pd.DataFrame(RES16D, columns=["company", "fy", "our PBT", "result"]))
left = VT[VT.restatement_status == "restated_flag"][["company", "fy", "variable"]]
print("flags left:", len(left)); display(left)

In [ ]:
# ===== CELL 17 | DERIVED METRICS (all arithmetic in Python; formulas follow the chart spec) =====
def pct_change(cur, prev):
    """None if prior value is zero/negative or missing; cur may be negative (profit -> loss shows a negative %)."""
    if cur is None or prev is None or pd.isna(cur) or pd.isna(prev) or prev <= 0: return np.nan
    return (cur - prev) / prev * 100.0
def ratio(a, b, mult=1.0):
    return np.nan if (a is None or b is None or pd.isna(a) or pd.isna(b) or b == 0) else a / b * mult
def cagr(first, last, n):
    return np.nan if (first is None or last is None or pd.isna(first) or pd.isna(last) or first <= 0 or last <= 0 or n <= 0) else ((last / first) ** (1.0 / n) - 1) * 100.0

RAW = {c: VT[VT.company == c].pivot_table(index="fy", columns="variable", values="value", aggfunc="first").sort_index() for c in COMPANIES}
STATUS = {c: VT[VT.company == c].pivot_table(index="fy", columns="variable", values="validation_status", aggfunc="first").sort_index() for c in COMPANIES}

def build_metrics(c):
    R = RAW[c]; fys = list(R.index); M = pd.DataFrame(index=fys); notes = []
    g = lambda fy, v: (R.at[fy, v] if (v in R.columns and fy in R.index and pd.notna(R.at[fy, v])) else np.nan)
    def opening(fy, v):
        if (fy - 1) in R.index and v in R.columns and pd.notna(R.at[fy - 1, v]): return R.at[fy - 1, v]
        return PRIORCOL.get((c, fy - 1, v), np.nan)
    def avg(fy, v):
        o, cl_ = opening(fy, v), g(fy, v)
        if pd.isna(cl_): return np.nan
        if pd.isna(o): notes.append(f"{c} FY{fy}: average {v} uses closing balance only"); return cl_
        return (o + cl_) / 2
    lease = INCLUDE_LEASES_IN_DEBT
    for fy in fys:
        rev, pbt, fin, da = g(fy, "revenue_from_operations"), g(fy, "profit_before_tax"), g(fy, "finance_costs"), g(fy, "depreciation_and_amortisation")
        fin0 = 0.0 if pd.isna(fin) and pd.notna(pbt) else fin
        ebit = pbt + (0.0 if pd.isna(fin) else fin) if pd.notna(pbt) else np.nan          # locked v4 definition: EBIT = PBT + finance costs (includes other income and exceptional items)
        ebitda = ebit + da if pd.notna(ebit) and pd.notna(da) else np.nan
        debt = sum(0.0 if pd.isna(g(fy, v)) else g(fy, v) for v in ("borrowings_noncurrent", "borrowings_current", "borrowings_current_maturities") + (("lease_liabilities",) if lease else ()))
        have_debt = any(pd.notna(g(fy, v)) for v in ("borrowings_noncurrent", "borrowings_current", "borrowings_current_maturities"))
        debt = debt if have_debt else np.nan
        cash, eq, ta, ca, cl_, pat, ocf, capex = (g(fy, v) for v in ("cash_and_cash_equivalents", "total_equity", "total_assets", "current_assets", "current_liabilities", "pat", "operating_cash_flow", "capex"))
        cogs = sum(0.0 if pd.isna(g(fy, v)) else g(fy, v) for v in ("cost_of_materials", "purchases_stock", "change_inventories"))
        cogs = cogs if pd.notna(g(fy, "cost_of_materials")) else np.nan
        ce = ta - cl_ if pd.notna(ta) and pd.notna(cl_) else np.nan
        M.loc[fy, ["ebit", "ebitda", "debt", "capital_employed", "cogs_proxy"]] = [ebit, ebitda, debt, ce, cogs]
        M.loc[fy, "net_debt"] = debt - cash if pd.notna(debt) and pd.notna(cash) else np.nan
        M.loc[fy, "opm_pct"], M.loc[fy, "ebitda_margin_pct"], M.loc[fy, "npm_pct"] = ratio(ebit, rev, 100), ratio(ebitda, rev, 100), ratio(pat, rev, 100)
        M.loc[fy, "gross_margin_pct"] = ratio(rev - cogs, rev, 100) if pd.notna(cogs) else np.nan
        ae, aa, ac = avg(fy, "total_equity"), avg(fy, "total_assets"), np.nan
        o_ce = (opening(fy, "total_assets") - opening(fy, "current_liabilities")) if pd.notna(opening(fy, "total_assets")) and pd.notna(opening(fy, "current_liabilities")) else np.nan
        ac = (o_ce + ce) / 2 if pd.notna(o_ce) and pd.notna(ce) else ce
        M.loc[fy, "roe_pct"], M.loc[fy, "roce_pct"] = ratio(pat, ae, 100), ratio(ebit, ac, 100)
        M.loc[fy, "asset_turnover"], M.loc[fy, "equity_multiplier"] = ratio(rev, aa), ratio(aa, ae)
        M.loc[fy, "dupont_roe_pct"] = (M.at[fy, "npm_pct"] / 100) * M.at[fy, "asset_turnover"] * M.at[fy, "equity_multiplier"] * 100
        fcf = ocf - capex if pd.notna(ocf) and pd.notna(capex) else np.nan
        M.loc[fy, "fcf"] = fcf
        M.loc[fy, "cash_conversion"], M.loc[fy, "ocf_to_ebitda"] = ratio(ocf, pat), ratio(ocf, ebitda)
        M.loc[fy, "ocf_to_revenue_pct"], M.loc[fy, "fcf_margin_pct"], M.loc[fy, "fcf_to_pat"] = ratio(ocf, rev, 100), ratio(fcf, rev, 100), ratio(fcf, pat)
        M.loc[fy, "capex_to_dep"], M.loc[fy, "capex_to_revenue_pct"] = ratio(capex, da), ratio(capex, rev, 100)
        M.loc[fy, "accruals_ratio"] = ratio(pat - ocf, aa) if pd.notna(pat) and pd.notna(ocf) else np.nan
        M.loc[fy, "debt_to_equity"], M.loc[fy, "net_debt_to_ebitda"], M.loc[fy, "debt_to_assets"] = ratio(debt, eq), ratio(M.at[fy, "net_debt"], ebitda), ratio(debt, ta)
        M.loc[fy, "interest_coverage"] = ratio(ebit, fin) if pd.notna(fin) and fin > 0 else np.nan          # NaN = no finance cost (debt-free), not zero
        M.loc[fy, "ebitda_interest"] = ratio(ebitda, fin) if pd.notna(fin) and fin > 0 else np.nan
        inv, rc, pay = g(fy, "inventories"), g(fy, "trade_receivables"), g(fy, "trade_payables")
        M.loc[fy, "current_ratio"], M.loc[fy, "quick_ratio"], M.loc[fy, "cash_ratio"] = ratio(ca, cl_), ratio(ca - inv, cl_) if pd.notna(inv) else np.nan, ratio(cash, cl_)
        M.loc[fy, "receivable_days"] = ratio(avg(fy, "trade_receivables"), rev, 365)            # credit sales = revenue from operations (proxy, labelled)
        M.loc[fy, "inventory_days"] = ratio(avg(fy, "inventories"), cogs, 365) if pd.notna(cogs) else np.nan
        M.loc[fy, "payable_days"] = ratio(avg(fy, "trade_payables"), cogs, 365) if pd.notna(cogs) else np.nan
        M.loc[fy, "cash_conversion_cycle"] = M.at[fy, "receivable_days"] + M.at[fy, "inventory_days"] - M.at[fy, "payable_days"]
        M.loc[fy, "nwc_to_revenue_pct"] = ratio(ca - cl_, rev, 100)
        M.loc[fy, "other_income_share_pct"] = ratio(g(fy, "other_income"), pbt, 100)
        M.loc[fy, "payout_pct"] = ratio(g(fy, "dividends_paid"), pat, 100)
    for v, src in (("revenue_growth_pct", "revenue_from_operations"), ("pat_growth_pct", "pat"), ("eps_growth_pct", "eps_basic"), ("ebit_growth_pct", None), ("ebitda_growth_pct", None)):
        for fy in fys:
            if src: cur, prev = g(fy, src), (g(fy - 1, src) if (fy - 1) in R.index else PRIORCOL.get((c, fy - 1, src), np.nan))
            else:
                col = "ebit" if v.startswith("ebit_") else "ebitda"
                cur = M.at[fy, col]; prev = M.at[fy - 1, col] if (fy - 1) in M.index else np.nan
                if pd.isna(prev) and col == "ebit" and (fy - 1) not in R.index:
                    pb, fn = PRIORCOL.get((c, fy - 1, "profit_before_tax"), np.nan), PRIORCOL.get((c, fy - 1, "finance_costs"), 0.0)
                    prev = pb + fn if pd.notna(pb) else np.nan
            M.loc[fy, v] = pct_change(cur, prev)
    M["dol"] = [ratio(M.at[fy, "ebit_growth_pct"], M.at[fy, "revenue_growth_pct"]) if pd.notna(M.at[fy, "revenue_growth_pct"]) and abs(M.at[fy, "revenue_growth_pct"]) > 0.5 else np.nan for fy in fys]
    mk = MARKET.get(c, {}); fv = mk.get("face_value")
    M["shares_cr"] = [ratio(g(fy, "equity_share_capital"), fv) if fv else np.nan for fy in fys]
    M["shares_change_pct"] = M["shares_cr"].pct_change() * 100
    return M, sorted(set(notes))

MET, MET_NOTES = {}, {}
for c in COMPANIES: MET[c], MET_NOTES[c] = build_metrics(c)

def cagr_of(c, series_name, source="raw"):
    _df = RAW[c] if source == "raw" else MET[c]
    if series_name not in _df.columns: return np.nan
    s = _df[series_name].dropna()
    return cagr(s.iloc[0], s.iloc[-1], s.index[-1] - s.index[0]) if len(s) >= 2 else np.nan

for c in COMPANIES:
    print(f"\n=== {c}: key metrics (INR crore / %) ===")
    show = MET[c][["ebit", "ebitda", "net_debt", "opm_pct", "npm_pct", "roe_pct", "roce_pct", "fcf", "cash_conversion", "current_ratio", "revenue_growth_pct", "pat_growth_pct"]].round(2)
    show.insert(0, "revenue", RAW[c].get("revenue_from_operations")); show.insert(1, "pat", RAW[c].get("pat")); display(show)
    print("Revenue CAGR %:", round(cagr_of(c, "revenue_from_operations"), 2), "| PAT CAGR %:", round(cagr_of(c, "pat"), 2), "| intervals:", len(RAW[c]) - 1)
    for n in MET_NOTES[c][:3]: print("  note:", n)
MET_ALL = pd.concat({c: MET[c] for c in COMPANIES}); MET_ALL.to_csv(DIRS["tables"] / "derived_metrics.csv")

In [ ]:
# ===== CELL 17A | other bank balances (current assets) -> VT, and net_debt_liquid -> MET =====
import re, pandas as pd
_DASH = {"-", "–", "—"}
vt_new, show_rows = [], []
for rid in sorted(REPORTS):
    if (rid, "bs") not in STMT: continue
    c, fy = REPORTS[rid]["company"], REPORTS[rid]["fy"]
    rows, K = _rows(rid, "bs"), _Kk(rid, "bs"); fac = unit_factor(rid, "bs")[0]
    if not fac: continue
    tot, used, page = 0.0, [], None
    for r in rows:
        lab = str(r.get("label") or ""); ll = lab.lower()
        if r.get("section") != "CA" or "bank balance" not in ll: continue
        if re.match(r"^\W*(\(?[a-z0-9]{1,3}\)?\s*)?cash (and|&) bank", ll): continue      # combined "cash and bank balances" line
        w = (r.get("nums") or [])[-K:]
        if len(w) < K: continue
        v = 0.0 if str(w[0]).strip() in _DASH else parse_num(w[0])
        if v is None: continue
        tot += v; used.append(lab[:48]); page = r.get("page")
    val = tot * fac
    vt_new.append({"company": c, "fy": fy, "variable": "other_bank_balances", "value": val, "unit": "INR crore", "method": "P_row",
                   "rule_branch": "sum of current-asset 'bank balances other than cash' rows" if used else "no bank-balance line printed -> 0",
                   "validation_status": "unverified", "value_status": "reported" if used else "nil", "actual_or_forecast": "actual",
                   "source_document": REPORTS[rid]["path"].name, "source_page": page, "source_label": " + ".join(used) if used else None, "review": False})
    show_rows.append((c, fy, round(val, 2), len(used), "; ".join(used)[:60]))
VT = pd.concat([VT[VT.variable != "other_bank_balances"], pd.DataFrame(vt_new)], ignore_index=True)
print("rows with no bank-balance line:", sum(1 for x in show_rows if x[3] == 0))
display(pd.DataFrame(show_rows, columns=["company", "fy", "other_bank_cr", "rows_used", "label"]))

for c in COMPANIES:
    df = MET[c]; s = VT[(VT.company == c) & (VT.variable == "other_bank_balances")].set_index("fy")["value"]
    df["other_bank_balances"] = s.reindex(df.index).values
    df["net_debt_liquid"] = df["net_debt"] - df["other_bank_balances"]
    print(f"\n{c}: net debt (cash only) vs liquid (cash + other bank balances), INR crore")
    display(df[["net_debt", "other_bank_balances", "net_debt_liquid"]].round(1))

In [ ]:
# ===== CELL 17B (v2) | per-share metrics: year-end shares on the CURRENT share basis (splits via face value, bonus via evidence) =====
import numpy as np, pandas as pd

def web_eps(c, fy):                                   # same source order as Cell 15 (o_value): yfinance first for EPS
    return o_value(c, fy, "eps_basic")[0]

PS, FACT_LOG = [], []
for c in COMPANIES:
    fv = (MARKET.get(c) or {}).get("face_value")
    esc_latest = vt_get(c, FYS[c][-1], "equity_share_capital")
    sh_latest = (esc_latest / fv) if (esc_latest and fv) else None
    for fy in FYS[c]:
        esc = vt_get(c, fy, "equity_share_capital")
        pat, basis = vt_get(c, fy, "pat_owners"), "pat_owners"
        if pat is None: pat, basis = vt_get(c, fy, "pat"), "pat"
        if not (esc and fv and pat is not None): continue
        sh0 = esc / fv; eps0 = pat / sh0; w = web_eps(c, fy); f, ev, status = 1.0, "no bonus evidence", "-"
        if w:
            r = eps0 / w
            if r >= 1.9 and abs(r - round(r)) <= 0.03 * round(r):
                f = float(round(r)); ev = f"derived EPS / web EPS = {r:.2f}"
                status = "confirmed by equity-capital step" if (sh_latest and abs(sh0 * f - sh_latest) / sh_latest < 0.005) else "CHECK in annual report"
                FACT_LOG.append((c, fy, f, ev, status))
        sh = sh0 * f
        eq, div, eps_rep = vt_get(c, fy, "total_equity"), vt_get(c, fy, "dividends_paid"), vt_get(c, fy, "eps_basic")
        eps_ss = pat / sh
        use_rep = (not w) and bool(eps_rep) and abs(eps_rep / eps_ss - 1) <= 0.15     # reported EPS is on the same share basis (weighted vs year-end only)
        eps_fin = w if w else (eps_rep if use_rep else eps_ss)
        gap = None if not w else 100 * (eps_ss / w - 1)
        PS.append({"company": c, "fy": fy, "shares_cr_adj": sh, "bonus_factor": f, "pat_basis": basis,
                   "eps_split_safe": eps_ss, "eps_reported": eps_rep, "eps_web": w, "eps_gap_pct": gap,
                   "eps_final": eps_fin,
                   "eps_method": ("web_weighted_avg_split_adjusted" if w else ("reported_weighted_avg" if use_rep else "pat_over_year_end_shares")) + (" | year-end shares differ >1.5%" if (gap is not None and abs(gap) > 1.5) else ""),
                   "bvps": None if eq is None else eq / sh, "dps_paid": None if div is None else abs(div) / sh})
PS = pd.DataFrame(PS, columns=["company", "fy", "shares_cr_adj", "bonus_factor", "pat_basis", "eps_split_safe", "eps_reported", "eps_web", "eps_gap_pct", "eps_final", "eps_method", "bvps", "dps_paid"])
PS["eps_gap_pct"] = pd.to_numeric(PS["eps_gap_pct"], errors="coerce")
for c in COMPANIES:
    s = PS[PS.company == c].set_index("fy")
    for col in ("shares_cr_adj", "bonus_factor", "eps_split_safe", "eps_final", "bvps", "dps_paid"):
        MET[c][col] = [s[col].get(int(i)) for i in MET[c].index]

print("Bonus/split factors applied (evidence-based, please confirm each in the annual report):")
display(pd.DataFrame(FACT_LOG, columns=["company", "fy", "factor", "evidence", "status"]))
print("Per-share table (INR per share on the current share basis):")
display(PS.drop(columns=["eps_method"]).round(2))
print("cells where eps_split_safe differs from web EPS by more than 1.5% (shares issued during the year -> weighted vs year-end):")
display(PS[PS.eps_gap_pct.abs() > 1.5][["company", "fy", "eps_split_safe", "eps_web", "eps_gap_pct", "eps_method"]].round(2))
for c in COMPANIES:
    v = (MARKET.get(c) or {}).get("book_value"); sub_ = PS[(PS.company == c) & (PS.fy == FYS[c][-1])]
    if v is None or sub_.empty or pd.isna(sub_.bvps.iloc[0]): continue
    print(f"  {c} FY{FYS[c][-1]} BVPS {sub_.bvps.iloc[0]:,.1f} vs market-data book value {v} (latest quarter, so close but not equal)")
PS.to_csv(DIRS["tables"] / "per_share_metrics.csv", index=False)

In [ ]:
# ===== CELL TOL | decision table: which differences are outside the project tolerance =====
# tolerance: prototype rule (Cell 15A): crore = 0.1% or rounding floor, near <= 1% | other variables = 0.1%. PDF (document) value stays on any mismatch.
unit_of = VT.drop_duplicates("variable").set_index("variable")["unit"].to_dict()
nxt = lambda r: f"{REPORTS[r]['company']}_{REPORTS[r]['fy'] + 1}"

# (1) net_borrowings vs next report prior column (NB from Cell 15C)
bad = [(REPORTS[r]["company"], REPORTS[r]["fy"], round(NB[r][0], 2), round(NB[nxt(r)][1], 2))
       for r in NB if nxt(r) in NB and not within_tol(NB[r][0], NB[nxt(r)][1])]
print("net_borrowings vs next report prior column, outside tolerance:", bad or "none")

# (2) restatement differences (from Cell 16)
rs = pd.DataFrame(flag_rows + info_rows)
if len(rs):
    rs["accepted"] = pd.Series([within_tol(a, b, unit_of.get(v, "INR crore")) for a, b, v in zip(rs.as_first_reported, rs.in_next_report, rs.variable)], index=rs.index, dtype=bool)
    print("\nrestatement differences: accepted", int(rs.accepted.sum()), "| outside tolerance (informational, cash-neutral or small restatements)", int((~rs.accepted).sum()))
    display(rs[~rs.accepted][["company", "fy", "variable", "as_first_reported", "in_next_report", "rel_diff_%"]])

# (3) document value vs web value
d = VT[VT.p_value.notna() & VT.o_value.notna()].copy()
d["class"] = [tol_class(a, b, u, "doc", "web") for a, b, u in zip(d.p_value, d.o_value, d.unit)]
d["ok"] = d["class"].isin(["exact", "within_tolerance"])
print("\ndocument value vs web value: accepted", int(d.ok.sum()), "| outside tolerance", int((~d.ok).sum()), "|", d["class"].value_counts().to_dict())
display(d[~d.ok][["company", "fy", "variable", "value", "p_value", "o_value", "class", "method"]].head(60))

In [ ]:
# ===== CELL 17C | FREEZE: final gate + final export =====
gates = {"REVIEW cells": int(VT.review.sum()),
         "missing cells": int((VT.validation_status == "missing").sum()),
         "restatement flags": int((VT.restatement_status == "restated_flag").sum()),
         "sanity findings": len(checks),
         "duplicate (company, fy, variable)": int(VT.duplicated(["company", "fy", "variable"]).sum()),
         "cells from web (excluding eps_basic, pat_owners)": int(((VT.method == "O") & ~VT.variable.isin(["eps_basic", "pat_owners"])).sum())}
display(pd.Series(gates, name="count").to_frame())
print("cash roll-forward gaps (informational, not blocking):", int((~cr.gap_ok).sum()))
FROZEN_CLEAN = not any(gates.values())
OPEN_GATES = {k: int(v) for k, v in gates.items() if v}
VT.to_csv(DIRS["tables"] / "variable_table_final.csv", index=False)
for c in COMPANIES: MET[c].to_csv(DIRS["tables"] / f"metrics_{c}.csv")
def _quality(c):
    s_ = VT[VT.company == c]
    return {"company": c, "cells": len(s_), "reported": int(s_.value_status.isin(["reported", "reported_restated", "nil"]).sum()), "derived": int((s_.value_status == "derived").sum()),
            "from_web": int(((s_.method == "O") & ~s_.variable.isin(["eps_basic", "pat_owners"])).sum()), "review": int(s_.review.sum()),
            "missing": int((s_.validation_status == "missing").sum()), "restated_flags": int((s_.restatement_status == "restated_flag").sum()),
            "sanity_findings": sum(1 for x in checks if x[0] == c)}
QUALITY = pd.DataFrame([_quality(c) for c in COMPANIES]); QUALITY.to_csv(DIRS["tables"] / "quality_report.csv", index=False)
display(QUALITY)
print("FROZEN: all gates are 0." if FROZEN_CLEAN else f"SAVED WITH OPEN ITEMS {OPEN_GATES}: the figures are auto-extracted and not hand-checked. Review the rows listed in Cell 16 / quality_report.csv before relying on them.")
print("files: variable_table_final.csv, metrics_<company>.csv, quality_report.csv | rows:", len(VT))

In [ ]:
# ===== CELL 18 | VALUATION: P/E, P/B, EV/EBITDA, dividend yield (paid), earnings yield, by company and FY, plus "current" =====
import numpy as np, pandas as pd, pathlib

assert "MARKET" in globals() and "PRICE_FY" in globals(), "MARKET / PRICE_FY missing: run Cells 13, 14 and 14B first"
TAB = DIRS["tables"]
COMP, TOL_RATIO = list(COMPANIES), 0.001            # 0.1% for ratio checks (locked)
nz = lambda x: np.nan if x is None or (isinstance(x, float) and np.isnan(x)) or pd.isna(x) else float(x)

# 1) save the dated market snapshot (market data is in no frozen file)
mk = pd.DataFrame(MARKET).T.rename_axis("company").reset_index()
pf = pd.DataFrame([{"company": c, "fy": fy, "price_fy_end": p} for c, d in PRICE_FY.items() for fy, p in d.items()])
mk.to_csv(TAB / "market_snapshot.csv", index=False); pf.to_csv(TAB / "price_fy_end.csv", index=False)

# 2) frozen inputs (read only)
MET_ = {c: pd.read_csv(TAB / f"metrics_{c}.csv", index_col=0) for c in COMP}
for c in COMP: MET_[c].index = MET_[c].index.astype(int)
PS_ = pd.read_csv(TAB / "per_share_metrics.csv")

def ratios(price, sh, eps, bv, dps, nd, ebitda, mcap):
    miss = [n for n, v in (("price", price), ("shares", sh), ("eps_final", eps), ("bvps", bv), ("dps_paid", dps), ("net_debt", nd), ("ebitda", ebitda), ("mcap", mcap)) if pd.isna(v)]
    ev = mcap + nd if not (pd.isna(mcap) or pd.isna(nd)) else np.nan
    pe  = price / eps if not (pd.isna(price) or pd.isna(eps)) and eps > 0 else np.nan          # loss-making: not meaningful, NaN (never 0)
    pb  = price / bv  if not (pd.isna(price) or pd.isna(bv)) and bv > 0 else np.nan
    eve = ev / ebitda if not (pd.isna(ev) or pd.isna(ebitda)) and ebitda > 0 else np.nan
    dy  = dps / price * 100 if not (pd.isna(dps) or pd.isna(price)) and price > 0 else np.nan
    ey  = eps / price * 100 if not (pd.isna(eps) or pd.isna(price)) and price > 0 else np.nan
    return dict(ev=ev, pe=pe, pb=pb, ev_ebitda=eve, div_yield_paid_pct=dy, earnings_yield_pct=ey, missing_inputs=", ".join(miss))

rows = []
for c in COMP:
    ps = PS_[PS_.company == c].sort_values("fy")
    if ps.empty: continue                       # no per-share inputs (face value / equity capital missing): no valuation for this company
    for _, r in ps.iterrows():
        fy = int(r.fy); m = MET_[c].loc[fy] if fy in MET_[c].index else None
        price = nz(PRICE_FY.get(c, {}).get(fy)); sh = nz(r.shares_cr_adj)
        mcap = price * sh if not (pd.isna(price) or pd.isna(sh)) else np.nan
        nd, eb = (nz(m["net_debt"]), nz(m["ebitda"])) if m is not None else (np.nan, np.nan)
        rows.append(dict(company=c, fy=fy, basis="fy_end_price", price=price, shares_cr=sh, mcap_cr=mcap, net_debt=nd, ebitda=eb,
                         eps_final=nz(r.eps_final), bvps=nz(r.bvps), dps_paid=nz(r.dps_paid), as_of=f"{fy}-03-31",
                         **ratios(price, sh, nz(r.eps_final), nz(r.bvps), nz(r.dps_paid), nd, eb, mcap)))
    # current: price today and market cap today on the latest FY's fundamentals
    last = ps.iloc[-1]; fyl = int(last.fy); m = MET_[c].loc[fyl]; mkc = MARKET.get(c, {})
    price = nz(mkc.get("price")); sh = nz(last.shares_cr_adj)
    mcap = nz(mkc.get("mcap_cr")); mcap = mcap if not pd.isna(mcap) else (price * sh if not (pd.isna(price) or pd.isna(sh)) else np.nan)
    rows.append(dict(company=c, fy=fyl, basis="current_price", price=price, shares_cr=sh, mcap_cr=mcap, net_debt=nz(m["net_debt"]), ebitda=nz(m["ebitda"]),
                     eps_final=nz(last.eps_final), bvps=nz(last.bvps), dps_paid=nz(last.dps_paid), as_of=mkc.get("as_of"),
                     **ratios(price, sh, nz(last.eps_final), nz(last.bvps), nz(last.dps_paid), nz(m["net_debt"]), nz(m["ebitda"]), mcap)))
VAL = pd.DataFrame(rows, columns=["company", "fy", "basis", "price", "shares_cr", "mcap_cr", "net_debt", "ebitda", "eps_final", "bvps", "dps_paid", "as_of", "ev", "pe", "pb", "ev_ebitda", "div_yield_paid_pct", "earnings_yield_pct", "missing_inputs"])
VAL.to_csv(TAB / "valuation_table.csv", index=False)

# 3) checks
ok = VAL.pe.notna() & VAL.eps_final.notna()
bad_pe = VAL[ok & (((VAL.pe * VAL.eps_final) - VAL.price).abs() / VAL.price > TOL_RATIO)]
miss = VAL[VAL.missing_inputs != ""]
cur = VAL[VAL.basis == "current_price"].copy()
cur["mcap_vs_price_x_shares_pct"] = (cur.mcap_cr / (cur.price * cur.shares_cr) - 1) * 100
print(f"valuation rows: {len(VAL)} | P/E x EPS != price (0.1%): {len(bad_pe)} | rows with a missing input (ratio left blank, never 0): {len(miss)}")
pd.set_option("display.width", 250, "display.max_columns", 30)
for c in COMP:
    print(f"\n=== {c} ===")
    display(VAL[VAL.company == c][["fy", "basis", "price", "mcap_cr", "net_debt", "ev", "pe", "pb", "ev_ebitda", "div_yield_paid_pct", "earnings_yield_pct", "missing_inputs"]].round(2))
print("\ncurrent mcap (Screener) vs price x year-end shares, informational:"); display(cur[["company", "mcap_cr", "price", "shares_cr", "mcap_vs_price_x_shares_pct"]].round(2))
if len(miss): print("\nrows with missing inputs:"); display(miss[["company", "fy", "basis", "missing_inputs"]])
print("\nNotes: EV = market cap + net debt (borrowings minus cash, leases excluded; minority interest not in VT). FY-end prices are split/bonus-adjusted yfinance closes on 31 Mar. "
      "'current' uses today's price on the latest FY fundamentals. Dividend yield uses dividends PAID per share, not declared DPS. Where net cash is large it may include customer advances or committed funds, which would understate EV/EBITDA.")

In [ ]:
# ===== CELL 18B | EV sensitivity: add net_debt_liquid (cash + other bank balances) to valuation_table.csv =====
import numpy as np, pandas as pd, pathlib
TAB = DIRS["tables"]
V = pd.read_csv(TAB / "valuation_table.csv")
V = V.drop(columns=[c for c in ("other_bank_balances", "net_debt_liquid", "ev_liquid", "ev_ebitda_liquid") if c in V.columns])
if V.empty:
    for col in ("other_bank_balances", "net_debt_liquid", "ev_liquid", "ev_ebitda_liquid"): V[col] = np.nan
    V.to_csv(TAB / "valuation_table.csv", index=False)
    print("no valuation rows (no company has per-share inputs): 18B has nothing to add")
else:
    M = pd.concat({c: pd.read_csv(TAB / f"metrics_{c}.csv", index_col=0).rename_axis("fy") for c in V.company.unique()}, names=["company"]).reset_index()
    assert {"other_bank_balances", "net_debt_liquid"} <= set(M.columns), "metrics CSVs lack the Cell 17A columns: tell me"
    M["fy"] = M.fy.astype(int)
    V = V.merge(M[["company", "fy", "other_bank_balances", "net_debt_liquid"]], on=["company", "fy"], how="left")
    V["ev_liquid"] = V.mcap_cr + V.net_debt_liquid                                   # NaN stays NaN, never 0
    V["ev_ebitda_liquid"] = np.where(V.ebitda > 0, V.ev_liquid / V.ebitda, np.nan)
    bad = V[V.net_debt_liquid > V.net_debt + 0.01]                                   # liquid net debt can never exceed cash-only net debt
    V.to_csv(TAB / "valuation_table.csv", index=False)
    print(f"rows: {len(V)} | rows without net_debt_liquid: {int(V.net_debt_liquid.isna().sum())} | liquid > cash-only (must be 0): {len(bad)}")
    pd.set_option("display.width", 250)
    display(V[V.basis == "current_price"][["company", "fy", "mcap_cr", "net_debt", "other_bank_balances", "net_debt_liquid", "ev", "ev_liquid", "ev_ebitda", "ev_ebitda_liquid"]].round(2))
    for c in V.company.unique():
        print(f"\n=== {c} ==="); display(V[(V.company == c) & (V.basis == "fy_end_price")][["fy", "net_debt", "net_debt_liquid", "ev_ebitda", "ev_ebitda_liquid"]].round(2))
    print("\nNotes: ev_liquid = market cap + (borrowings - cash - other bank balances). It is a sensitivity next to the cash-only EV; the PDF-based net_debt definition is unchanged.")

In [ ]:
# ===== CELL 19 | NARRATIVES: template text from computed numbers only; every sentence points to fact ids that point to VT rows =====
import json, numpy as np, pandas as pd, pathlib
TAB = DIRS["tables"]
COMP = list(COMPANIES)
nz = lambda x: np.nan if x is None or pd.isna(x) else float(x)
VT_ = pd.read_csv(TAB / "variable_table_final.csv")
MET_ = {c: pd.read_csv(TAB / f"metrics_{c}.csv", index_col=0) for c in COMP}
for c in COMP: MET_[c].index = MET_[c].index.astype(int)
PS_, VAL_ = pd.read_csv(TAB / "per_share_metrics.csv"), pd.read_csv(TAB / "valuation_table.csv")
MKS = pd.read_csv(TAB / "market_snapshot.csv").set_index("company")
assert "ev_ebitda_liquid" in VAL_.columns, "run Cell 18B first"

FACTS, SKIPPED = [], []
def fact(c, fy, metric, value, unit, deps=(), label="reported_historical", note=None):
    fid = f"F{len(FACTS) + 1:04d}"; inp = []
    for var, fyy in deps:
        r = VT_[(VT_.company == c) & (VT_.fy == fyy) & (VT_.variable == var)]
        if len(r):
            r = r.iloc[0]; inp.append({"variable": var, "fy": int(fyy), "value": float(r.value), "source_document": r.source_document, "source_page": None if pd.isna(r.source_page) else int(r.source_page)})
    FACTS.append({"fact_id": fid, "company": c, "fy": int(fy), "metric": metric, "value": float(value), "unit": unit, "source_label": label, "inputs": inp, "note": note}); return fid
def cagr(a, b, n): return np.nan if pd.isna(a) or pd.isna(b) or a <= 0 or b <= 0 or n <= 0 else ((b / a) ** (1 / n) - 1) * 100
def f1(x, d=1): return f"{x:,.{d}f}"

REV, PAT, EBITDA_D = "revenue_from_operations", "pat", ["profit_before_tax", "finance_costs", "depreciation_and_amortisation"]
def build(c):
    S = []; M = MET_[c]; fys = sorted(M.index); a, b = fys[0], fys[-1]; n = b - a
    R = VT_[VT_.company == c].pivot_table(index="fy", columns="variable", values="value", aggfunc="first")
    P = PS_[PS_.company == c].set_index("fy"); V = VAL_[VAL_.company == c]
    g = lambda col, fy=b: nz(M.at[fy, col]) if (col in M.columns and fy in M.index) else np.nan
    rvv = lambda var, fy: nz(R.at[fy, var]) if (var in R.columns and fy in R.index) else np.nan
    def add(sec, text, fids): S.append({"company": c, "section": sec, "text": text, "fact_ids": fids})
    def skip(sec, why): SKIPPED.append(f"{c} | {sec} | {why}")

    # --- growth
    r0, r1 = rvv(REV, a), rvv(REV, b); rc = cagr(r0, r1, n)
    if not pd.isna(rc):
        ids = [fact(c, a, "revenue", r0, "INR crore", [(REV, a)]), fact(c, b, "revenue", r1, "INR crore", [(REV, b)]), fact(c, b, f"revenue_cagr_FY{a}_FY{b}", rc, "%", [(REV, a), (REV, b)])]
        add("growth", f"{c} revenue moved from INR {f1(r0,0)} cr in FY{a} to INR {f1(r1,0)} cr in FY{b}, a {n}-year CAGR of {f1(rc)}%.", ids)
    else: skip("growth", "revenue CAGR not computable")
    yo = g("revenue_growth_pct")
    if not pd.isna(yo): add("growth", f"In FY{b}, revenue {'grew' if yo >= 0 else 'fell'} {f1(abs(yo))}% from FY{b-1}.", [fact(c, b, "revenue_growth_yoy", yo, "%", [(REV, b), (REV, b - 1)])])
    p0, p1 = rvv(PAT, a), rvv(PAT, b); pc = cagr(p0, p1, n)
    if not pd.isna(pc):
        ids = [fact(c, a, "pat", p0, "INR crore", [(PAT, a)]), fact(c, b, "pat", p1, "INR crore", [(PAT, b)]), fact(c, b, f"pat_cagr_FY{a}_FY{b}", pc, "%", [(PAT, a), (PAT, b)])]
        rel = "faster than" if (not pd.isna(rc) and pc > rc) else "slower than"
        add("growth", f"PAT moved from INR {f1(p0,0)} cr to INR {f1(p1,0)} cr (CAGR {f1(pc)}%), {rel} revenue.", ids)
    else: skip("growth", "PAT CAGR not computable (non-positive endpoint)")
    e0, e1 = nz(P.eps_final.get(a)), nz(P.eps_final.get(b)); ec = cagr(e0, e1, n)
    if not pd.isna(ec): add("growth", f"EPS (final, on the current share basis) moved from INR {f1(e0,2)} to INR {f1(e1,2)}, a CAGR of {f1(ec)}%.", [fact(c, a, "eps_final", e0, "INR", label="reported_historical", note="eps_final, per_share_metrics.csv"), fact(c, b, "eps_final", e1, "INR", note="eps_final, per_share_metrics.csv"), fact(c, b, f"eps_cagr_FY{a}_FY{b}", ec, "%", note="from eps_final")])
    else: skip("growth", "EPS CAGR not computable (non-positive endpoint or missing eps_final)")
    # --- margins and returns
    m0, m1 = g("ebitda_margin_pct", a), g("ebitda_margin_pct")
    if not (pd.isna(m0) or pd.isna(m1)):
        ids = [fact(c, a, "ebitda_margin", m0, "%", [(REV, a)] + [(v, a) for v in EBITDA_D]), fact(c, b, "ebitda_margin", m1, "%", [(REV, b)] + [(v, b) for v in EBITDA_D])]
        add("profitability", f"EBITDA margin moved from {f1(m0)}% in FY{a} to {f1(m1)}% in FY{b} ({f1(m1-m0)} pp). EBITDA here is PBT + finance costs + D&A, so it includes other income.", ids)
    nm = g("npm_pct")
    if not pd.isna(nm): add("profitability", f"Net profit margin in FY{b} was {f1(nm)}%.", [fact(c, b, "net_margin", nm, "%", [(PAT, b), (REV, b)])])
    ro, rc2 = g("roe_pct"), g("roce_pct")
    if not (pd.isna(ro) or pd.isna(rc2)): add("profitability", f"FY{b} return on equity was {f1(ro)}% and return on capital employed {f1(rc2)}%.", [fact(c, b, "roe", ro, "%", [(PAT, b), ("total_equity", b), ("total_equity", b - 1)]), fact(c, b, "roce", rc2, "%", [("profit_before_tax", b), ("finance_costs", b), ("total_assets", b), ("current_liabilities", b)])])
    # --- cash flow
    cc, fcf = g("cash_conversion"), g("fcf")
    if not pd.isna(cc): add("cash_flow", f"Operating cash flow was {f1(cc,2)}x PAT in FY{b}.", [fact(c, b, "ocf_to_pat", cc, "x", [("operating_cash_flow", b), (PAT, b)])])
    if not pd.isna(fcf): add("cash_flow", f"Free cash flow (operating cash flow minus capex) in FY{b} was INR {f1(fcf,0)} cr, {'positive' if fcf > 0 else 'negative'}.", [fact(c, b, "fcf", fcf, "INR crore", [("operating_cash_flow", b), ("capex", b)])])
    # --- leverage
    nd, nde, ic = g("net_debt"), g("net_debt_to_ebitda"), g("interest_coverage")
    if not pd.isna(nd):
        dd = [("borrowings_noncurrent", b), ("borrowings_current", b), ("borrowings_current_maturities", b), ("cash_and_cash_equivalents", b)]
        add("leverage", f"At FY{b} end, {c} had {'net cash' if nd < 0 else 'net debt'} of INR {f1(abs(nd),0)} cr (borrowings minus cash and cash equivalents, leases excluded).", [fact(c, b, "net_debt", nd, "INR crore", dd)])
    else: skip("leverage", "net debt missing")
    if not pd.isna(ic): add("leverage", f"EBIT covered finance costs {f1(ic,1)}x in FY{b}.", [fact(c, b, "interest_coverage", ic, "x", [("profit_before_tax", b), ("finance_costs", b)])])
    # --- working capital
    c0, c1 = g("cash_conversion_cycle", a), g("cash_conversion_cycle")
    if not (pd.isna(c0) or pd.isna(c1)): add("working_capital", f"The cash conversion cycle was {f1(c1,0)} days in FY{b} versus {f1(c0,0)} days in FY{a}.", [fact(c, a, "ccc_days", c0, "days", [("trade_receivables", a), ("inventories", a), ("trade_payables", a)]), fact(c, b, "ccc_days", c1, "days", [("trade_receivables", b), ("inventories", b), ("trade_payables", b)])])
    cr_ = g("current_ratio")
    if not pd.isna(cr_): add("working_capital", f"The current ratio was {f1(cr_,2)} in FY{b}.", [fact(c, b, "current_ratio", cr_, "x", [("current_assets", b), ("current_liabilities", b)])])
    # --- valuation (market data, dated)
    _cr = V[V.basis == "current_price"]
    cur = _cr.iloc[0] if len(_cr) else pd.Series({k: np.nan for k in ("pe", "pb", "ev_ebitda", "ev_ebitda_liquid", "div_yield_paid_pct", "earnings_yield_pct", "as_of")})
    hist = V[V.basis == "fy_end_price"]; asof = cur.as_of
    if not len(_cr): skip("valuation", "no per-share / market inputs (face value or price missing)")
    if not pd.isna(cur.pe):
        med = hist.pe.dropna().median(); d = (cur.pe / med - 1) * 100 if not pd.isna(med) else np.nan
        ids = [fact(c, b, "pe_current", cur.pe, "x", label="market_data", note=f"price as of {asof} / eps_final FY{b}")]
        txt = f"At the {asof} price, {c} trades at {f1(cur.pe)}x FY{b} earnings (P/E)"
        if not pd.isna(d): ids.append(fact(c, b, "pe_5y_median", med, "x", label="market_data", note="median of FY-end P/E, FY%d-FY%d" % (a, b))); txt += f", {f1(abs(d),0)}% {'above' if d > 0 else 'below'} its five-year median of {f1(med)}x"
        add("valuation", txt + ".", ids)
    if not pd.isna(cur.pb): add("valuation", f"Price to book is {f1(cur.pb,2)}x on FY{b} book value per share.", [fact(c, b, "pb_current", cur.pb, "x", label="market_data", note=f"price as of {asof}")])
    if not (pd.isna(cur.ev_ebitda) or pd.isna(cur.ev_ebitda_liquid)): add("valuation", f"EV/EBITDA is {f1(cur.ev_ebitda)}x using cash-only net debt and {f1(cur.ev_ebitda_liquid)}x if other bank balances are also netted off.", [fact(c, b, "ev_ebitda_cash_only", cur.ev_ebitda, "x", label="market_data"), fact(c, b, "ev_ebitda_liquid", cur.ev_ebitda_liquid, "x", label="market_data")])
    if not pd.isna(cur.div_yield_paid_pct): add("valuation", f"The dividend yield on dividends paid is {f1(cur.div_yield_paid_pct,2)}%.", [fact(c, b, "dividend_yield_paid", cur.div_yield_paid_pct, "%", label="market_data")])
    if len(_cr): add("limitation", f"Current multiples use FY{b} (year to March {b}) earnings against a {asof} price, so they are not trailing-twelve-month multiples. EV excludes minority interest.", [])
    # --- data-driven limits (counted from the variable table; no company-specific text)
    vc = VT_[VT_.company == c]
    n_est = int(((vc.method == "O") & ~vc.variable.isin(["eps_basic", "pat_owners"])).sum()); n_der = int((vc.value_status == "derived").sum())
    n_rst = int(vc.restatement_status.isin(["restated_info", "restated_flag"]).sum()) if "restatement_status" in vc.columns else 0
    n_rev = int(vc.review.sum()) if "review" in vc.columns else 0
    bits = []
    if n_est: bits.append(f"{n_est} figure(s) come from web sources because the report line could not be read")
    if n_der: bits.append(f"{n_der} figure(s) are derived (for example PBT less PAT for tax)")
    if n_rst: bits.append(f"{n_rst} prior-year figure(s) were restated in the next report")
    if n_rev: bits.append(f"{n_rev} figure(s) still need review")
    if bits: add("limitation", "Data notes for " + c + ": " + "; ".join(bits) + ".", [])
    nl = g("net_debt_liquid")
    if not pd.isna(nl) and nl < 0: add("limitation", f"Net cash of INR {f1(abs(nl),0)} cr in FY{b} (cash plus other bank balances) can include customer advances or other committed funds, so it is not necessarily free cash.", [fact(c, b, "net_debt_liquid", nl, "INR crore", [("cash_and_cash_equivalents", b)], note="net_debt minus other_bank_balances")])
    for note in COMPANY_NOTES.get(c, []): add("limitation", note, [])
    return S

NARR = {c: build(c) for c in COMP}
(TAB / "narratives.json").write_text(json.dumps(NARR, indent=1)); (TAB / "facts.json").write_text(json.dumps(FACTS, indent=1))

# checks
no_fact = [s for c in COMP for s in NARR[c] if s["section"] != "limitation" and not s["fact_ids"]]
no_src = [f for f in FACTS if f["source_label"] == "reported_historical" and not f["inputs"] and "per_share_metrics" not in (f["note"] or "") and "eps_cagr" not in f["metric"]]
for f in no_src[:5]: print("  no source row:", f["company"], f["fy"], f["metric"])
print(f"sentences: {sum(len(v) for v in NARR.values())} | facts: {len(FACTS)} | numeric sentences without a fact id (must be 0): {len(no_fact)} | reported facts with no VT source row (must be 0): {len(no_src)} | sections skipped: {len(SKIPPED)}")
for s in SKIPPED: print("  skipped:", s)
for c in COMP:
    print(f"\n### {c}")
    for s in NARR[c]: print(f"- [{s['section']}] {s['text']} {s['fact_ids']}")

In [ ]:
# =====================================================================================================
# CELL 19B | NARRATIVE GAPS: "latest FY vs previous FY" and "shareholder returns" + tidy the coverage sentence
# Same discipline as Cell 19: text from computed numbers only, every numeric sentence carries fact ids.
# =====================================================================================================
assert "FACTS" in globals() and "NARR" in globals(), "run Cell 19 first"
assert not any(f["metric"] == "pat_growth_yoy" for f in FACTS), "19B already ran on this FACTS list: re-run Cell 19 first"
FBY = {f["fact_id"]: f for f in FACTS}
ndtxt = lambda v: (f"net cash of INR {f1(abs(v), 0)} cr" if v < 0 else f"net debt of INR {f1(v, 0)} cr")

def build_gap(c):
    S = []; M = MET_[c]; fys = sorted(M.index); a, b = fys[0], fys[-1]; p = b - 1
    R = VT_[VT_.company == c].pivot_table(index="fy", columns="variable", values="value", aggfunc="first")
    P = PS_[PS_.company == c].set_index("fy"); V = VAL_[VAL_.company == c]
    g  = lambda col, fy=b: nz(M.at[fy, col]) if (col in M.columns and fy in M.index) else np.nan
    rv = lambda var, fy=b: nz(R.at[fy, var]) if (var in R.columns and fy in R.index) else np.nan
    def add(sec, text, fids): S.append({"company": c, "section": sec, "text": text, "fact_ids": fids})
    sec1, sec2 = "latest_vs_previous", "shareholder_returns"

    # ---- latest FY vs previous FY
    pg, p0, p1 = g("pat_growth_pct"), rv(PAT, p), rv(PAT, b)
    if not (pd.isna(pg) or pd.isna(p0) or pd.isna(p1)):
        add(sec1, f"PAT {'rose' if pg >= 0 else 'fell'} {f1(abs(pg))}% in FY{b} to INR {f1(p1, 0)} cr from INR {f1(p0, 0)} cr in FY{p}.",
            [fact(c, b, "pat_growth_yoy", pg, "%", [(PAT, b), (PAT, p)])])
    m0, m1 = g("ebitda_margin_pct", p), g("ebitda_margin_pct")
    if not (pd.isna(m0) or pd.isna(m1)):
        add(sec1, f"EBITDA margin was {f1(m1)}% in FY{b} versus {f1(m0)}% in FY{p} ({f1(m1 - m0)} pp).",
            [fact(c, p, "ebitda_margin", m0, "%", [(REV, p)] + [(v, p) for v in EBITDA_D]), fact(c, b, "ebitda_margin", m1, "%", [(REV, b)] + [(v, b) for v in EBITDA_D])])
    r0_, r1_ = g("roe_pct", p), g("roe_pct")
    if not (pd.isna(r0_) or pd.isna(r1_)):
        add(sec1, f"Return on equity was {f1(r1_)}% in FY{b} versus {f1(r0_)}% in FY{p}.",
            [fact(c, p, "roe", r0_, "%", [(PAT, p), ("total_equity", p), ("total_equity", p - 1)]), fact(c, b, "roe", r1_, "%", [(PAT, b), ("total_equity", b), ("total_equity", b - 1)])])
    o0, o1 = rv("operating_cash_flow", p), rv("operating_cash_flow")
    if not (pd.isna(o0) or pd.isna(o1)):
        add(sec1, f"Operating cash flow was INR {f1(o1, 0)} cr in FY{b} versus INR {f1(o0, 0)} cr in FY{p}.",
            [fact(c, p, "operating_cash_flow", o0, "INR crore", [("operating_cash_flow", p)]), fact(c, b, "operating_cash_flow", o1, "INR crore", [("operating_cash_flow", b)])])
    n0, n1 = g("net_debt", p), g("net_debt")
    if not (pd.isna(n0) or pd.isna(n1)):
        add(sec1, f"The balance sheet moved from {ndtxt(n0)} at FY{p} end to {ndtxt(n1)} at FY{b} end (borrowings minus cash and cash equivalents, leases excluded).",
            [fact(c, p, "net_debt", n0, "INR crore", [("borrowings_noncurrent", p), ("borrowings_current", p), ("borrowings_current_maturities", p), ("cash_and_cash_equivalents", p)]),
             fact(c, b, "net_debt", n1, "INR crore", [("borrowings_noncurrent", b), ("borrowings_current", b), ("borrowings_current_maturities", b), ("cash_and_cash_equivalents", b)])])
    e0, e1 = nz(P.eps_final.get(p)), nz(P.eps_final.get(b))
    if not (pd.isna(e0) or pd.isna(e1)) and e0 > 0:
        eg = (e1 / e0 - 1) * 100
        add(sec1, f"EPS (final, current share basis) {'rose' if eg >= 0 else 'fell'} {f1(abs(eg))}% to INR {f1(e1, 2)} in FY{b} from INR {f1(e0, 2)} in FY{p}.",
            [fact(c, p, "eps_final", e0, "INR", note="eps_final, per_share_metrics.csv"), fact(c, b, "eps_final", e1, "INR", note="eps_final, per_share_metrics.csv"),
             fact(c, b, "eps_growth_yoy", eg, "%", note="from eps_final, per_share_metrics.csv")])

    # ---- shareholder returns
    dv, po = rv("dividends_paid"), g("payout_pct")
    if not (pd.isna(dv) or pd.isna(po)) and abs(dv) > 0:
        add(sec2, f"Dividends paid in FY{b} were INR {f1(abs(dv), 0)} cr, a payout of {f1(abs(po))}% of PAT.",
            [fact(c, b, "dividends_paid", abs(dv), "INR crore", [("dividends_paid", b)]), fact(c, b, "payout_ratio", abs(po), "%", [("dividends_paid", b), (PAT, b)])])
    d0, d1 = nz(P.dps_paid.get(a)), nz(P.dps_paid.get(b))
    if not pd.isna(d1) and d1 > 0:
        dn = "per_share_metrics.csv dps_paid"
        if not pd.isna(d0) and d0 > 0:
            dc = cagr(d0, d1, b - a)
            add(sec2, f"Dividend paid per share moved from INR {f1(d0, 2)} in FY{a} to INR {f1(d1, 2)} in FY{b}, a CAGR of {f1(dc)}%.",
                [fact(c, a, "dps_paid", d0, "INR", note=dn), fact(c, b, "dps_paid", d1, "INR", note=dn), fact(c, b, f"dps_cagr_FY{a}_FY{b}", dc, "%", note="from " + dn)])
        else:
            add(sec2, f"Dividend paid per share was INR {f1(d1, 2)} in FY{b}.", [fact(c, b, "dps_paid", d1, "INR", note=dn)])
    _cr = V[V.basis == "current_price"]
    cur = _cr.iloc[0] if len(_cr) else pd.Series({k: np.nan for k in ("earnings_yield_pct", "div_yield_paid_pct", "as_of")})
    if not pd.isna(cur.earnings_yield_pct):
        add(sec2, f"The earnings yield at the {cur.as_of} price is {f1(cur.earnings_yield_pct, 2)}% (FY{b} EPS over price), against a dividend yield on dividends paid of {f1(cur.div_yield_paid_pct, 2)}%."
            if not pd.isna(cur.div_yield_paid_pct) else f"The earnings yield at the {cur.as_of} price is {f1(cur.earnings_yield_pct, 2)}% (FY{b} EPS over price).",
            [fact(c, b, "earnings_yield", cur.earnings_yield_pct, "%", label="market_data", note=f"price as of {cur.as_of}")] +
            ([fact(c, b, "dividend_yield_paid", cur.div_yield_paid_pct, "%", label="market_data")] if not pd.isna(cur.div_yield_paid_pct) else []))
    sc = g("shares_change_pct")
    if not pd.isna(sc):
        add(sec2, f"The year-end share count (equity share capital divided by face value) changed by {f1(sc, 1)}% in FY{b}.",
            [fact(c, b, "shares_change", sc, "%", [("equity_share_capital", b), ("equity_share_capital", p)], note="equity_share_capital / face_value")])
    return S

for c in COMP:
    NARR[c].extend(build_gap(c))
    # tidy: a coverage of 100x+ means finance costs are negligible, say that instead of a meaningless multiple
    for s in NARR[c]:
        if s["text"].startswith("EBIT covered finance costs") and s["fact_ids"]:
            v = FBY[s["fact_ids"][0]]["value"]; fy_ = FBY[s["fact_ids"][0]]["fy"]
            if v > 100: s["text"] = f"Finance costs are negligible: EBIT covered them {f1(v, 0)}x in FY{fy_}."

(TAB / "narratives.json").write_text(json.dumps(NARR, indent=1)); (TAB / "facts.json").write_text(json.dumps(FACTS, indent=1))
no_fact = [s for c in COMP for s in NARR[c] if s["section"] != "limitation" and not s["fact_ids"]]
no_src = [f for f in FACTS if f["source_label"] == "reported_historical" and not f["inputs"] and "per_share_metrics" not in (f["note"] or "") and "eps_cagr" not in f["metric"]]
for f in no_src[:8]: print("  no source row:", f["company"], f["fy"], f["metric"])
print(f"sentences: {sum(len(v) for v in NARR.values())} | facts: {len(FACTS)} | numeric sentences without a fact id (must be 0): {len(no_fact)} | reported facts with no VT source row (must be 0): {len(no_src)}")
for c in COMP:
    print(f"\n### {c} (new sections)")
    for s in NARR[c]:
        if s["section"] in ("latest_vs_previous", "shareholder_returns"): print(f"- [{s['section']}] {s['text']} {s['fact_ids']}")

In [ ]:
# =====================================================================================================
# CELL 20 | SCENARIOS: reverse DCF (Q17) and stress test (Q19). All arithmetic in Python; assumptions from Cell 1, printed.
# Reverse DCF: which constant growth rate over DCF_YEARS, then TERMINAL_G forever, makes the PV of the base cash flow equal today's market cap?
# Stress: FY base year, revenue -STRESS_REV_DROP and borrowing cost +STRESS_RATE_UP, two explicit cost-behaviour cases (bounds, not forecasts).
# =====================================================================================================
assert "FACTS" in globals() and "NARR" in globals(), "run Cell 19 / 19B first"
assert not any(f["metric"].startswith("rdcf_") for f in FACTS), "Cell 20 already ran on this FACTS list: re-run Cell 19 and 19B first"
r0, tg, N = COST_OF_EQUITY, TERMINAL_G, DCF_YEARS
assert r0 > tg, "cost of equity must exceed terminal growth"
print(f"ASSUMPTIONS | cost of equity {r0:.1%} | terminal growth {tg:.1%} | explicit years {N} | stress: revenue -{STRESS_REV_DROP:.0%}, borrowing cost +{STRESS_RATE_UP * 1e4:.0f} bps")

def pv_equity(base, g1, r):
    t = np.arange(1, N + 1); cf = base * (1 + g1) ** t
    return float((cf / (1 + r) ** t).sum() + cf[-1] * (1 + tg) / (r - tg) / (1 + r) ** N)

def implied_growth(base, target, r, lo=-0.30, hi=1.00):
    if pd.isna(base) or pd.isna(target) or base <= 0 or target <= 0: return np.nan, "base or market cap is not positive"
    if pv_equity(base, lo, r) > target: return np.nan, f"market cap is below the value even at {lo:.0%} growth"
    if pv_equity(base, hi, r) < target: return np.nan, f"needs growth above {hi:.0%} a year"
    x0, x1 = lo, hi
    for _ in range(80):
        mid = (x0 + x1) / 2
        if pv_equity(base, mid, r) < target: x0 = mid
        else: x1 = mid
    return (x0 + x1) / 2, "ok"

# self-test of the solver: a base grown at 12% for N years with r=12% and terminal g must reproduce itself
_g, _ = implied_growth(100.0, pv_equity(100.0, 0.12, r0), r0); assert abs(_g - 0.12) < 1e-6, _g

CUR = VAL_[VAL_.basis == "current_price"].set_index("company")
SC_ROWS, SK20 = [], []
def skip20(c, sec, why): SKIPPED.append(f"{c} | {sec} | {why}"); SK20.append(f"{c} | {sec} | {why}")

for c in COMP:
    M = MET_[c]; fys = sorted(M.index); a, b = fys[0], fys[-1]; n = b - a
    R = VT_[VT_.company == c].pivot_table(index="fy", columns="variable", values="value", aggfunc="first")
    P = PS_[PS_.company == c].set_index("fy")
    rv = lambda var, fy=b: nz(R.at[fy, var]) if (var in R.columns and fy in R.index) else np.nan
    cur = CUR.loc[c] if c in CUR.index else pd.Series({"mcap_cr": np.nan, "as_of": "n/a", "price": np.nan, "pe": np.nan})
    mcap, asof, price = nz(cur.mcap_cr), cur.as_of, nz(cur.price)
    def add(sec, text, fids): NARR[c].append({"company": c, "section": sec, "text": text, "fact_ids": fids})

    # ------------------------------ Q17 reverse DCF
    pat_b = rv(PAT); hist = cagr(rv(PAT, a), pat_b, n)
    gp, why_p = implied_growth(pat_b, mcap, r0)
    fcf_s = M["fcf"].dropna(); fcf_avg = nz(fcf_s.mean()) if len(fcf_s) >= 3 else np.nan
    gf, why_f = implied_growth(fcf_avg, mcap, r0)
    note = f"cost of equity {r0:.0%}, terminal growth {tg:.0%}, {N} explicit years, market cap as of {asof}"
    if not pd.isna(gp):
        ids = [fact(c, b, "rdcf_market_cap", mcap, "INR crore", label="market_data", note=f"price as of {asof}"),
               fact(c, b, "rdcf_base_pat", pat_b, "INR crore", [(PAT, b)]),
               fact(c, b, "rdcf_implied_growth_pat", gp * 100, "%", label="model_output", note=note)]
        txt = (f"A {N}-year reverse DCF on the {asof} market cap of INR {f1(mcap, 0)} cr (cost of equity {r0:.0%}, terminal growth {tg:.0%}, FY{b} PAT of INR {f1(pat_b, 0)} cr as the base) "
               f"implies PAT growth of {f1(gp * 100)}% a year for {N} years")
        if not pd.isna(hist): ids.append(fact(c, b, "rdcf_hist_pat_cagr", hist, "%", [(PAT, a), (PAT, b)])); txt += f", against an FY{a} to FY{b} PAT CAGR of {f1(hist)}%"
        add("scenario_reverse_dcf", txt + ".", ids)
        lo_, _ = implied_growth(pat_b, mcap, r0 - 0.01); hi_, _ = implied_growth(pat_b, mcap, r0 + 0.01)
        if not (pd.isna(lo_) or pd.isna(hi_)):
            add("scenario_reverse_dcf", f"With the cost of equity at {r0 - 0.01:.0%} the implied growth is {f1(lo_ * 100)}%, and at {r0 + 0.01:.0%} it is {f1(hi_ * 100)}%.",
                [fact(c, b, "rdcf_implied_growth_r_minus_1pp", lo_ * 100, "%", label="model_output", note=note.replace(f"{r0:.0%}", f"{r0 - 0.01:.0%}", 1)),
                 fact(c, b, "rdcf_implied_growth_r_plus_1pp", hi_ * 100, "%", label="model_output", note=note.replace(f"{r0:.0%}", f"{r0 + 0.01:.0%}", 1))])
            SC_ROWS += [dict(company=c, scenario="reverse_dcf_pat", case=f"r={r0 + d:.0%}", value=v * 100, unit="% growth") for d, v in ((-0.01, lo_), (0.0, gp), (0.01, hi_))]
    else: skip20(c, "scenario_reverse_dcf", f"PAT base: {why_p}")
    if not pd.isna(fcf_avg):
        if not pd.isna(gf):
            add("scenario_reverse_dcf", f"Using the {len(fcf_s)}-year average free cash flow of INR {f1(fcf_avg, 0)} cr as the base instead, the implied growth is {f1(gf * 100)}% a year.",
                [fact(c, b, "rdcf_base_fcf_avg", fcf_avg, "INR crore", [("operating_cash_flow", y) for y in fcf_s.index] + [("capex", y) for y in fcf_s.index]),
                 fact(c, b, "rdcf_implied_growth_fcf", gf * 100, "%", label="model_output", note=note)])
        else:
            add("scenario_reverse_dcf", f"The {len(fcf_s)}-year average free cash flow of INR {f1(fcf_avg, 0)} cr is not a usable base ({why_f}), so the reverse DCF rests on PAT and assumes earnings eventually convert to cash.",
                [fact(c, b, "rdcf_base_fcf_avg", fcf_avg, "INR crore", [("operating_cash_flow", y) for y in fcf_s.index] + [("capex", y) for y in fcf_s.index])])
    SC_ROWS.append(dict(company=c, scenario="reverse_dcf_pat", case=f"r={r0:.0%}", value=gp * 100 if not pd.isna(gp) else np.nan, unit="% growth", note=why_p))
    SC_ROWS.append(dict(company=c, scenario="reverse_dcf_fcf_avg", case=f"r={r0:.0%}", value=gf * 100 if not pd.isna(gf) else np.nan, unit="% growth", note=why_f))

    # ------------------------------ Q19 stress test (base year b, deltas on reported figures)
    rev, te, fin, pbt, tax, pat = rv(REV), rv("total_expenses"), rv("finance_costs"), rv("profit_before_tax"), rv("tax_expense"), rv(PAT)
    cogs, debt, ebit, eps = nz(M.at[b, "cogs_proxy"]), nz(M.at[b, "debt"]), nz(M.at[b, "ebit"]), nz(P.eps_final.get(b))
    miss = [k for k, v in dict(revenue=rev, total_expenses=te, finance_costs=fin, pbt=pbt, tax=tax, pat=pat, ebit=ebit).items() if pd.isna(v)]
    if miss or pbt <= 0 or pat <= 0:
        skip20(c, "scenario_stress", f"missing {miss} or non-positive PBT/PAT"); continue
    t_rate = tax / pbt
    d_rev = -STRESS_REV_DROP * rev
    save = {"proportional": STRESS_REV_DROP * (te - fin), "fixed_costs": (STRESS_REV_DROP * cogs) if (not pd.isna(cogs) and 0 <= cogs <= te - fin) else np.nan}
    if pd.isna(save["fixed_costs"]): skip20(c, "scenario_stress", f"fixed-cost case skipped: material-linked cost proxy {cogs} is missing or outside 0..operating costs")
    d_int = (STRESS_RATE_UP * debt) if not pd.isna(debt) else np.nan
    dep = [(REV, b), ("total_expenses", b), ("finance_costs", b), ("profit_before_tax", b), ("tax_expense", b), (PAT, b)]
    nt = f"revenue -{STRESS_REV_DROP:.0%}, borrowing cost +{STRESS_RATE_UP * 1e4:.0f} bps on FY{b} year-end borrowings, tax rate {t_rate:.1%}, other income flat"
    res = {}
    for case, sv in save.items():
        if pd.isna(sv): continue
        d_ebit = d_rev + sv; d_pbt = d_ebit - (0.0 if pd.isna(d_int) else d_int)
        pat_s = pat + d_pbt * (1 - t_rate); chg = (pat_s / pat - 1) * 100
        den = fin + (0.0 if pd.isna(d_int) else d_int); cov = (ebit + d_ebit) / den if den > 0 else np.nan
        eps_s = eps * pat_s / pat if not pd.isna(eps) else np.nan
        pe_s = price / eps_s if (not pd.isna(price) and not pd.isna(eps_s) and eps_s > 0) else np.nan
        res[case] = dict(pat_s=pat_s, chg=chg, cov=cov, pe=pe_s)
        for k, v, u in (("pat_stressed", pat_s, "INR crore"), ("pat_change_pct", chg, "%"), ("interest_cover", cov, "x"), ("pe_stressed", pe_s, "x")):
            SC_ROWS.append(dict(company=c, scenario="stress", case=case, value=v, unit=u, note=k))
    if len(res) == 2:
        pr, fx = res["proportional"], res["fixed_costs"]
        add("scenario_stress", f"If FY{b} revenue were {STRESS_REV_DROP:.0%} lower, PAT would fall about {f1(abs(pr['chg']))}% if every operating cost scaled with revenue, and about {f1(abs(fx['chg']))}% if only materials, purchases and inventory change fell and employee, depreciation and other costs stayed fixed (these two cases are bounds, not forecasts).",
            [fact(c, b, "stress_pat_change_proportional", pr["chg"], "%", dep, label="scenario_output", note=nt), fact(c, b, "stress_pat_change_fixed_costs", fx["chg"], "%", dep + [("cost_of_materials", b), ("purchases_stock", b), ("change_inventories", b)], label="scenario_output", note=nt)])
    elif len(res) == 1:
        k_, v_ = next(iter(res.items()))
        add("scenario_stress", f"If FY{b} revenue were {STRESS_REV_DROP:.0%} lower with every operating cost scaling with revenue, PAT would fall about {f1(abs(v_['chg']))}%.",
            [fact(c, b, "stress_pat_change_" + k_, v_["chg"], "%", dep, label="scenario_output", note=nt)])
    if not pd.isna(d_int):
        w = res.get("fixed_costs") or res.get("proportional")
        if debt > 0 and w:
            cv = w["cov"]; cvt = ">100x" if (not pd.isna(cv) and cv > 100) else (f"{f1(cv, 1)}x" if not pd.isna(cv) else "not meaningful")
            add("scenario_stress", f"A {STRESS_RATE_UP * 1e4:.0f} bps rise in borrowing cost on INR {f1(debt, 0)} cr of borrowings adds INR {f1(d_int, 0)} cr of annual interest; with the revenue shock, EBIT would cover finance costs {cvt}.",
                [fact(c, b, "stress_extra_interest", d_int, "INR crore", [("borrowings_noncurrent", b), ("borrowings_current", b), ("borrowings_current_maturities", b)], label="scenario_output", note=nt)] +
                ([fact(c, b, "stress_interest_cover", cv, "x", dep, label="scenario_output", note=nt)] if not pd.isna(cv) else []))
        elif debt == 0:
            add("scenario_stress", f"{c} had no borrowings at FY{b} end, so a {STRESS_RATE_UP * 1e4:.0f} bps rise in borrowing cost has no direct effect on its interest bill.",
                [fact(c, b, "stress_extra_interest", 0.0, "INR crore", [("borrowings_noncurrent", b), ("borrowings_current", b), ("borrowings_current_maturities", b)], label="scenario_output", note=nt)])
    else: skip20(c, "scenario_stress", "borrowings missing: rate shock skipped")
    w = res.get("fixed_costs") or res.get("proportional")
    if w and not pd.isna(w["pe"]) and not pd.isna(cur.pe):
        add("scenario_stress", f"On stressed earnings the {asof} price would be {f1(w['pe'])}x P/E instead of {f1(cur.pe)}x.",
            [fact(c, b, "stress_pe", w["pe"], "x", dep, label="scenario_output", note=nt), fact(c, b, "pe_current", cur.pe, "x", label="market_data", note=f"price as of {asof} / eps_final FY{b}")])
    add("limitation", f"Scenario outputs are mechanical: the reverse DCF holds one growth rate for {N} years and then {tg:.0%}, and the stress test applies one year of shocks to FY{b} figures with other income, tax rate and borrowings held flat. FY{b} earnings are set against a {asof} price.", [])

pd.DataFrame(SC_ROWS).to_csv(TAB / "scenario_table.csv", index=False)
(TAB / "narratives.json").write_text(json.dumps(NARR, indent=1)); (TAB / "facts.json").write_text(json.dumps(FACTS, indent=1))
no_fact = [s for c in COMP for s in NARR[c] if s["section"] != "limitation" and not s["fact_ids"]]
dup = pd.Series([(f["company"], f["fy"], f["metric"], f["source_label"]) for f in FACTS]).duplicated().sum()
print(f"sentences: {sum(len(v) for v in NARR.values())} | facts: {len(FACTS)} | numeric sentences without a fact id (must be 0): {len(no_fact)} | duplicate facts (info, same number re-used): {int(dup)} | scenario skips: {len(SK20)}")
for s in SK20: print("  skipped:", s)
for c in COMP:
    print(f"\n### {c} (scenarios)")
    for s in NARR[c]:
        if s["section"].startswith("scenario") or (s["section"] == "limitation" and s["text"].startswith("Scenario")): print(f"- [{s['section']}] {s['text']} {s['fact_ids']}")

In [ ]:
# ===== CELL 21 | LOAD THE QUESTION BANK AND ANSWER EVERY QUESTION FOR EVERY SELECTED COMPANY =====
import json
from pathlib import Path

# Required setup from earlier notebook cells
assert "DRIVE_ROOT" in globals(), "Run Cell 1 first."
assert "COMP" in globals() and len(COMP) >= 1, "Run the company setup/data cells first."
assert "NARR" in globals(), "Run the narrative cells first."
assert "llm" in globals() and llm is not None, "Run the Llama model loading cell first."
assert "llm_chat" in globals(), "Run the LLM helper setup cell first."

# ----- Load and flatten the question bank -----
BANK_PATH = Path(DRIVE_ROOT) / QUESTION_BANK

if not BANK_PATH.is_file():
    raise FileNotFoundError(
        f"Question bank not found: {BANK_PATH}\n"
        "Put 22_questions_LLM.json in the Model_1 folder in Google Drive."
    )

raw_bank = json.loads(BANK_PATH.read_text(encoding="utf-8"))
if not isinstance(raw_bank, dict) or not isinstance(raw_bank.get("categories"), list):
    raise ValueError("Expected the JSON bank to contain a 'categories' list.")

BANK = []
for category_block in raw_bank["categories"]:
    category_name = category_block.get("category", "")
    for item in category_block.get("questions", []):
        BANK.append({
            "qid": str(item["id"]),
            "question": str(item["question"]),
            "category": str(category_name),
            "raw": item,
        })

if len(BANK) != 22:
    raise ValueError(f"Expected 22 questions in the bank; found {len(BANK)}.")

print(f"Loaded {len(BANK)} questions from {BANK_PATH}")
print("IDs:", ", ".join(q["qid"] for q in BANK))

# ----- Prepare saved output and resume cache -----
TAB = Path(TAB)
TAB.mkdir(parents=True, exist_ok=True)

CACHE_PATH = TAB / "question_answers_llm.json"
if CACHE_PATH.exists():
    try:
        cached_rows = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    except Exception:
        cached_rows = []
else:
    cached_rows = []

answer_cache = {
    (str(row.get("qid")), str(row.get("company"))): row
    for row in cached_rows
    if isinstance(row, dict)
}

WEB_RESULTS = globals().get("WEB", {}) or {}

SYSTEM_PROMPT = (
    "You are a careful financial analyst. Answer the exact question asked "
    "for the named company using only the supplied evidence. Do not invent "
    "facts, figures, causes, forecasts, or external information. Preserve "
    "financial-year labels and units. Explain what the evidence supports and "
    "identify what is missing. Distinguish reported facts from assumptions. "
    "If the evidence is insufficient, say so explicitly. Write a direct, "
    "question-specific answer in 2–5 sentences."
)

def make_company_context(company, q):
    # Include the company's existing fact-linked narratives.
    evidence = []
    for item in NARR.get(company, []):
        if item.get("section") == "limitation":
            continue
        text = str(item.get("text", "")).strip()
        if text:
            evidence.append({
                "section": item.get("section", ""),
                "text": text,
                "fact_ids": item.get("fact_ids", []),
            })

    # Include web evidence if the web research cell has already run.
    web_item = WEB_RESULTS.get(company, {}).get(str(q["qid"]))
    web_evidence = []
    if isinstance(web_item, dict) and web_item.get("status") == "ok":
        web_evidence = web_item.get("sentences", [])

    return {
        "company": company,
        "question_id": q["qid"],
        "question": q["question"],
        "category": q["category"],
        "required_variables": q["raw"].get("required_variables", []),
        "evidence": evidence,
        "web_evidence": web_evidence,
    }

def generate_company_answer(company, q):
    context = make_company_context(company, q)

    prompt = (
        f"Answer this exact question for {company}.\n\n"
        f"Question ID: {q['qid']}\n"
        f"Category: {q['category']}\n"
        f"Question: {q['question']}\n"
        f"Variables the question asks about: "
        f"{json.dumps(q['raw'].get('required_variables', []), ensure_ascii=False)}\n\n"
        "Use only the evidence below. Keep the answer specific to this "
        "question and this company. Cite supplied fact IDs in square brackets "
        "after supported claims. Do not reuse unrelated narrative text. "
        "If a requested item is absent, say 'not available from the supplied "
        "evidence'. Do not make up a number or a conclusion.\n\n"
        "Evidence:\n"
        f"{json.dumps(context, ensure_ascii=False, indent=2)}"
    )

    answer = llm_chat(
        SYSTEM_PROMPT,
        prompt,
        max_new_tokens=350,
    )
    answer = str(answer).strip()

    if not answer:
        answer = "Not available from the supplied evidence."

    return answer, context

# ----- Ask all 22 questions for all four companies -----
total = len(BANK) * len(COMP)

for q in BANK:
    for company in COMP:
        key = (str(q["qid"]), str(company))

        # Reuse completed answers if rerunning after an interruption.
        ctx_hash = hashlib.sha256(json.dumps(make_company_context(company, q), sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
        existing = answer_cache.get(key)
        if existing and existing.get("status") == "generated" and existing.get("answer") and existing.get("ctx_hash") == ctx_hash:
            continue            # same question, same evidence as before: reuse

        try:
            answer, context = generate_company_answer(company, q)
            row = {
                "ctx_hash": ctx_hash,
                "qid": str(q["qid"]),
                "category": q["category"],
                "question": q["question"],
                "company": company,
                "answer": answer,
                "status": "generated",
                "fact_ids_available": sorted({
                    str(fid)
                    for item in context["evidence"]
                    for fid in (item.get("fact_ids") or [])
                }),
                "web_evidence_available": bool(context["web_evidence"]),
            }
        except Exception as exc:
            row = {
                "ctx_hash": ctx_hash,
                "qid": str(q["qid"]),
                "category": q["category"],
                "question": q["question"],
                "company": company,
                "answer": (
                    "Generation failed; rerun Cell 21 to retry. "
                    f"{type(exc).__name__}: {str(exc)[:250]}"
                ),
                "status": "failed",
                "fact_ids_available": [],
                "web_evidence_available": False,
            }

        answer_cache[key] = row

        # Save after each answer so an interrupted run can resume.
        ordered_so_far = [
            answer_cache[(str(question["qid"]), str(c))]
            for question in BANK
            for c in COMP
            if (str(question["qid"]), str(c)) in answer_cache
        ]
        CACHE_PATH.write_text(
            json.dumps(ordered_so_far, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        completed = len(answer_cache)
        print(f"[{completed}/{total}] {q['qid']} — {company}: {row['status']}")

# ----- Verify every question/company pair -----
expected_pairs = {
    (str(q["qid"]), str(company))
    for q in BANK
    for company in COMP
}
missing_pairs = expected_pairs - set(answer_cache)

if missing_pairs:
    raise RuntimeError(f"Missing answer pairs: {sorted(missing_pairs)}")

ANSWERS = [
    answer_cache[(str(q["qid"]), str(company))]
    for q in BANK
    for company in COMP
]

if len(ANSWERS) != len(BANK) * len(COMP):
    raise RuntimeError(f"Expected {len(BANK) * len(COMP)} answers, found {len(ANSWERS)}.")

# ----- Save final JSON and Markdown -----
JSON_OUT = TAB / "question_answers.json"
JSON_OUT.write_text(
    json.dumps(ANSWERS, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

markdown = [f"# Answers to the {len(BANK)} Questions — {RUN_NAME}"]
for q in BANK:
    markdown.append(f"\n## {q['qid']}: {q['question']}")
    markdown.append(f"\n**Category:** {q['category']}")
    for row in (a for a in ANSWERS if a["qid"] == str(q["qid"])):
        markdown.append(f"\n### {row['company']}\n")
        markdown.append(row["answer"])
        markdown.append(
            f"\n\n_Status: {row['status']} | "
            f"Fact IDs available: "
            f"{', '.join(row['fact_ids_available']) or 'none'}_"
        )

MD_OUT = TAB / "question_answers.md"
MD_OUT.write_text("\n".join(markdown), encoding="utf-8")

print(f"\nComplete: {len(ANSWERS)} answers ({len(BANK)} questions × {len(COMP)} companies).")
print("Generated:", sum(a["status"] == "generated" for a in ANSWERS))
print("Failed:", sum(a["status"] == "failed" for a in ANSWERS))
print("Saved:", JSON_OUT)
print("Saved:", MD_OUT)

In [ ]:
# ===== CELL 22 | SEVEN COMPARABLE CHARTS WITH SOURCE VALUES IN HOVER =====
COMPANIES = [c for c in META if c in FYS]      # the short keys every other variable uses
print("COMPANIES reset to:", COMPANIES)
import pandas as pd
import plotly.graph_objects as go

assert "COMPANIES" in globals() and COMPANIES, "Run the company setup cells first."
assert all(name in globals() for name in ("VT_", "MET_", "PS_")), (
    "Run through Cell 19 first so VT_, MET_, and PS_ are available."
)

YEARS = list(range(2022, 2027))
YEAR_LABELS = [f"FY{year}" for year in YEARS]
PALETTE = ["#4f81bd", "#e8703a", "#5aa469", "#8e6bbf", "#d4a72c"]

# Chart name: (title, y-axis unit, transformation)
CHARTS = {
    "revenue_index": (
        "Revenue growth — index, first available FY = 100",
        "Index (first available FY = 100)",
        "index_revenue",
    ),
    "pat_index": (
        "PAT growth — index, first available FY = 100",
        "Index (first available FY = 100)",
        "index_pat",
    ),
    "eps_index": (
        "EPS growth — index, first available FY = 100",
        "Index (first available FY = 100)",
        "index_eps",
    ),
    "ocf_margin_pct": (
        "Operating cash flow / revenue",
        "% of revenue",
        "ocf_margin",
    ),
    "fcf_margin_pct": (
        "Free cash flow / revenue",
        "% of revenue",
        "fcf_margin",
    ),
    "net_debt_to_revenue_pct": (
        "Net debt / revenue",
        "% of revenue",
        "debt_to_revenue",
    ),
    "ebitda_margin_pct": (
        "EBITDA margin",
        "%",
        "ebitda_margin",
    ),
}


def _source_series(owner, company, column):
    """Return {FY: numeric value} from the notebook's annual data tables."""
    if owner == "VT":
        rows = VT_.loc[
            (VT_["company"] == company) & (VT_["variable"] == column),
            ["fy", "value"],
        ].copy()
        rows["fy"] = pd.to_numeric(rows["fy"], errors="coerce")
        rows["value"] = pd.to_numeric(rows["value"], errors="coerce")
        rows = rows.dropna(subset=["fy", "value"]).drop_duplicates(
            "fy", keep="last"
        )
        return {int(row.fy): float(row.value) for row in rows.itertuples()}

    if owner == "PS":
        rows = PS_.loc[PS_["company"] == company, ["fy", column]].copy()
        rows["fy"] = pd.to_numeric(rows["fy"], errors="coerce")
        rows[column] = pd.to_numeric(rows[column], errors="coerce")
        rows = rows.dropna(subset=["fy", column]).drop_duplicates(
            "fy", keep="last"
        )
        return {
            int(row.fy): float(getattr(row, column))
            for row in rows.itertuples()
        }

    frame = MET_[company]
    if column not in frame.columns:
        return {}

    values = {}
    for fy, value in frame[column].items():
        try:
            if pd.notna(value):
                values[int(fy)] = float(value)
        except (TypeError, ValueError):
            continue
    return values


def _first_available_positive(values):
    available = [(fy, values[fy]) for fy in YEARS if fy in values]
    if not available:
        return None
    first_fy, first_value = available[0]
    return (first_fy, first_value) if first_value > 0 else None


def _money(value, unit):
    if value is None or pd.isna(value):
        return "not available"
    return f"{value:,.2f} {unit}"


FIGS = []
INDEX_NOTES = []

for chart_name, (title, y_unit, transform) in CHARTS.items():
    fig = go.Figure()

    for color_index, company in enumerate(COMPANIES):
        revenue = _source_series("VT", company, "revenue_from_operations")
        pat = _source_series("VT", company, "pat")
        eps = _source_series("PS", company, "eps_final")
        ocf = _source_series("VT", company, "operating_cash_flow")
        fcf = _source_series("MET", company, "fcf")
        net_debt = _source_series("MET", company, "net_debt")
        ebitda_margin = _source_series("MET", company, "ebitda_margin_pct")
        ebitda = _source_series("MET", company, "ebitda")

        plotted = {}
        hover_details = {}

        if transform in ("index_revenue", "index_pat", "index_eps"):
            if transform == "index_revenue":
                source, raw_name, raw_unit = revenue, "Revenue", "INR crore"
            elif transform == "index_pat":
                source, raw_name, raw_unit = pat, "PAT", "INR crore"
            else:
                source, raw_name, raw_unit = eps, "EPS", "INR per share"

            base = _first_available_positive(source)
            if base is None:
                INDEX_NOTES.append(
                    f"{company} — {chart_name}: no index shown because its "
                    "first available value is not positive or is absent."
                )
            else:
                base_fy, base_value = base
                plotted = {
                    fy: value / base_value * 100
                    for fy, value in source.items()
                }
                for fy, value in source.items():
                    hover_details[fy] = (
                        f"Reported {raw_name}: {_money(value, raw_unit)}"
                        f" | Index base: FY{base_fy} = "
                        f"{_money(base_value, raw_unit)}"
                    )

        elif transform == "ocf_margin":
            for fy in YEARS:
                if fy in ocf and fy in revenue and revenue[fy] > 0:
                    plotted[fy] = ocf[fy] / revenue[fy] * 100
                    hover_details[fy] = (
                        f"Operating cash flow: {_money(ocf[fy], 'INR crore')}"
                        f" | Revenue: {_money(revenue[fy], 'INR crore')}"
                    )

        elif transform == "fcf_margin":
            for fy in YEARS:
                if fy in fcf and fy in revenue and revenue[fy] > 0:
                    plotted[fy] = fcf[fy] / revenue[fy] * 100
                    hover_details[fy] = (
                        f"Free cash flow: {_money(fcf[fy], 'INR crore')}"
                        f" | Revenue: {_money(revenue[fy], 'INR crore')}"
                    )

        elif transform == "debt_to_revenue":
            for fy in YEARS:
                if fy in net_debt and fy in revenue and revenue[fy] > 0:
                    plotted[fy] = net_debt[fy] / revenue[fy] * 100
                    hover_details[fy] = (
                        f"Net debt: {_money(net_debt[fy], 'INR crore')}"
                        f" | Revenue: {_money(revenue[fy], 'INR crore')}"
                    )

        else:  # EBITDA margin
            plotted = ebitda_margin
            for fy, margin in ebitda_margin.items():
                details = [f"EBITDA margin: {margin:,.2f}%"]
                if fy in ebitda:
                    details.append(
                        f"EBITDA: {_money(ebitda[fy], 'INR crore')}"
                    )
                if fy in revenue:
                    details.append(
                        f"Revenue: {_money(revenue[fy], 'INR crore')}"
                    )
                hover_details[fy] = " | ".join(details)

        fig.add_trace(go.Bar(
            x=YEAR_LABELS,
            y=[plotted.get(fy) for fy in YEARS],
            customdata=[
                hover_details.get(fy, "Source value not available")
                for fy in YEARS
            ],
            name=company.replace("_", " "),
            marker_color=PALETTE[color_index % len(PALETTE)],
            hovertemplate=(
                "%{fullData.name}<br>"
                "%{x}: %{y:,.2f}<br>"
                "%{customdata}"
                "<extra></extra>"
            ),
        ))

    fig.update_layout(
        title={"text": title, "x": 0.01, "xanchor": "left"},
        template="plotly_white",
        barmode="group",
        bargap=0.22,
        autosize=True,
        width=None,
        height=300,
        margin={"t": 52, "b": 35, "l": 62, "r": 12},
        legend={
            "orientation": "h",
            "x": 0,
            "y": 1.02,
            "yanchor": "bottom",
            "font": {"size": 10},
        },
        font={"size": 10},
        meta="comparable_with_source_hover_v1",
    )
    fig.update_xaxes(
        type="category",
        categoryorder="array",
        categoryarray=YEAR_LABELS,
        tickangle=0,
        title_text=None,
    )
    fig.update_yaxes(title_text=y_unit, gridcolor="#eef1f6")

    FIGS.append((chart_name, title, fig))

# Keep the original chart figures for Cell 22B.
FIGS_RAW = list(FIGS)

print(f"Created {len(FIGS)} comparable charts.")
print("Hover over a bar to see its reported source value(s).")
print("Missing inputs remain blank; no values are filled or estimated.")
if INDEX_NOTES:
    print("\nIndex notes:")
    for note in INDEX_NOTES:
        print(" -", note)

In [ ]:
# ===== CELL 22B | REGROUP CHARTS, KEEP SOURCE HOVER DETAILS, MATCH COMPANY NAMES SAFELY =====
import re
import pandas as pd
import plotly.graph_objects as go

GROUP_BY = "company"      # "company": x = company names, one bar per FY | "year": x = FY, one bar per company
CHART_HEIGHT = 320
PALETTE = ["#4f81bd", "#e8703a", "#9a9a9a", "#f2b632", "#5aa469"]
REGROUP_TAG = "grouped_with_source_hover_v1"

_is_regrouped = lambda fig: getattr(fig.layout, "meta", None) == REGROUP_TAG

# FIGS_RAW must hold the charts exactly as Cell 22 built them.
# - Fresh charts from Cell 22 (untagged)       -> refresh FIGS_RAW.
# - Charts already regrouped by this cell      -> keep the existing FIGS_RAW (never overwrite it with them).
_fresh = bool(FIGS) and not any(_is_regrouped(f) for _, _, f in FIGS)
if _fresh or "FIGS_RAW" not in globals():
    FIGS_RAW = list(FIGS)
if any(_is_regrouped(f) for _, _, f in FIGS_RAW):
    print("WARNING: FIGS_RAW already holds regrouped charts. Re-run Cell 22 first if a company is missing below.")

PCT_TOKENS = {"ratio", "margin", "growth", "yield", "roe", "roce", "roa", "payout", "pct", "percent", "return", "returns"}
MULT_TOKENS = {"turnover", "coverage", "multiple", "pe", "pb", "ev", "leverage"}
DAY_TOKENS = {"days", "dso", "dpo", "dio", "ccc"}

# 'APOLLO MICROSYSTEMS' / 'apollo-microsystems' -> the exact pipeline key 'APOLLO_MICROSYSTEMS'
_CANON = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in COMPANIES}


def canon(name):
    return _CANON.get(re.sub(r"[^a-z0-9]", "", str(name).lower()), name)


def unit_label(name, title, original):
    tokens = set(re.split(r"[^a-z0-9]+", f"{name} {title}".lower()))
    if tokens & PCT_TOKENS:
        return "%"
    if "index" in tokens:
        return "Index (first FY = 100)"
    if "eps" in tokens:
        return "INR per share"
    if tokens & DAY_TOKENS:
        return "days"
    if tokens & MULT_TOKENS:
        return "x (times)"
    return original or "INR crore"


def period_label(value):
    text = str(value).strip()
    match = re.fullmatch(r"(?:FY)?\s*(\d{4})(?:\.0+)?", text, flags=re.I)
    return (f"FY{match.group(1)}", int(match.group(1))) if match else (text, 10 ** 9)


def facet_names(fig):
    """Map each trace x-axis id ('x', 'x2', ...) to the company named in that panel's title."""
    mapping = {}
    anns = [a for a in (fig.layout.annotations or []) if a.text and a.x is not None]
    for ax in fig.select_xaxes():
        ax_id = "x" + re.sub(r"\D", "", ax.plotly_name)
        dom = tuple(ax.domain) if ax.domain is not None else (0.0, 1.0)
        for ann in anns:
            try:
                inside = dom[0] - 1e-6 <= float(ann.x) <= dom[1] + 1e-6
            except (TypeError, ValueError):
                continue
            if inside:
                text = re.sub(r"<[^>]+>", "", str(ann.text))
                mapping[ax_id] = canon(re.sub(r"^[^=]*=", "", text).strip())
                break
    return mapping


def extract_records(fig, unmapped=None):
    """Return (company, period, value, hover detail) for each bar; names that match no company go to `unmapped`."""
    if unmapped is None:
        unmapped = set()
    records = []
    companies = set(COMPANIES)
    facets = facet_names(fig)

    for trace in fig.data:
        xs = list(trace.x) if trace.x is not None else []
        ys = list(trace.y) if trace.y is not None else []
        custom = list(trace.customdata) if trace.customdata is not None else []
        if len(custom) < len(xs):
            custom.extend([None] * (len(xs) - len(custom)))
        axis = trace.xaxis or "x"

        for x_value, y_value, detail in zip(xs, ys, custom):
            try:
                value = float(y_value)
            except (TypeError, ValueError):
                continue
            if pd.isna(value):
                continue

            if canon(trace.name) in companies:
                company, period = canon(trace.name), x_value
            elif facets.get(axis) in companies:
                company, period = facets[axis], x_value
            elif canon(x_value) in companies:
                company, period = canon(x_value), trace.name
            else:
                unmapped.add(f"{trace.name!r}/{x_value!r}")
                continue
            records.append((company, period, value, detail))
    return records


UNMAPPED = {}


def regroup(name, title, fig):
    if not fig.data or any(trace.type != "bar" for trace in fig.data):
        return fig

    unmapped = set()
    records = extract_records(fig, unmapped)
    if unmapped:
        UNMAPPED[name] = sorted(unmapped)[:6]
    if not records:
        return fig

    values, details, periods, companies_seen = {}, {}, {}, []
    for company, raw_period, value, detail in records:
        label, sort_key = period_label(raw_period)
        values[(company, label)] = value
        details[(company, label)] = detail
        periods[label] = sort_key
        if company not in companies_seen:
            companies_seen.append(company)

    ordered_periods = sorted(periods, key=lambda p: (periods[p], p))
    ordered_companies = [c for c in COMPANIES if c in companies_seen]

    if GROUP_BY == "year":
        categories, series = ordered_periods, ordered_companies
        lookup = lambda category, company: values.get((company, category))
        detail_lookup = lambda category, company: details.get((company, category))
        category_labels = categories
        series_label = lambda company: str(company).replace("_", " ")
    else:
        categories, series = ordered_companies, ordered_periods
        lookup = lambda company, period: values.get((company, period))
        detail_lookup = lambda company, period: details.get((company, period))
        category_labels = [str(company).replace("_", " ") for company in categories]
        series_label = str

    new = go.Figure()
    for index, item in enumerate(series):
        new.add_trace(go.Bar(
            x=category_labels,
            y=[lookup(category, item) for category in categories],
            customdata=[detail_lookup(category, item) or "" for category in categories],
            name=series_label(item),
            marker_color=PALETTE[index % len(PALETTE)],
            hovertemplate="%{x}<br>%{fullData.name}: %{y:,.2f}<br>%{customdata}<extra></extra>",
        ))

    original_unit = None
    try:
        original_unit = fig.layout.yaxis.title.text
    except Exception:
        pass

    new.update_layout(
        title={"text": title, "x": 0.01, "xanchor": "left", "y": 0.97, "yanchor": "top"},
        template="plotly_white",
        barmode="group", bargap=0.25, bargroupgap=0.06,
        autosize=True, width=None, height=CHART_HEIGHT,
        margin={"t": 78, "b": 42, "l": 60, "r": 12},
        legend={"orientation": "h", "x": 0, "y": 1.02, "yanchor": "bottom", "font": {"size": 10}},
        font={"size": 10},
        meta=REGROUP_TAG,
    )
    new.update_xaxes(title_text=None)
    new.update_yaxes(title_text=unit_label(name, title, original_unit), gridcolor="#eef1f6")
    return new


FIGS = [(name, title, regroup(name, title, figure)) for name, title, figure in FIGS_RAW]
print(f"Regrouped {len(FIGS)} charts with source hover details preserved.")

# ---- coverage check: every selected company should appear in every grouped bar chart ----
problems = []
for name, title, fig in FIGS:
    if not _is_regrouped(fig):
        continue
    present = set()
    for trace in fig.data:
        for x in (list(trace.x) if trace.x is not None else []):
            present.add(canon(x))
        if canon(trace.name) in COMPANIES:
            present.add(canon(trace.name))
    absent = [c for c in COMPANIES if c not in present]
    if absent:
        problems.append((name, absent))

if problems:
    print("\nCOMPANIES MISSING FROM CHARTS:")
    for name, absent in problems:
        print(f"  {name}: {absent}")
    print("If FIGS_RAW was already regrouped, re-run Cell 22 and then this cell. "
          "If it is still missing, the company has no value in Cell 22's data for that metric.")
else:
    print(f"Coverage check passed: all {len(COMPANIES)} companies appear in every grouped chart.")
if UNMAPPED:
    print("\nBars skipped because their names match no company:", UNMAPPED)

In [ ]:
rows = []
for name, title, fig in FIGS:
    counts = {c: 0 for c in COMPANIES}
    for t in fig.data:
        xs = list(t.x) if t.x is not None else []
        ys = list(t.y) if t.y is not None else []
        for x, y in zip(xs, ys):
            c = canon(x)
            if c in counts and y is not None and y == y:
                counts[c] += 1
    rows.append({"metric": name, **counts})
display(pd.DataFrame(rows))

In [ ]:
print("FIGS_RAW stale-regrouped check:", [getattr(f.layout, "meta", None) for _, _, f in FIGS_RAW][:7])
for name, title, fig in FIGS_RAW:
    if name != "revenue_index":
        continue
    print("annotations:", [a.text for a in (fig.layout.annotations or [])])
    for i, t in enumerate(fig.data):
        print(i, "trace name:", repr(t.name), "| axis:", t.xaxis,
              "| x:", list(t.x)[:6] if t.x is not None else None,
              "| y:", list(t.y)[:6] if t.y is not None else None)

In [ ]:
import re
keys = {re.sub(r"[^a-z]", "", c.lower()): c for c in COMPANIES}
for name, title, fig in FIGS_RAW:
    found = set()
    for t in fig.data:
        for x in (list(t.x) if t.x is not None else []):
            k = re.sub(r"[^a-z]", "", str(x).lower())
            if k in keys:
                found.add(keys[k])
    print(f"{name:28s} companies: {sorted(found)}")

needle = "grouped_with_source_hover_v1"
hits = [i for i, src in enumerate(In) if needle in src]
print("\ncells containing the chart tag:", hits)
for i in hits[-1:]:
    print("=" * 20, f"In[{i}]", "=" * 20)
    print(In[i][:8000])

In [ ]:
# ===== CELL 22C | DIAGNOSE MISSING (company, FY) VALUES IN CHARTS (self-contained) =====
import re
from collections import defaultdict
import pandas as pd

assert "FIGS_RAW" in globals(), "Run Cell 22 then 22B first (FIGS_RAW missing)."
GROUP_BY = globals().get("GROUP_BY", "company")


# ---- helpers (same logic as Cell 22B, copied here so this cell runs on its own) ----
def facet_names(fig):
    """Map each trace x-axis id ('x', 'x2', ...) to the facet title shown above that panel."""
    mapping = {}
    anns = [a for a in (fig.layout.annotations or []) if a.text and a.x is not None]
    for ax in fig.select_xaxes():
        ax_id = "x" + re.sub(r"\D", "", ax.plotly_name)
        dom = tuple(ax.domain) if ax.domain is not None else (0.0, 1.0)
        for ann in anns:
            try:
                inside = dom[0] - 1e-6 <= float(ann.x) <= dom[1] + 1e-6
            except (TypeError, ValueError):
                continue
            if inside:
                text = re.sub(r"<[^>]+>", "", str(ann.text))
                mapping[ax_id] = re.sub(r"^[^=]*=", "", text).strip()
                break
    return mapping


def period_label(value):
    text = str(value).strip()
    match = re.fullmatch(r"(?:FY)?\s*(\d{4})(?:\.0+)?", text, flags=re.I)
    return (f"FY{match.group(1)}", int(match.group(1))) if match else (text, 10 ** 9)


# ---- diagnosis ----
def scan_raw(fig):
    """key (company, FY) -> list of ('ok', value) / ('nan', raw). Also returns unmapped names."""
    facets = facet_names(fig)
    comps = set(COMPANIES)
    cells, unmapped = defaultdict(list), set()
    for trace in fig.data:
        xs = list(trace.x) if trace.x is not None else []
        ys = list(trace.y) if trace.y is not None else []
        axis = trace.xaxis or "x"
        for xv, yv in zip(xs, ys):
            if trace.name in comps:               company, period = trace.name, xv
            elif facets.get(axis) in comps:       company, period = facets[axis], xv
            elif str(xv) in comps:                company, period = str(xv), (trace.name or "Latest")
            elif facets.get(axis):                company, period = facets[axis], xv
            else:                                 company, period = (trace.name or "series"), xv
            if company not in comps:
                unmapped.add(str(company))
            label = period_label(period)[0]
            try:
                v = float(yv)
                status = ("nan", yv) if v != v else ("ok", v)
            except (TypeError, ValueError):
                status = ("nan", yv)
            cells[(company, label)].append(status)
    return cells, unmapped


def new_present(fig):
    """(company, FY) pairs that have a real value in the regrouped figure."""
    present = set()
    for t in fig.data:
        xs = list(t.x) if t.x is not None else []
        ys = list(t.y) if t.y is not None else []
        for x, y in zip(xs, ys):
            if y is None or y != y:
                continue
            present.add((str(x), t.name) if GROUP_BY == "company" else (t.name, str(x)))
    return present


rows = []
for (name, title, raw), (_, _, new) in zip(FIGS_RAW, FIGS):
    if not raw.data or any(t.type != "bar" for t in raw.data):
        continue
    cells, unmapped = scan_raw(raw)
    periods = sorted({p for _, p in cells}, key=lambda p: (period_label(p)[1], p))
    present = new_present(new)
    pretty = lambda c: str(c).replace("_", " ")

    for comp in COMPANIES:
        for p in periods:
            entries = cells.get((comp, p), [])
            oks = [v for s, v in entries if s == "ok"]
            if not entries:
                cause = "ABSENT: no bar in Cell 22 (upstream/data layer)"
            elif not oks:
                cause = f"NaN/None in Cell 22 (raw={entries[0][1]!r})"
            elif len({round(v, 6) for v in oks}) > 1:
                cause = f"COLLISION: {len(oks)} different values {sorted(set(oks))[:3]}, last wins"
            else:
                key = (pretty(comp), p) if GROUP_BY == "company" else (p, comp)
                if (pretty(comp), p) in present or (p, pretty(comp)) in present or key in present:
                    continue
                cause = "LOST IN 22B: value existed in Cell 22 but is missing in regrouped chart"
            rows.append({"metric": name, "company": comp, "FY": p, "cause": cause})

    if unmapped:
        rows.append({"metric": name, "company": "-", "FY": "-",
                     "cause": f"FACET MAPPING FAILED: unmapped names {sorted(unmapped)}"})

diag = pd.DataFrame(rows)
pd.set_option("display.max_rows", 500, "display.max_colwidth", 90, "display.width", 200)

if diag.empty:
    print("No missing (company, FY) values found in any bar chart.")
else:
    diag["type"] = diag["cause"].str.split(":|\\(", regex=True).str[0].str.strip()
    print("=== 1. Missing cells by cause ===")
    print(diag["type"].value_counts().to_string(), "\n")

    miss = diag[diag["company"] != "-"]
    print("=== 2. Missing count by FY (a whole FY missing everywhere = data-layer gap) ===")
    print(miss.groupby("FY").size().sort_index().to_string(), "\n")
    print("=== 3. Missing count by company ===")
    print(miss.groupby("company").size().sort_values(ascending=False).to_string(), "\n")
    print("=== 4. Missing count by metric ===")
    print(miss.groupby("metric").size().sort_values(ascending=False).to_string(), "\n")

    print("=== 5. Company x FY heatmap (number of metrics missing) ===")
    print(miss.pivot_table(index="company", columns="FY", values="metric",
                           aggfunc="count", fill_value=0).to_string(), "\n")

    print("=== 6. Full detail ===")
    print(diag.drop(columns="type").to_string(index=False))

In [ ]:
# ===== CELL 22E | EXACT PRESENT vs MISSING =====
import pandas as pd

rows = []
_printed = False
for name, title, raw in FIGS_RAW:
    if not raw.data or any(t.type != "bar" for t in raw.data):
        continue
    recs = extract_records(raw)
    if not recs:
        continue
    if not _printed:
        print("record length:", len(recs[0]), "| sample:", recs[0], "\n")
        _printed = True
    periods = {period_label(r[1])[0] for r in recs}
    present = {(r[0], period_label(r[1])[0]) for r in recs}
    total = len(COMPANIES) * len(periods)
    rows.append({"metric": name, "FYs_in_chart": len(periods),
                 "expected": total, "present": len(present),
                 "missing": total - len(present)})

df = pd.DataFrame(rows)
print(df[["expected", "present", "missing"]].sum().to_string())
print(f"coverage = {df.present.sum() / df.expected.sum():.1%}\n")
print(df.sort_values("missing", ascending=False).to_string(index=False))

In [ ]:
print(COMPANIES, {c: FYS[c] for c in COMPANIES})

In [ ]:
# ===== CELL 22F | FY labels per company in the charted data =====
from collections import defaultdict

core = ["revenue", "pat", "eps_final", "operating_cash_flow", "fcf", "net_debt"]
for name, title, raw in FIGS_RAW:
    if name not in core:
        continue
    by_co = defaultdict(set)
    for c, p, v in extract_records(raw):
        by_co[c].add(period_label(p)[0])
    print(name)
    for c in COMPANIES:
        print(f"   {c:22s} {sorted(by_co.get(c, []))}")

In [ ]:
# ===== CELL 23 | DISPLAY ALL QUESTIONS AND THE ANSWERS FOR EVERY SELECTED COMPANY ======
from IPython.display import display, Markdown

assert "ANSWERS" in globals(), "Run Cell 21 first to generate the answers."
assert len(ANSWERS) == len(BANK) * len(COMP), f"Expected {len(BANK) * len(COMP)} answers; found {len(ANSWERS)}."

for question in BANK:
    qid = str(question["qid"])

    display(Markdown(
        f"## {qid}: {question['question']}\n\n"
        f"**Category:** {question.get('category', '')}"
    ))

    for company in COMP:
        row = next(
            (
                answer for answer in ANSWERS
                if answer["qid"] == qid and answer["company"] == company
            ),
            None,
        )

        if row is None:
            answer_text = "_Answer missing._"
        else:
            answer_text = row.get("answer", "_Answer missing._")

        display(Markdown(f"### {company}\n\n{answer_text}"))

    display(Markdown("---"))

**Final cell: builds the report with every question and answer for each selected company and all charts, then downloads it.**

In [ ]:
# ===== CELL 23B | LIVE PRICE CHARTS: selectable windows with price and change per window =====
import time
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import display, Markdown

# (button label, period used for the cutoff, interval)
PRICE_WINDOWS = [
    ("1D", "1d", "5m"),
    ("1 month", "1mo", "1d"),
    ("3 months", "3mo", "1d"),
    ("6 months", "6mo", "1d"),
    ("1 year", "1y", "1d"),
    ("2 years", "2y", "1d"),
    ("3 years", "3y", "1d"),
    ("5 years", "5y", "1d"),
]
UP_COLOR, DOWN_COLOR = "#00b894", "#ff5a3d"

PRICE_CHARTS = []
PRICE_CHART_ERRORS = []
PRICE_CHANGES = []


def _clean_close(series):
    """Return numeric, sorted close prices with timezone-naive timestamps."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    idx = pd.to_datetime(s.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    s.index = idx
    return s.sort_index()


def _history(ticker, period, interval):
    """Download closing prices with retries; return (series, error text)."""
    last_error = ""
    for attempt in range(3):
        try:
            frame = yf.Ticker(ticker).history(
                period=period, interval=interval, auto_adjust=False
            )
            if frame is not None and not frame.empty and "Close" in frame.columns:
                series = _clean_close(frame["Close"])
                if not series.empty:
                    return series, ""
            last_error = "no rows returned"
        except Exception as error:
            last_error = f"{type(error).__name__}: {str(error)[:100]}"
        if attempt < 2:
            time.sleep(1.5 * (attempt + 1))
    return pd.Series(dtype=float), last_error


def _change(base, end):
    base, end = float(base), float(end)
    change = end - base
    pct = change / base * 100 if base else float("nan")
    return change, pct


def _price_header(end, change, pct, label):
    """HTML summary styled like a quote header, suitable for Plotly title."""
    color = UP_COLOR if change >= 0 else DOWN_COLOR
    sign = "+" if change >= 0 else ""
    return (
        f"<span style='font-size:28px;font-weight:700;color:#ffffff'>"
        f"₹{end:,.2f}</span>"
        f"&nbsp;&nbsp;"
        f"<span style='font-size:19px;font-weight:600;color:{color}'>"
        f"{sign}{change:,.2f} ({sign}{pct:.2f}%)</span>"
        f"&nbsp;&nbsp;"
        f"<span style='font-size:17px;color:#a8a8a8'>{label}</span>"
    )


def _title(company, ticker, label, end, change, pct, basis):
    header = _price_header(end, change, pct, label)
    return (
        f"<b>{company} ({ticker})</b><br>{header}"
        f"<br><span style='font-size:11px;color:#888'>{basis}</span>"
    )


for company in COMPANIES:
    symbol = META.get(company, {}).get("symbol")
    if not symbol:
        PRICE_CHART_ERRORS.append(
            f"{company}: skipped because its NSE symbol is unresolved."
        )
        continue

    ticker = f"{symbol}.NS"
    try:
        daily_close, err = _history(ticker, "5y", "1d")
        if daily_close.empty:
            PRICE_CHART_ERRORS.append(
                f"{company} ({ticker}): no daily price data ({err})."
            )
            continue

        intraday_close, _ = _history(ticker, "5d", "5m")
        if not intraday_close.empty:
            last_session = intraday_close.index.max().normalize()
            intraday_close = intraday_close[
                intraday_close.index.normalize() == last_session
            ]

        last_ts = daily_close.index.max()
        windows = {}

        for label, period, _interval in PRICE_WINDOWS:
            if label == "1D":
                if not intraday_close.empty:
                    session_date = intraday_close.index.max().normalize()
                    prev = daily_close[
                        daily_close.index.normalize() < session_date
                    ]
                    base = prev.iloc[-1] if len(prev) else intraday_close.iloc[0]
                    windows[label] = {
                        "values": intraday_close,
                        "base": base,
                        "end": intraday_close.iloc[-1],
                        "basis": "Change vs previous close; 5-minute prices",
                        "intraday": True,
                    }
                elif len(daily_close) >= 2:
                    windows[label] = {
                        "values": daily_close.tail(5),
                        "base": daily_close.iloc[-2],
                        "end": daily_close.iloc[-1],
                        "basis": "Intraday unavailable; latest daily close vs previous close",
                        "intraday": False,
                    }
                    PRICE_CHART_ERRORS.append(
                        f"{company} ({ticker}): intraday prices unavailable; "
                        "1D shows the latest daily close."
                    )
                continue

            if period.endswith("mo"):
                cutoff = last_ts - pd.DateOffset(months=int(period[:-2]))
            else:
                cutoff = last_ts - pd.DateOffset(years=int(period[:-1]))

            values = daily_close[daily_close.index >= cutoff]
            if values.empty:
                continue

            before = daily_close[daily_close.index < cutoff]
            base = before.iloc[-1] if len(before) else values.iloc[0]
            windows[label] = {
                "values": values,
                "base": base,
                "end": values.iloc[-1],
                "basis": "Change vs close just before the window; daily closes",
                "intraday": False,
            }

        if not windows:
            PRICE_CHART_ERRORS.append(
                f"{company} ({ticker}): no chartable price history."
            )
            continue

        labels = [label for label, _, _ in PRICE_WINDOWS]
        first_window = next(label for label in labels if label in windows)

        fig = go.Figure()
        titles = {}

        for label in labels:
            info = windows.get(label)
            if info is None:
                fig.add_trace(go.Scatter(x=[], y=[], name=label, visible=False))
                continue

            change, pct = _change(info["base"], info["end"])
            titles[label] = _title(
                company, ticker, label, float(info["end"]),
                change, pct, info["basis"]
            )

            PRICE_CHANGES.append({
                "company": company,
                "ticker": ticker,
                "window": label,
                "start_price": round(float(info["base"]), 2),
                "end_price": round(float(info["end"]), 2),
                "change_inr": round(change, 2),
                "change_pct": round(pct, 2),
            })

            fig.add_trace(go.Scatter(
                x=info["values"].index,
                y=info["values"].values,
                mode="lines",
                name=label,
                visible=(label == first_window),
                line={
                    "width": 2,
                    "color": UP_COLOR if change >= 0 else DOWN_COLOR,
                },
                hovertemplate=(
                    "%{x|%d %b %H:%M}" if info["intraday"]
                    else "%{x|%d %b %Y}"
                ) + "<br>₹%{y:,.2f}<extra></extra>",
            ))

        buttons = []
        for index, label in enumerate(labels):
            if label not in windows:
                continue
            buttons.append({
                "label": label,
                "method": "update",
                "args": [
                    {"visible": [i == index for i in range(len(labels))]},
                    {"title": {
                        "text": titles[label],
                        "x": 0.01,
                        "xanchor": "left",
                        "y": 0.98,
                        "yanchor": "top",
                    }},
                ],
            })

        fig.update_layout(
            title={
                "text": titles[first_window],
                "x": 0.01,
                "xanchor": "left",
                "y": 0.98,
                "yanchor": "top",
                "font": {"color": "#ffffff"},
            },
            template="plotly_dark",
            paper_bgcolor="#111111",
            plot_bgcolor="#111111",
            height=560,
            xaxis_title="Date",
            yaxis_title="Price (INR)",
            hovermode="x unified",
            showlegend=False,
            updatemenus=[{
                "type": "buttons",
                "direction": "right",
                "x": 0,
                "y": 1.16,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": buttons,
                "font": {"color": "#ffffff"},
                "bgcolor": "#222222",
                "bordercolor": "#555555",
                "active": next(
                    i for i, label in enumerate(labels) if label == first_window
                ),
            }],
            margin={"t": 155, "b": 50, "l": 70, "r": 30},
        )

        fig.update_xaxes(
            gridcolor="#333333",
            zerolinecolor="#444444",
            linecolor="#444444",
        )
        fig.update_yaxes(
            gridcolor="#333333",
            zerolinecolor="#444444",
            linecolor="#444444",
        )

        PRICE_CHARTS.append((company, ticker, fig))
        display(Markdown(f"### {company} — {ticker}"))
        fig.show()

    except Exception as error:
        PRICE_CHART_ERRORS.append(
            f"{company} ({ticker}): price chart failed — "
            f"{type(error).__name__}: {error}"
        )

if PRICE_CHANGES:
    PRICE_CHANGE_DF = pd.DataFrame(PRICE_CHANGES)
    order = [label for label, _, _ in PRICE_WINDOWS]
    summary = (
        PRICE_CHANGE_DF.pivot(
            index="company", columns="window", values="change_pct"
        )
        .reindex(columns=[
            window for window in order
            if window in set(PRICE_CHANGE_DF["window"])
        ])
    )
    display(Markdown("**Price change by window (%)**"))
    display(summary.round(2))
    display(Markdown("**Price change by window (INR and %), full detail**"))
    display(PRICE_CHANGE_DF)

if PRICE_CHART_ERRORS:
    display(Markdown(
        "**Price chart notes**\n\n"
        + "\n".join(f"- {message}" for message in PRICE_CHART_ERRORS)
    ))

In [ ]:
# ===== CELL 24 | FINAL REPORT: full-width charts, answers, and share-price charts =====
import datetime as dt
import html as _html
import re
from plotly.offline import get_plotlyjs

assert "ANSWERS" in globals() and ANSWERS, "Run Cell 21 first."
assert "FIGS" in globals() and FIGS, "Run Cells 22 and 22B first."
assert "PRICE_CHARTS" in globals(), "Run Cell 23B first."

esc = lambda s: _html.escape(str(s if s is not None else ""))
FACT_BY_ID = {f["fact_id"]: f for f in FACTS} if "FACTS" in globals() else {}
dname = lambda company: (
    f"{company} - {META[company]['name']}"
    if META.get(company, {}).get("name") not in (None, company)
    else company
)
CHART_CONFIG = {"displaylogo": False, "responsive": True}


def answer_html(text):
    value = esc(text)
    value = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", value)
    value = re.sub(
        r"F\d{4}",
        lambda match: f'<a class="fid" href="#fact-{match.group(0)}">{match.group(0)}</a>',
        value,
    )
    return value.replace("\n", "<br>")


def fmt_val(value, unit):
    try:
        return f"{float(value):,.2f} {unit}".strip()
    except Exception:
        return f"{value} {unit}".strip()


def facts_table(company, answers):
    fact_ids = sorted({
        fact_id
        for answer in answers
        for fact_id in re.findall(r"F\d{4}", str(answer.get("answer", "")))
        if fact_id in FACT_BY_ID
        and FACT_BY_ID[fact_id]["company"] == company
    })
    if not fact_ids:
        return ""

    rows = []
    for fact_id in fact_ids:
        fact = FACT_BY_ID[fact_id]
        sources = "; ".join(
            f"{item['variable']} FY{item['fy']} = {item['value']:,.2f} "
            f"({esc(item['source_document'])}"
            + (f", p.{item['source_page']}" if item.get("source_page") else "")
            + ")"
            for item in fact.get("inputs", [])
        ) or esc(fact.get("note") or fact.get("source_label"))

        rows.append(
            f'<tr id="fact-{fact_id}">'
            f"<td>{fact_id}</td><td>{esc(fact['metric'])}</td>"
            f"<td>FY{fact['fy']}</td>"
            f"<td>{esc(fmt_val(fact['value'], fact['unit']))}</td>"
            f"<td>{esc(fact['source_label'])}</td>"
            f'<td class="small">{sources}</td></tr>'
        )

    return (
        "<details><summary>Fact register: numbers cited in these answers "
        f"({len(fact_ids)})</summary><table>"
        "<tr><th>ID</th><th>Metric</th><th>FY</th><th>Value</th>"
        f"<th>Type</th><th>Source rows</th></tr>{''.join(rows)}</table></details>"
    )


# Company identity and data-quality table
quality_rows = []
for company in COMPANIES:
    identity = IDENT.get(company, {})
    market = MARKET.get(company, {})
    quality = (
        QUALITY[QUALITY.company == company].iloc[0]
        if "QUALITY" in globals()
        and len(QUALITY[QUALITY.company == company])
        else None
    )

    extracted = (
        f"<td>{int(quality.cells)} cells: {int(quality.reported)} reported, "
        f"{int(quality.derived)} derived, {int(quality.from_web)} from web; "
        f"review {int(quality.review)}, missing {int(quality.missing)}, "
        f"sanity findings {int(quality.sanity_findings)}</td>"
        if quality is not None
        else "<td>n/a</td>"
    )

    quality_rows.append(
        f"<tr><td><b>{esc(company)}</b></td>"
        f"<td>{esc(META[company]['name'])}</td>"
        f"<td>{esc(META[company].get('symbol') or 'not resolved')}<br>"
        f"<span class='small'>{esc(META[company].get('isin') or '')}</span></td>"
        f"<td>{esc(identity.get('status', ''))}</td>"
        f"<td>FY{FYS[company][0]}–FY{FYS[company][-1]} "
        f"({len(FYS[company])} reports)</td>"
        f"<td>{esc(market.get('web_check', 'no web data'))}</td>"
        f"{extracted}</tr>"
    )

clean = bool(globals().get("FROZEN_CLEAN", False))
open_gates = globals().get("OPEN_GATES", {})
quality_note = (
    "All data-layer gates passed."
    if clean
    else (
        "Numbers were extracted automatically and have not been hand-checked. "
        "Open data-layer items: "
        + (", ".join(
            f"{key}: {value}" for key, value in open_gates.items()
        ) or "none recorded")
        + ". Source files and pages are listed in tables/variable_table_final.csv."
    )
)


# Financial charts
chart_blocks = []
import plotly.graph_objects as go
for index, (name, title, figure) in enumerate(FIGS):
    figure = go.Figure(figure)                                   # copy, so the notebook figure is untouched
    figure.update_layout(title=None, margin={"t": 34})           # the HTML <h3> already shows the title
    chart_html = figure.to_html(
        full_html=False,
        include_plotlyjs=False,
        div_id=f"financial_chart_{index}",
        config=CHART_CONFIG,
        default_width="100%",
        default_height=f"{figure.layout.height or 320}px",
    )
    chart_blocks.append(
        '<div class="chart">'
        f"<h3>{esc(title)}</h3>"
        f"{chart_html}"
        "</div>"
    )


# Share-price charts
price_chart_blocks = []
for index, (company, ticker, figure) in enumerate(PRICE_CHARTS):
    chart_html = figure.to_html(
        full_html=False,
        include_plotlyjs=False,
        div_id=f"price_chart_{index}",
        config=CHART_CONFIG,
        default_width="100%",
        default_height=f"{figure.layout.height or 500}px",
    )
    price_chart_blocks.append(
        '<div class="chart price-chart">'
        f"<h3>{esc(company)} — share price ({esc(ticker)})</h3>"
        f"{chart_html}"
        "</div>"
    )


# Market-data/API status for the report header.
web_log = globals().get("WEB_LOG", [])
yf_ok = sum(1 for row in web_log if row.get("yf_cells", 0) > 0)
screener_ok = sum(1 for row in web_log if row.get("screener_cells", 0) > 0)
price_chart_count = len(PRICE_CHARTS)

market_status = (
    f"Market data: Yahoo Finance {yf_ok}/{len(COMPANIES)}, "
    f"Screener {screener_ok}/{len(COMPANIES)}; "
    f"price charts {price_chart_count}/{len(COMPANIES)}"
    " (Screener cache may be used)"
)


# Answers grouped by company
answers_by_company = {
    company: [answer for answer in ANSWERS if answer["company"] == company]
    for company in COMPANIES
}

category_order = []
for question in BANK:
    if question["category"] not in category_order:
        category_order.append(question["category"])

contents = "".join(
    f'<li><a href="#company-{esc(company)}">{esc(dname(company))}</a></li>'
    for company in COMPANIES
)

answer_sections = []
for company in COMPANIES:
    answers = {
        answer["qid"]: answer
        for answer in answers_by_company[company]
    }
    inner = []

    for category in category_order:
        inner.append(f"<h3>{esc(category)}</h3>")

        for question in [q for q in BANK if q["category"] == category]:
            answer = answers.get(str(question["qid"]))
            body = (
                answer_html(answer["answer"])
                if answer
                else "<i>Answer missing.</i>"
            )
            warning = (
                ""
                if answer and answer.get("status") == "generated"
                else ' <span class="warn">(generation failed)</span>'
            )
            inner.append(
                f'<div class="qa"><div class="q">'
                f'<span class="qid">{esc(question["qid"])}</span> '
                f"{esc(question['question'])}</div>"
                f'<div class="a">{body}{warning}</div></div>'
            )

    answer_sections.append(
        f'<section class="company" id="company-{esc(company)}">'
        f"<h2>{esc(dname(company))}: questions and answers</h2>"
        '<details class="qa-toggle">'
        f"<summary>Show all {len(BANK)} questions and answers "
        f"for {esc(company)}</summary>"
        f"{''.join(inner)}</details>"
        f"{facts_table(company, answers_by_company[company])}"
        "</section>"
    )


assumptions = (
    f"Cost of equity {COST_OF_EQUITY:.0%}; terminal growth {TERMINAL_G:.0%}; "
    f"{DCF_YEARS}-year explicit period for reverse DCF. Stress test: revenue "
    f"-{STRESS_REV_DROP:.0%}, borrowing cost +{STRESS_RATE_UP * 1e4:.0f} bps. "
    f"Debt is borrowings only (leases "
    f"{'included' if INCLUDE_LEASES_IN_DEBT else 'excluded'}). "
    f"Reporting basis: {BASIS}. EBIT = PBT + finance costs; "
    "EBITDA = EBIT + D&A (includes other income). Market multiples use the "
    "run-date price against the latest financial year's figures."
)

CSS = """
*{box-sizing:border-box}
html,body{overflow-x:hidden}
body{font-family:Arial,sans-serif;max-width:1180px;margin:24px auto;padding:0 18px;
color:#1d2433;line-height:1.5}
h1{margin-bottom:2px}
h2{border-bottom:2px solid #1f4e79;padding-bottom:4px;margin-top:38px;color:#1f4e79}
h3{margin:18px 0 6px;color:#333}
table{border-collapse:collapse;width:100%;font-size:12px;margin:10px 0;table-layout:auto}
th,td{border:1px solid #cfd6e0;padding:5px 8px;text-align:left;vertical-align:top;
overflow-wrap:anywhere;word-break:break-word}
th{background:#eef2f8}
.small{font-size:11px;color:#667}
.meta{color:#556;font-size:13px;overflow-wrap:anywhere}
.note{background:#fff6e0;border-left:4px solid #e0a800;padding:8px 12px;margin:12px 0;
overflow-wrap:anywhere}
.ok{background:#e9f7ee;border-left-color:#2e9e5b}
.qa{margin:10px 0 14px}
.q{font-weight:600}
.qid{background:#1f4e79;color:#fff;border-radius:3px;padding:0 6px;margin-right:6px}
.a{margin:3px 0 0 4px;padding-left:10px;border-left:3px solid #cfd6e0;overflow-wrap:anywhere}
.fid{font-size:11px;color:#1f4e79}
.warn{color:#b00020}
.chart{min-width:0;width:100%;margin:0}
.chart h3{font-size:14px;margin:2px 0 0}
.chart-grid{display:grid;grid-template-columns:minmax(0,1fr);gap:18px}
.chart-grid .chart{width:100%;min-width:0;border:1px solid #e3e8ef;border-radius:6px;padding:8px 10px}
.chart-grid .plotly-graph-div{width:100%!important}
.price-grid{display:grid;grid-template-columns:1fr;gap:16px}
.price-grid .chart{border:1px solid #e3e8ef;border-radius:6px;padding:8px 10px}
.price-grid .chart h3{text-transform:none;font-size:15px}
details.qa-toggle{border:1px solid #cfd6e0;border-radius:6px;margin:10px 0;padding:0 14px}
details.qa-toggle>summary{cursor:pointer;font-weight:600;padding:10px 0;color:#1f4e79}
details>summary{cursor:pointer}
.tools button{border:1px solid #1f4e79;background:#fff;color:#1f4e79;border-radius:4px;
padding:4px 10px;margin-right:6px;cursor:pointer;font-size:12px}
.tools button:hover{background:#eef2f8}
@media(max-width:600px){
  body{padding:0 10px}
  .chart-grid{gap:12px}
  .chart-grid .chart{padding:6px}
}
"""

TOGGLE_JS = (
    "function toggleAll(open){document.querySelectorAll('details.qa-toggle')"
    ".forEach(function(d){d.open=open;});}"
)

document = (
    "<!DOCTYPE html><html><head><meta charset='utf-8'>"
    "<meta name='viewport' content='width=device-width, initial-scale=1'>"
    f"<title>{esc(REPORT_TITLE)} - {esc(RUN_NAME)}</title>"
    f"<style>{CSS}</style>"
    f"<script>{get_plotlyjs()}</script></head><body>"
    f"<h1>{esc(REPORT_TITLE)}</h1>"
    f"<div class='meta'>Companies: {esc(', '.join(COMPANIES))} | "
    f"Generated {dt.datetime.now().strftime('%Y-%m-%d %H:%M')} | "
    f"{len(FIGS)} financial charts | "
    f"{len(BANK)} questions, {len(ANSWERS)} answers | "
    f"{esc(market_status)}</div>"
    f"<div class='note {'ok' if clean else ''}'>{esc(quality_note)}</div>"
    "<h2>1. Scope, identity and data quality</h2>"
    "<table><tr><th>Key</th><th>Official name</th><th>NSE symbol / ISIN</th>"
    "<th>Identity check</th><th>Coverage</th><th>Web data check</th>"
    f"<th>Extracted cells</th></tr>{''.join(quality_rows)}</table>"
    f"<h2>2. Financial charts</h2>"
    f"<div class='chart-grid'>{''.join(chart_blocks)}</div>"
    f"<h2>3. Share-price charts</h2>"
    f"<div class='price-grid'>{''.join(price_chart_blocks)}</div>"
    "<h2>4. Contents and answers</h2>"
    f"<ul>{contents}</ul>"
    "<div class='tools'><button onclick='toggleAll(true)'>Expand all answers</button>"
    "<button onclick='toggleAll(false)'>Collapse all answers</button></div>"
    f"{''.join(answer_sections)}"
    f"<h2>5. Assumptions and method</h2><p>{esc(assumptions)}</p>"
    "<p class='small'>Share-price history is fetched from Yahoo Finance when "
    "the price-chart cell runs. Screener data may be cached. Financial chart "
    "hover details show the reported values used to calculate each plotted "
    "measure. Missing source values remain blank.</p>"
    f"<script>{TOGGLE_JS}</script>"
    "</body></html>"
)

REPORT_PATH = OUT / f"Report_{RUN_NAME}_{TODAY}.html"
REPORT_PATH.write_text(document, encoding="utf-8")

print(
    f"Report written: {REPORT_PATH} | "
    f"{REPORT_PATH.stat().st_size / 1e6:.1f} MB | "
    f"{len(COMPANIES)} companies, {len(FIGS)} financial charts, "
    f"{len(PRICE_CHARTS)} price charts, "
    f"{len(BANK)} questions, {len(ANSWERS)} answers"
)

if AUTO_DOWNLOAD_REPORT:
    try:
        from google.colab import files
        files.download(str(REPORT_PATH))
        print("HTML report download started.")
    except Exception as error:
        print(
            "Automatic download unavailable; report is at",
            REPORT_PATH,
            error,
        )

In [ ]:
# ===== CELL 25 | RESEARCH SETUP: select ONE company, auto-detect its sector, confirm =====
import datetime as _dt
import os
import subprocess
import sys

import ipywidgets as W
from IPython.display import display

try:
    from google import genai
    from google.genai import types
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    from google import genai
    from google.genai import types

# ---------------- settings ----------------
GEMINI_MODEL = "gemini-3.8-flash"     # confirm the model name is current on your account
# ------------------------------------------

assert "COMPANIES" in globals() and COMPANIES, "Run the earlier pipeline cells (company selection) first."


def get_api_key():
    key = globals().get("GEMINI_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    from getpass import getpass
    return getpass("Paste Gemini API key (hidden): ")


GEMINI_CLIENT = genai.Client(api_key=get_api_key())
RESEARCH = None          # filled when you press Confirm


def _meta(company):
    return (globals().get("META") or {}).get(company, {})


def _official(company):
    name = _meta(company).get("name") or company
    if name == company:
        name = company.replace("_", " ").title()
    symbol = _meta(company).get("symbol")
    return f"{name} (NSE: {symbol})" if symbol else name


def detect_sector(company):
    """Ask Gemini (with Google Search) for a short sector label. Falls back to Yahoo's industry field."""
    symbol = _meta(company).get("symbol")
    yahoo_hint = ""
    if symbol:
        try:
            import yfinance as yf
            info = yf.Ticker(f"{symbol}.NS").info or {}
            yahoo_hint = ", ".join(x for x in (info.get("industry"), info.get("sector")) if x)
        except Exception:
            pass

    prompt = (
        f"Company: {_official(company)}. Listed in India."
        + (f" Yahoo Finance classifies it as: {yahoo_hint}." if yahoo_hint else "")
        + "\nIn 2 to 5 words, name the main sector this company operates in, the way an Indian equity "
          "investor would describe it (for example: 'Defence electronics', 'Quick commerce', "
          "'Power transmission'). Reply with ONLY that label, nothing else."
    )
    try:
        response = GEMINI_CLIENT.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())],
                temperature=0.0,
                max_output_tokens=60,
            ),
        )
        text = (response.text or "").strip().splitlines()[0].strip(" .\"'*`")
        if text:
            return text[:60], "detected by Gemini"
    except Exception as error:
        if yahoo_hint:
            return yahoo_hint.split(",")[0].strip(), f"Gemini failed ({type(error).__name__}); using Yahoo industry"
        return "", f"detection failed ({type(error).__name__}); type the sector"
    return (yahoo_hint.split(",")[0].strip(), "using Yahoo industry") if yahoo_hint else ("", "could not detect; type the sector")


company_dd = W.Dropdown(
    options=[(f"{c}  -  {_official(c)}", c) for c in COMPANIES],
    description="Company:",
    layout=W.Layout(width="640px"),
    style={"description_width": "80px"},
)
sector_tx = W.Text(
    description="Sector:",
    layout=W.Layout(width="640px"),
    style={"description_width": "80px"},
)
sector_note = W.HTML("")
date_tx = W.Text(
    value=_dt.date.today().isoformat(),
    description="Date:",
    layout=W.Layout(width="640px"),
    style={"description_width": "80px"},
)
confirm_btn = W.Button(description="Confirm selection", button_style="success", icon="check")
status = W.Output()


def _refresh_sector(_=None):
    sector_note.value = "<i>detecting sector...</i>"
    label, note = detect_sector(company_dd.value)
    sector_tx.value = label
    sector_note.value = f"<span style='color:#667'>Sector {note}. You can edit it before confirming.</span>"


def _on_confirm(_):
    global RESEARCH
    with status:
        status.clear_output()
        sector = sector_tx.value.strip()
        if not sector:
            print("Sector is empty: type it, then press Confirm again.")
            return
        key = company_dd.value
        RESEARCH = {
            "company_key": key,
            "company_name": _official(key),
            "sector_name": sector,
            "research_date": date_tx.value.strip() or _dt.date.today().isoformat(),
        }
        print("CONFIRMED")
        for k, v in RESEARCH.items():
            print(f"  {k}: {v}")
        print("Now run Cell 26.")


company_dd.observe(lambda change: _refresh_sector() if change["name"] == "value" else None, names="value")
confirm_btn.on_click(_on_confirm)

display(W.VBox([
    W.HTML("<b>Select ONE company for the Gemini research calls</b>"),
    company_dd, sector_tx, sector_note, date_tx, confirm_btn, status,
]))
_refresh_sector()

In [ ]:
from google.genai import types
try:
    r = GEMINI_CLIENT.models.generate_content(
        model="gemini-3.1-flash-lite", contents="Reply with the single word OK.",
        config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())], max_output_tokens=20),
    )
    print("SEARCH WORKS:", r.text)
except Exception as e:
    print(str(e))
    print("--- details ---")
    print(getattr(e, "details", None))

In [ ]:
# ===== CELL 26 | RESEARCH ANSWERS: Tavily web search -> Gemini writes compact, complete, cited answers. Nothing is saved. =====
import json
import re
import time
from pathlib import Path

import requests
from IPython.display import display, Markdown
from google.genai import types

# ---------------- settings ----------------
DRIVE_FOLDER = BASE_DIR     # only used to READ the questions file
QUESTIONS_JSON_PATH = DRIVE_FOLDER / "LLM_api_calling_questions.json"
MODEL_CHAIN = ["gemini-3.5-flash", "gemini-3.1-flash-lite"]   # tried in order if one is overloaded / out of quota
ANSWER_MAX_WORDS = 300              # default length of each answer
ANSWER_MAX_BULLETS = 10
WORD_LIMITS = {"Q6": 380}           # optional per-question override (Q6 has the most sub-parts)
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 4096            # headroom only; the word limit above controls the length
PAUSE_BETWEEN_CALLS = 15            # seconds between questions (free tier: 5 requests/min on some models)
MAX_RETRIES = 3
RESULTS_PER_QUERY = 5               # web results fetched per search
MAX_SOURCES = 12                    # numbered sources given to Gemini per question
SNIPPET_CHARS = 1000                # characters of each page snippet given to Gemini
# ------------------------------------------

if not DRIVE_FOLDER.exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
if not QUESTIONS_JSON_PATH.exists():
    found = sorted(DRIVE_FOLDER.rglob("LLM_api_calling_questions.json"))
    if found:
        QUESTIONS_JSON_PATH = found[0]
print("Questions file:", QUESTIONS_JSON_PATH)

assert globals().get("RESEARCH"), "Run Cell 25 and press 'Confirm selection' first."
assert "GEMINI_CLIENT" in globals(), "Run Cell 25 first."
assert QUESTIONS_JSON_PATH.exists(), f"Questions file not found: {QUESTIONS_JSON_PATH}"


def get_tavily_key():
    import os
    key = globals().get("TAVILY_API_KEY") or os.environ.get("TAVILY_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get("TAVILY_API_KEY")
    except Exception:
        pass
    from getpass import getpass
    return getpass("Paste Tavily API key (hidden): ")


TAVILY_KEY = get_tavily_key()

spec = json.loads(QUESTIONS_JSON_PATH.read_text(encoding="utf-8"))
VARS = {
    "company_name": RESEARCH["company_name"],
    "sector_name": RESEARCH["sector_name"],
    "research_date": RESEARCH["research_date"],
}


def fill(text):
    for name, value in VARS.items():
        text = text.replace("{" + name + "}", value)
    return text


# Rules from the JSON are kept, except the ones that ask for long structure (we want compact answers).
SKIP_RULE_STARTS = ("use web search", "conclude with", "keep each answer concise")
SYSTEM = (
    "You are a careful equity-research analyst. You are given numbered web sources retrieved on the research date.\n"
    "Research rules:\n"
    + "\n".join("- " + fill(rule) for rule in spec["shared_research_instructions"]
                if not rule.lower().startswith(SKIP_RULE_STARTS))
    + "\n- Base every external fact ONLY on the numbered sources provided in the prompt, and cite them as [n] "
      "immediately after the claim. Never cite a number that is not in the list and never type URLs yourself.\n"
      "- If the sources do not cover something, say 'Not found in the retrieved sources' in a few words.\n"
      "- If sources conflict on a number, say so briefly and cite both.\n"
      "- Prefix each bullet with 'Fact:', 'Forecast:' or 'Inference:'. Inferences are your own and need no citation.\n"
      "- Write money as 'Rs' or 'INR' or 'USD', never with the dollar symbol.\n"
      f"- FORMAT: at most {ANSWER_MAX_WORDS} words and {ANSWER_MAX_BULLETS} bullet points. "
      "No headings, no tables, no conclusion section, no preamble.\n"
      "- COVERAGE: address EVERY sub-question asked, each with at least one bullet, keeping the key numbers "
      "(amounts, percentages, years) with their dates. To save space, drop background, repetition and generic "
      "statements, never a requested item. If the sources have nothing on a sub-question, write one short bullet "
      "'Not found in the retrieved sources: <that item>'.\n"
      "- SIGNAL: after the last bullet add ONE final line in exactly this form: "
      "'SIGNAL: positive | <reason in max 12 words>' where the label is positive, mixed or negative. "
      "It states whether this answer's findings are favourable, mixed or unfavourable for the company's prospects; "
      "judge only from the evidence in your bullets, and use mixed when favourable and unfavourable evidence both matter. "
      "This line does not count toward the word limit."
)

QUESTIONS = [
    {"qid": q["id"], "title": q["title"], "prompt": fill(q["prompt"])}
    for q in spec["questions"]
]

C, S = RESEARCH["company_name"].split(" (NSE")[0], RESEARCH["sector_name"]
YEAR = RESEARCH["research_date"][:4]
SEARCH_QUERIES = {
    "Q1": [f"{S} India Union Budget allocation schemes incentives {YEAR}",
           f"{S} India government scheme subsidy PLI projects {C}"],
    "Q2": [f"global {S} market trends demand outlook {YEAR}",
           f"{S} geopolitics supply chain global spending trends"],
    "Q3": [f"{S} India market size growth forecast {YEAR}",
           f"global {S} market forecast CAGR competition risks"],
    "Q4": [f"{C} annual report order book revenue growth profitability",
           f"{C} competitors peers {S} India market share"],
    "Q5": [f"{C} share price valuation PE analyst outlook {YEAR}",
           f"{C} risks catalysts guidance next 3 years"],
    "Q6": [f"{C} new product launch order win capacity expansion {YEAR}",
           f"{C} upcoming contracts partnership acquisition approvals"],
}


def tavily_search(query):
    last = ""
    for attempt in range(3):
        try:
            r = requests.post(
                "https://api.tavily.com/search",
                headers={"Authorization": f"Bearer {TAVILY_KEY}", "Content-Type": "application/json"},
                json={"query": query, "search_depth": "basic", "max_results": RESULTS_PER_QUERY,
                      "include_answer": False},
                timeout=40,
            )
            if r.status_code == 200:
                return r.json().get("results", []), ""
            last = f"HTTP {r.status_code}: {r.text[:200]}"
            if r.status_code in (401, 403, 432):
                break
        except Exception as error:
            last = f"{type(error).__name__}: {str(error)[:150]}"
        time.sleep(3 * (attempt + 1))
    return [], last


def collect_sources(qid):
    sources, seen, used, errors = [], set(), [], []
    for query in SEARCH_QUERIES.get(qid, [f"{C} {S}"]):
        results, error = tavily_search(query)
        used.append(query)
        if error:
            errors.append(error)
        for item in results:
            url = (item.get("url") or "").strip()
            if not url or url in seen or len(sources) >= MAX_SOURCES:
                continue
            seen.add(url)
            sources.append({
                "n": len(sources) + 1,
                "title": (item.get("title") or "").strip(),
                "url": url,
                "snippet": (item.get("content") or "").strip()[:SNIPPET_CHARS],
                "date": item.get("published_date") or "",
            })
    return sources, used, "; ".join(errors)


def build_prompt(question, sources):
    blocks = []
    for s in sources:
        date = f" | published {s['date']}" if s["date"] else ""
        blocks.append(f"[{s['n']}] {s['title']} | {s['url']}{date}\n{s['snippet']}")
    return (
        f"{question['prompt']}\n\n"
        f"ANSWER FORMAT: at most {WORD_LIMITS.get(question['qid'], ANSWER_MAX_WORDS)} words and "
        f"{ANSWER_MAX_BULLETS} bullets in total, each bullet with its [n] citations. "
        "Cover every sub-question asked above with at least one bullet, keeping key numbers and dates.\n\n"
        f"WEB SOURCES (retrieved {RESEARCH['research_date']}):\n\n" + "\n\n".join(blocks)
    )


def cited_numbers(text):
    numbers = set()
    for group in re.findall(r"\[(\d+(?:\s*,\s*\d+)*)\]", text):
        numbers.update(int(x) for x in re.split(r"\s*,\s*", group))
    return numbers


def call_gemini(prompt):
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM, temperature=TEMPERATURE, max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    last_error = ""
    for model in MODEL_CHAIN:
        for attempt in range(MAX_RETRIES):
            try:
                response = GEMINI_CLIENT.models.generate_content(model=model, contents=prompt, config=config)
                if (response.text or "").strip():
                    return response, model, ""
                last_error = f"{model}: empty response"
                break
            except Exception as error:
                message = str(error)
                code = getattr(error, "code", None)
                last_error = f"{model}: {type(error).__name__}: {message[:300]}"
                if code == 429 or "RESOURCE_EXHAUSTED" in message:
                    break                                   # quota: go to the next model
                if code in (500, 503, 504) or "UNAVAILABLE" in message:
                    time.sleep(10 * (attempt + 1))
                    continue                                # overload: retry this model
                break
    return None, None, last_error


GEMINI_RESULTS = []      # kept in memory only; Cell 27 reads it. Nothing is written to Drive.
print(f"Researching: {RESEARCH['company_name']} | sector: {RESEARCH['sector_name']} | models: {MODEL_CHAIN}")

for position, question in enumerate(QUESTIONS):
    qid = question["qid"]
    print(f"{qid}: searching the web ...")
    sources, used_queries, search_error = collect_sources(qid)
    if not sources:
        GEMINI_RESULTS.append({
            "qid": qid, "title": question["title"], "answer": "", "sources": [], "queries": used_queries,
            "finish": "", "status": "failed", "error": "no web results retrieved. " + search_error,
        })
        print(f"{qid}: FAILED - no web results. {search_error}")
        continue

    print(f"{qid}: {len(sources)} sources, asking Gemini ...")
    response, model_used, error = call_gemini(build_prompt(question, sources))
    if response is None:
        GEMINI_RESULTS.append({
            "qid": qid, "title": question["title"], "answer": "", "sources": [], "queries": used_queries,
            "finish": "", "status": "failed", "error": error,
        })
        print(f"{qid}: FAILED - {error}")
    else:
        answer = response.text.strip()
        signal, signal_reason = "unknown", ""
        sig_matches = list(re.finditer(r"SIGNAL\W*\s*(positive|mixed|negative)\W*(.*)$", answer, flags=re.I | re.M))
        if sig_matches:
            sig = sig_matches[-1]
            signal = sig.group(1).lower()
            signal_reason = sig.group(2).strip(" |*-:_")
            answer = answer[:sig.start()].rstrip(" \t\n*•-_")      # remove the tag line from the displayed answer
        cited = cited_numbers(answer)
        kept = [s for s in sources if s["n"] in cited] or sources      # only sources the answer actually cites
        candidate = response.candidates[0] if getattr(response, "candidates", None) else None
        finish = str(getattr(candidate, "finish_reason", "") or "")
        GEMINI_RESULTS.append({
            "qid": qid, "title": question["title"], "answer": answer,
            "sources": [{"n": s["n"], "title": s["title"], "url": s["url"]} for s in kept],
            "queries": used_queries, "finish": finish, "status": "ok", "error": "", "model": model_used,
            "signal": signal, "signal_reason": signal_reason,
        })
        note = " (output may be truncated: raise MAX_OUTPUT_TOKENS)" if "MAX_TOKENS" in finish else ""
        print(f"{qid}: ok via {model_used}, {len(answer.split())} words, {len(kept)} sources cited, tone: {signal}{note}")

    if position < len(QUESTIONS) - 1:
        time.sleep(PAUSE_BETWEEN_CALLS)

esc_md = lambda t: str(t).replace("$", r"\$")      # stop Jupyter treating $...$ as maths
for result in GEMINI_RESULTS:
    if result["status"] != "ok":
        display(Markdown(f"### {result['qid']} - {result['title']}\n\n**Failed:** {esc_md(result['error'])}"))
        continue
    source_lines = "\n".join(
        f"{s['n']}. [{esc_md(s['title'] or s['url'])}]({s['url']})" for s in result["sources"]
    ) or "_No sources._"
    display(Markdown(
        f"### {result['qid']} - {result['title']}\n\n{esc_md(result['answer'])}\n\n**Sources**\n\n{source_lines}"
    ))

print("Done. Answers are held in memory (GEMINI_RESULTS). Run Cell 27 to add them to the HTML report.")



from collections import Counter

esc_md = lambda t: str(t).replace("$", r"\$")      # stop Jupyter treating $...$ as maths

ok_results = [r for r in GEMINI_RESULTS if r["status"] == "ok"]
tally = Counter(r.get("signal", "unknown") for r in ok_results)
failed_count = len(GEMINI_RESULTS) - len(ok_results)
SIGNAL_SUMMARY = (
    f"Answer tone across {len(ok_results)} answers: {tally.get('positive', 0)} positive, "
    f"{tally.get('mixed', 0)} mixed, {tally.get('negative', 0)} negative"
    + (f", {tally.get('unknown', 0)} unlabelled" if tally.get("unknown") else "")
    + (f" ({failed_count} question(s) failed and are not counted)" if failed_count else "")
)

for result in GEMINI_RESULTS:
    if result["status"] != "ok":
        display(Markdown(f"### {result['qid']} - {result['title']}\n\n**Failed:** {esc_md(result['error'])}"))
        continue
    source_lines = "\n".join(
        f"{s['n']}. [{esc_md(s['title'] or s['url'])}]({s['url']})" for s in result["sources"]
    ) or "_No sources._"
    tone = result.get("signal", "unknown")
    reason = f" - {esc_md(result['signal_reason'])}" if result.get("signal_reason") else ""
    display(Markdown(
        f"### {result['qid']} - {result['title']}\n\n**Answer tone: {tone}**{reason}\n\n"
        f"{esc_md(result['answer'])}\n\n**Sources**\n\n{source_lines}"
    ))

display(Markdown(
    f"## {SIGNAL_SUMMARY}\n\n"
    "_Tone labels are the model's reading of each answer, not a recommendation. Q1 to Q3 describe the sector and "
    "government context, Q4 to Q6 the company._"
))
print("Done. Answers are held in memory (GEMINI_RESULTS). Run Cell 27 to add them to the HTML report.")

In [ ]:
# ===== CELL 27 | SINGLE-COMPANY REPORT: only the company selected in Cell 25 (analysis + Gemini research) =====
import copy
import html as _html
import re
import datetime as dt
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs

needed = ["CSS", "TOGGLE_JS", "answer_html", "facts_table", "ANSWERS", "BANK", "FIGS", "PRICE_CHARTS",
          "META", "IDENT", "FYS", "REPORTS", "OUT", "REPORT_TITLE", "RUN_NAME", "TODAY"]
missing_names = [n for n in needed if n not in globals()]
assert not missing_names, f"Run Cell 24 first (it defines: {missing_names})."
assert globals().get("GEMINI_RESULTS") and globals().get("RESEARCH"), "Run Cells 25 and 26 first."

key = RESEARCH["company_key"]
assert key in META and key in FYS, (
    f"{key} not found in META/FYS ({list(META)}). Re-run Cell 25 and pick the company again."
)
_c = lambda x: re.sub(r"(limited|ltd)$", "", re.sub(r"[^a-z0-9]", "", str(x).lower()))


def _names(k):
    return {_c(k), _c(META.get(k, {}).get("name", ""))} - {""}


targets = _names(key)                                   # works with the edits from my last patch
target = re.sub(r"[^a-z0-9]", "", key.lower())          # works with the original Cell 27
print("Report company key:", key)

esc = lambda s: _html.escape(str(s if s is not None else ""))
chart_config = globals().get("CHART_CONFIG", {"displaylogo": False, "responsive": True})
canon_key = lambda x: re.sub(r"[^a-z0-9]", "", str(x).lower())
target = canon_key(key)

try:
    import markdown as _md
    to_html = lambda t: _md.markdown(t, extensions=["tables", "sane_lists"])
except Exception:
    to_html = lambda t: "<p>" + esc(t).replace("\n", "<br>") + "</p>"

dname_ = lambda c: (f"{c} - {META[c]['name']}" if META.get(c, {}).get("name") not in (None, c) else c)


# ---- 1. identity and data quality (this company only) ----
identity = IDENT.get(key, {})
market = (globals().get("MARKET") or {}).get(key, {})
quality = None
if "QUALITY" in globals() and len(QUALITY[QUALITY.company == key]):
    quality = QUALITY[QUALITY.company == key].iloc[0]
extracted = (
    f"<td>{int(quality.cells)} cells: {int(quality.reported)} reported, {int(quality.derived)} derived, "
    f"{int(quality.from_web)} from web; review {int(quality.review)}, missing {int(quality.missing)}, "
    f"sanity findings {int(quality.sanity_findings)}</td>"
    if quality is not None else "<td>n/a</td>"
)
quality_row = (
    f"<tr><td><b>{esc(key)}</b></td><td>{esc(META[key]['name'])}</td>"
    f"<td>{esc(META[key].get('symbol') or 'not resolved')}<br>"
    f"<span class='small'>{esc(META[key].get('isin') or '')}</span></td>"
    f"<td>{esc(identity.get('status', ''))}</td>"
    f"<td>FY{FYS[key][0]}–FY{FYS[key][-1]} ({len(FYS[key])} reports)</td>"
    f"<td>{esc(market.get('web_check', 'no web data'))}</td>{extracted}</tr>"
)
quality_note = globals().get("quality_note", "")
clean = bool(globals().get("FROZEN_CLEAN", False))


# ---- 2. financial charts filtered to this company ----
def only_company(fig):
    new = go.Figure(layout=copy.deepcopy(fig.layout))
    for trace in fig.data:
        if canon_key(trace.name) == target:                    # group-by-year charts: one trace per company
            new.add_trace(copy.deepcopy(trace))
            continue
        xs = list(trace.x) if trace.x is not None else []
        idx = [i for i, x in enumerate(xs) if canon_key(x) == target]
        if not idx:
            continue
        t2 = type(trace)(trace)
        t2.x = [xs[i] for i in idx]
        t2.y = [list(trace.y)[i] for i in idx]
        if trace.customdata is not None:
            custom = list(trace.customdata)
            t2.customdata = [custom[i] if i < len(custom) else "" for i in idx]
        new.add_trace(t2)
    new.update_layout(title=None, margin={"t": 34})              # the HTML <h3> already shows the title
    return new


chart_blocks = []
for index, (name, title, fig) in enumerate(FIGS):
    sub = only_company(fig)
    if not sub.data:
        continue
    chart_html = sub.to_html(full_html=False, include_plotlyjs=False, div_id=f"fin_chart_{index}",
                             config=chart_config, default_width="100%",
                             default_height=f"{sub.layout.height or 320}px")
    chart_blocks.append(f'<div class="chart"><h3>{esc(str(title).replace("_", " "))}</h3>{chart_html}</div>')

price_blocks = []
for index, (company, ticker, fig) in enumerate(PRICE_CHARTS):
    if company != key:
        continue
    chart_html = fig.to_html(full_html=False, include_plotlyjs=False, div_id=f"price_chart_{index}",
                             config=chart_config, default_width="100%",
                             default_height=f"{fig.layout.height or 500}px")
    price_blocks.append(f'<div class="chart price-chart"><h3>{esc(company)} — share price ({esc(ticker)})</h3>{chart_html}</div>')
if not price_blocks:
    price_blocks = ["<p class='small'>No share-price chart is available for this company (symbol unresolved or no price data).</p>"]


# ---- 4. the 22 questions and answers (this company only, collapsed) ----
answers_this = [a for a in ANSWERS if a["company"] == key]
answers_by_qid = {a["qid"]: a for a in answers_this}
category_order = []
for question in BANK:
    if question["category"] not in category_order:
        category_order.append(question["category"])

inner = []
for category in category_order:
    inner.append(f"<h3>{esc(category)}</h3>")
    for question in [q for q in BANK if q["category"] == category]:
        answer = answers_by_qid.get(str(question["qid"]))
        body = answer_html(answer["answer"]) if answer else "<i>Answer missing.</i>"
        warning = "" if answer and answer.get("status") == "generated" else ' <span class="warn">(generation failed)</span>'
        inner.append(
            f'<div class="qa"><div class="q"><span class="qid">{esc(question["qid"])}</span> '
            f"{esc(question['question'])}</div><div class=\"a\">{body}{warning}</div></div>"
        )
qa_section = (
    f'<section class="company" id="company-{esc(key)}"><h2>{esc(dname_(key))}: questions and answers</h2>'
    f'<details class="qa-toggle"><summary>Show all {len(BANK)} questions and answers for {esc(key)}</summary>'
    f"{''.join(inner)}</details>{facts_table(key, answers_this)}</section>"
)


# ---- 4B. Gemini research (the selected company and its sector) ----
tone_colors = {"positive": "#2e9e5b", "mixed": "#c98a00", "negative": "#c62828", "unknown": "#667"}
intel_blocks = []
for r in GEMINI_RESULTS:
    if r.get("status") == "ok" and r.get("answer"):
        body = to_html(r["answer"])
    else:
        body = "<i>No answer generated: " + esc(r.get("error", "")) + "</i>"
    tone = r.get("signal", "unknown")
    badge = (f'<span style="color:{tone_colors.get(tone, "#667")};font-weight:700">tone: {esc(tone)}</span>'
             + (f' <span class="small">- {esc(r.get("signal_reason", ""))}</span>' if r.get("signal_reason") else ""))
    items = "".join(
        '<li><a href="' + esc(s["url"]) + '" target="_blank" rel="noopener">' + esc(s["title"] or s["url"]) + "</a></li>"
        for s in r.get("sources", [])
    )
    sources_html = '<p class="small"><b>Sources</b></p><ol class="small">' + items + "</ol>" if items else ""
    intel_blocks.append(
        '<details class="intel-toggle"><summary>' + esc(r["qid"]) + " - " + esc(r["title"])
        + '</summary><div class="intel"><p>' + badge + "</p>" + body + sources_html + "</div></details>"
    )

summary_line = esc(globals().get("SIGNAL_SUMMARY", ""))
intel_section = (
    "<style>.intel{overflow-wrap:anywhere;font-size:14px}"
    ".intel table{display:block;overflow-x:auto;border-collapse:collapse}"
    ".intel th,.intel td{border:1px solid #cfd6e0;padding:4px 8px}.intel li{margin:2px 0}"
    "details.intel-toggle{border:1px solid #cfd6e0;border-radius:6px;margin:10px 0;padding:0 14px}"
    "details.intel-toggle>summary{cursor:pointer;font-weight:600;padding:10px 0;color:#1f4e79}</style>"
    "<h2>4B. Company &amp; sector intelligence: " + esc(RESEARCH["company_name"]) + " (" + esc(RESEARCH["sector_name"]) + ")</h2>"
    "<p class='small'>Web pages retrieved with Tavily search on " + esc(RESEARCH["research_date"])
    + "; answers written by Gemini from those pages only. Numbered links are the sources used; verify key figures "
      "against company filings before relying on them. Tone labels are the model's reading of each answer, not a recommendation.</p>"
    + (f"<p><b>{summary_line}</b></p>" if summary_line else "")
    + "".join(intel_blocks)
)


# ---- assemble and write ----
assumptions_text = globals().get("assumptions", "")
document = (
    "<!DOCTYPE html><html><head><meta charset='utf-8'>"
    "<meta name='viewport' content='width=device-width, initial-scale=1'>"
    f"<title>{esc(REPORT_TITLE)} - {esc(key)} - {esc(RUN_NAME)}</title>"
    f"<style>{CSS}</style><script>{get_plotlyjs()}</script></head><body>"
    f"<h1>{esc(REPORT_TITLE)}: {esc(RESEARCH['company_name'])}</h1>"
    f"<div class='meta'>Company: {esc(key)} | Sector: {esc(RESEARCH['sector_name'])} | "
    f"Generated {dt.datetime.now().strftime('%Y-%m-%d %H:%M')}</div>"
    + (f"<div class='note {'ok' if clean else ''}'>{esc(quality_note)}</div>" if quality_note else "")
    + "<h2>1. Scope, identity and data quality</h2>"
    "<table><tr><th>Key</th><th>Official name</th><th>NSE symbol / ISIN</th><th>Identity check</th>"
    f"<th>Coverage</th><th>Web data check</th><th>Extracted cells</th></tr>{quality_row}</table>"
    f"<h2>2. Financial charts</h2><div class='chart-grid'>{''.join(chart_blocks)}</div>"
    f"<h2>3. Share-price chart</h2><div class='price-grid'>{''.join(price_blocks)}</div>"
    "<h2>4. Questions and answers</h2>"
    "<div class='tools'><button onclick='toggleAll(true)'>Expand all answers</button>"
    "<button onclick='toggleAll(false)'>Collapse all answers</button></div>"
    f"{qa_section}{intel_section}"
    f"<h2>5. Assumptions and method</h2><p>{esc(assumptions_text)}</p>"
    f"<script>{TOGGLE_JS}</script></body></html>"
)

COMPANY_REPORT_PATH = OUT / f"Report_{RUN_NAME}_{key}_{TODAY}.html"
COMPANY_REPORT_PATH.write_text(document, encoding="utf-8")
print(f"Single-company report written: {COMPANY_REPORT_PATH} | {COMPANY_REPORT_PATH.stat().st_size / 1e6:.1f} MB | "
      f"{len(chart_blocks)} financial charts, {len(answers_this)} answers, {len(GEMINI_RESULTS)} research answers")

if globals().get("AUTO_DOWNLOAD_REPORT"):
    try:
        from google.colab import files
        files.download(str(COMPANY_REPORT_PATH))
        print("HTML report download started.")
    except Exception as error:
        print("Automatic download unavailable; report is at", COMPANY_REPORT_PATH, error)

In [ ]:
# ===== CELL 28 (FINAL): SAVE DAILY ANALYSIS TO DRIVE (Model_1/Analysis) =====
import os, io, json, base64, html, datetime
from zoneinfo import ZoneInfo

BASE_DIR = str(BASE_DIR)
OUT_DIR  = os.path.join(BASE_DIR, "Analysis")

# Set CHART_VAR to the 22B variable name if you know it (e.g. "FIGS_GROUPED"); None = auto-pick
CHART_VAR = None
EXPECTED_CHARTS = 7

now   = datetime.datetime.now(ZoneInfo("Asia/Kolkata"))
TODAY = now.strftime("%Y-%m-%d")
os.makedirs(OUT_DIR, exist_ok=True)
esc = lambda x: html.escape(str(x))

# ---- pick sources ----
ANS = globals().get("ANSWERS")
RES = globals().get("GEMINI_RESULTS")
if CHART_VAR:
    c_name = CHART_VAR
else:
    c_name = next((n for n in ["FIGS_GROUPED","GROUPED_FIGS","FIGS_22B","GROUPED","FIGS"]
                   if n in globals() and globals()[n]), None)
CH = globals().get(c_name) if c_name else None
print(f"answers  <- ANSWERS ({type(ANS).__name__}, {len(ANS) if ANS else 0} items)")
print(f"charts   <- {c_name} ({type(CH).__name__}, {len(CH) if CH else 0} items)")
print(f"research <- GEMINI_RESULTS ({type(RES).__name__}, {len(RES) if RES else 0} items)")

# ---- companies ----
def company_list():
    if isinstance(globals().get("META"), dict) and META:
        out = []
        for k, v in META.items():
            nm = (v.get("official_name") or v.get("name")) if isinstance(v, dict) else None
            out.append(str(nm or k))
        return out
    return []
COMPANIES = company_list()

# ---- renderers ----
def render_value(v):
    if isinstance(v, dict):
        return "".join(f"<p><b>{esc(k)}:</b> {render_value(x) if isinstance(x,(dict,list,tuple)) else esc(x)}</p>" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return "<ul>" + "".join(f"<li>{render_value(x)}</li>" for x in v) + "</ul>"
    return f"<span>{esc(v).replace(chr(10), '<br>')}</span>"

def render_answers(a):
    if not a:
        return "<p><i>Nothing found.</i></p>"
    if isinstance(a, dict):
        return "".join(f"<details open><summary>{esc(k)}</summary>{render_value(v)}</details>" for k, v in a.items())
    if isinstance(a, (list, tuple)):
        return "".join(f"<details open><summary>Item {i}</summary>{render_value(v)}</details>" for i, v in enumerate(a, 1))
    return render_value(a)

RENDERED = {"n": 0, "failed": 0}
def render_chart(fig):
    try:
        if hasattr(fig, "to_html"):                                   # plotly
            RENDERED["n"] += 1
            return fig.to_html(full_html=False, include_plotlyjs=False)
        if hasattr(fig, "savefig"):                                   # matplotlib
            buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
            RENDERED["n"] += 1
            return f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" style="max-width:100%">'
        if isinstance(fig, (bytes, bytearray)):                       # raw PNG bytes
            RENDERED["n"] += 1
            return f'<img src="data:image/png;base64,{base64.b64encode(fig).decode()}" style="max-width:100%">'
        if isinstance(fig, str) and ("<svg" in fig or "<div" in fig or "<img" in fig or "<canvas" in fig):
            RENDERED["n"] += 1
            return fig                                                # ready-made HTML/SVG
        if isinstance(fig, dict):
            for key in ("html", "svg", "fig", "figure", "chart"):
                if key in fig:
                    return render_chart(fig[key])
        if isinstance(fig, (tuple, list)) and fig:
            return "".join(render_chart(x) for x in fig if not isinstance(x, str) or "<" in x)
    except Exception as e:
        RENDERED["failed"] += 1
        return f"<p><i>Chart could not be rendered: {esc(e)}</i></p>"
    RENDERED["failed"] += 1
    return f"<p><i>Unsupported chart type: {esc(type(fig).__name__)}</i></p>"

def render_charts(c):
    if not c:
        return "<p><i>No charts found.</i></p>"
    items = list(c.items()) if isinstance(c, dict) else list(enumerate(c, 1))
    return "".join(f"<h3>{esc(k)}</h3>{render_chart(f)}" for k, f in items)

# ---- assemble ----
doc = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Analysis {TODAY}</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
 body{{font-family:Arial,sans-serif;max-width:1100px;margin:auto;padding:24px;color:#222}}
 .sub{{color:#555;margin-bottom:24px}}
 details{{border:1px solid #ddd;border-radius:6px;padding:8px 12px;margin:10px 0}}
 summary{{cursor:pointer;font-weight:bold}}
</style></head><body>
<h1>Analysis dated {TODAY}</h1>
<div class="sub"><b>Companies analysed:</b> {esc(", ".join(COMPANIES)) or "not detected"}</div>
<h2>1. Report questions (22) - answers</h2>{render_answers(ANS)
<h2>2. Financial charts</h2>{render_charts(CH)}
<h2>3. API calling questions (Gemini + Tavily research)</h2>{render_answers(RES)}
</body></html>"""

out_path = os.path.join(OUT_DIR, f"Analysis_{TODAY}.html")
if os.path.exists(out_path):
    out_path = os.path.join(OUT_DIR, f"Analysis_{TODAY}_{now.strftime('%H%M')}.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(doc)

print(f"\nSaved: {out_path}")
print(f"Companies: {COMPANIES}")
print(f"Size: {os.path.getsize(out_path)/1024:.0f} KB")
print(f"Charts rendered: {RENDERED['n']} (expected {EXPECTED_CHARTS}), failed: {RENDERED['failed']}")
if RENDERED["n"] != EXPECTED_CHARTS:
    print("WARNING: chart count differs. Check the CHART_VAR setting or the type of the chart variable:")
    if CH:
        first = next(iter(CH.values())) if isinstance(CH, dict) else CH[0]
        print("  first chart item type:", type(first).__name__)